# TrafficFlowBench Task 1·2·3 — Colab 통합 노트북

**`task123_kaggle.ipynb`의 모델·학습·평가·병합 흐름을 Colab으로 옮겼습니다.** 기반 소스와 물리 보정 모듈, 아래 Task 2·3 개선을 셀에 포함하며 메모리에서 불러옵니다. Git clone이나 별도 프로젝트 파일 없이 Google Drive의 대회 데이터를 사용합니다.

- Task 1·3: MagiNet, `temporal_direct`, current003 학습, 관측 문맥·채널별 공간 보정, 과거 구간 검색을 포함합니다.
- Task 2: 공개 `T` 입력, 44특징, epoch별 표본 재추출, 조건별 출력층·확률 라벨·미래 과거 평균·균등 학습·날짜 분산 검증을 포함합니다.
- 최종 제출: Task 1·2 예측과 **기존 Task 4 CSV**를 연결합니다. Task 4 파일을 지정하지 않으면 기존 `λ=0.05, p=0` NNLS로 채울 수 있습니다. Task 3의 별도 제출 행은 없습니다.

**실행 순서**

1. Colab에 이 `.ipynb`를 업로드하고 GPU 런타임을 선택합니다. 위에서부터 실행하며 Drive 연결을 승인해 주세요.
2. 첫 설정 셀의 `RELEASE_ROOT`를 대회 데이터 폴더로 지정합니다. 해당 폴더 또는 바로 아래 `kaggle_public/`에 `corridors/`, `task1/`, `task2/`가 있어야 합니다.
3. `RUN_MODE="train"`, `PANELS=None`으로 전체 학습합니다. 예측만 할 때는 `RUN_MODE="predict"`를 사용합니다. 학습 전에 입력·backend 검사를 수행합니다.
4. 전체 학습은 `RUN_MODE="train"`, `PANELS=None`으로 실행합니다. Task 2의 기본 backend는 **Mamba2**입니다. 필요한 고정 버전 패키지를 설치하고 GPU 호환성을 먼저 확인합니다. `QUEUE_BACKEND="scan"`은 직접 선택할 수 있으며 모델 구조가 달라집니다.
5. 같은 Drive 결과를 재개하려면 `SAVE_ROOT`를 유지하고 `RESUME=True`로 설정합니다. epoch는 **누적 목표**이며 과제별로 새 학습·재개·완료를 판단합니다. 예측만 할 때는 `RUN_MODE="predict"`입니다.

체크포인트·보고서는 `SAVE_ROOT/reports/`, 완성된 제출은 `SAVE_ROOT/submission.csv`에 바로 저장합니다. 기본 `SAVE_ROOT`는 `/content/drive/MyDrive/dataset/trafficflow_task123`입니다. 입력 연결·전처리 캐시는 `/content/trafficflow_task123/`의 런타임 임시 디스크에 두므로 런타임이 종료되면 다시 생성합니다.

**기존 Kaggle 결과 가져오기:** 저장한 Output의 `trafficflow_task123/` 폴더를 Drive에 올리고 `RESTORE_ROOT`로 지정합니다. 비어 있는 `SAVE_ROOT/reports/`에 보고서·모델을 한 번 복사합니다. 이후 `RESTORE_ROOT=None`으로 되돌려 사용해 주세요. 학습 재개는 `RESUME=True`, 예측은 `RUN_MODE="predict"`로 실행하며, 기존 모델을 쓸 때는 원래 실험명·Queue tag/flags/backend·입력 데이터를 지정합니다. Task 1·3은 `current003` 결과 폴더를 사용하고, 새 Queue는 `QUEUE_TAG="conditional-future-sampled-v4"`에 저장합니다. Queue는 원본 파일 크기·수정 시각도 검사하므로 데이터 사본의 메타데이터가 달라지면 재개가 거부될 수 있습니다.

원본: [task 123 커밋](https://github.com/mmt-1234/trafficflowbench/commit/cdf888214fe8a54b73302f3988b764b74b3f30a4). State는 로컬 Task 1 지표이며 Queue·Physics는 대리 지표입니다. 이 노트북에 대회 데이터로 새로 측정한 점수는 들어 있지 않습니다.

Task 2 공개 조건과 점수식 근거: [공식 설정](https://github.com/jacky850/trafficflowbench-public/blob/main/config/synthetic_release_v1.json), [공식 평가 명세](https://github.com/jacky850/trafficflowbench-public/blob/main/docs/SCORING_SPEC.md), 확인일: 2026-10-28. 숨겨진 정답·경계 유량 없이 계산하는 로컬 지표는 대리 지표입니다.


## 1. 실행 설정

Task 1은 `temporal_direct`와 채널별 공간 보정을 사용하는 `channel_graph` 구조로 고정합니다. `MAGINET_EXPERIMENT`는 결과 폴더 이름이며 이전 구조의 새 학습 선택지는 제거했습니다. 현재 구조로 저장한 체크포인트는 같은 파라미터 이름으로 불러옵니다.

`QUEUE_FLAGS`를 비우고 `QUEUE_LABEL_MODE="hard"`로 설정하면 추가 입력·그래프·확률 라벨을 끈 선형 출력층 비교를 할 수 있습니다. 기존 v3 표본 규칙을 유지하며 새 기능이 켜지면 설정 버전 v4를 기록합니다. 단독 옵션 비교 시 `QUEUE_TAG`를 바꾸어 별도 결과 폴더에 저장해 주세요. 모든 비교에서 검증 날짜 분산 설정과 학습 예산을 맞춰야 합니다. `USE_RETRIEVAL=True`는 1월 전반에서 검색 방법·속도/교통량 혼합 비율을 선택하고 후반·예측에 고정 적용합니다. 2월은 이미 개발에 사용한 추가 진단 구간입니다.

Task 1·3 학습 손실은 `current003`(`N=0.05`, `dN=0.03`, `FD=0`)으로 고정합니다. 가중치 비교 프리셋과 항별 기울기 진단은 제거했습니다. 모델·보고서는 `reports/full/maginet/<실험>-current003/`에 저장합니다. 같은 설정의 모델만 학습을 재개하며, 저장된 모델의 예측에는 기존 설정을 사용합니다.

Task 2 v4는 조건별 출력층·미래 시점별 과거 평균·확률 라벨·패널/조건 균등 학습을 기본으로 합니다. `QUEUE_FLAGS`에서 기능을 끄고 `QUEUE_LABEL_MODE="hard"`로 바꿔 개별 비교할 수 있으며, 새 비교마다 `QUEUE_TAG`를 분리합니다.


In [ ]:
from pathlib import Path

RUN_MODE = "train"             # "train", "predict"
RELEASE_ROOT = "/content/drive/MyDrive/dataset/trafficflow/kaggle_public"  # 또는 바로 아래 kaggle_public/
WORK_ROOT = Path("/content/trafficflow_task123")  # 입력 연결·캐시: 런타임 임시 디스크
SAVE_ROOT = Path("/content/drive/MyDrive/dataset/trafficflow_task123")  # 모델·보고서·제출: Drive
PANELS = None                   # None: 전체. 일부 패널은 이름 목록으로 지정합니다.
RESTORE_ROOT = None             # 기존 Kaggle Output 등을 Drive에 올린 trafficflow_task123/ 폴더
RESUME = False                  # 같은 SAVE_ROOT에서 재개할 때 True. epoch는 누적 목표입니다.

MAGINET_EXPERIMENT = "temporal_direct_dn003_channel_graph"  # 결과 폴더 이름; 모델 구조는 고정
# 기존 current003 모델로 보정만 학습: RUN_MODE="predict".
CORRECTOR_MODE = "off"          # "off", "train"(B/C 비교·선택), "predict"(저장된 선택 적용)
CORRECTOR_SELECTION = None       # predict: 보정 실험의 selection.json 절대 경로
CORRECTOR_EPOCHS = 10
CORRECTOR_STEPS_PER_GROUP = 4     # B/C에 같은 panel×regime별 업데이트 수
MAGINET_EPOCHS = 50
MAGINET_BATCH_SIZE = 2
QUEUE_EPOCHS = 75
QUEUE_BACKEND = "mamba2"         # 기본 학습 backend
QUEUE_TAG = "conditional-future-sampled-v4"  # 기존 Queue와 분리한 새 실험
QUEUE_INIT_CHECKPOINT = None    # 선택: 기존 pooled-mlp/conditional Queue의 queue.pt
QUEUE_LABEL_MODE = "sample"    # "hard", "soft", "sample"
QUEUE_LABEL_MARGIN = 5.0       # 임계속도 ±5km/h에서 확률 전이; 실험 시작값
QUEUE_FLAGS = ["--queue-origin", "--queue-resample", "--queue-condition-gap",
               "--queue-changes", "--queue-head", "conditional", "--graph",
               "--queue-future-prior", "--queue-balanced"]
VALIDATION_WINDOWS = 5          # 패널·조건별 고정 검증 표본 수
QUEUE_PUBLIC_VALIDATION = True  # 새 Queue 학습의 검증 구간에 공개 선정 조건 적용
CALIBRATE_QUEUE = True          # 기존 모델의 predict 모드에서도 1월 확률로 선택 가능
QUEUE_THRESHOLDS = (0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9)
USE_RETRIEVAL = False           # 첫 비교는 보간 혼합만 사용합니다.
RUN_FEBRUARY = False
MAKE_PREDICTIONS = True
MAKE_SUBMISSION = True          # 전체 패널 예측에서만 완성합니다.
TASK4_FILES = []                # validation/private를 모두 포함하는 Task 4 CSV 경로 목록
BUILD_TASK4_BASELINE = True     # TASK4_FILES가 없을 때 기존 NNLS 사용
INSTALL_MISSING_PACKAGES = True
CPU_THREADS = 2
SEED = 42


## 2. Drive 연결·패키지와 입력 경로

Drive 연결을 승인한 뒤 Colab에 설치된 PyTorch를 사용합니다. 부족한 일반 패키지만 설치하며, Mamba2의 고정 버전은 전체 학습·해당 체크포인트 예측에서 필요할 때 설치합니다. GPU에서 backend가 동작하지 않으면 긴 학습 전에 중단합니다.

데이터 전체를 복사하지 않고 로컬 `WORK_ROOT`에 원본을 참조하는 링크를 만듭니다. Queue의 과거 평균 캐시와 MagiNet 전처리 캐시는 임시 디스크에 저장하고, 모델·로그·예측은 Drive의 `SAVE_ROOT`에 저장합니다. 새 실험을 기존 결과와 분리하려면 실험명·Queue tag 또는 `SAVE_ROOT`를 바꿔 주세요.

복원은 임시 폴더에 끝까지 복사한 뒤 `reports/`로 확정합니다. 중단된 임시 폴더는 다음 복원을 막지 않습니다. Queue 재개는 저장된 BF16과 현재 GPU의 호환성을 긴 학습 전에 검사하며, 평가·예측에서는 미지원 장치에 FP32를 사용합니다.


In [4]:
from google.colab import drive
drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import ast
import contextlib
import gc
import hashlib
import importlib.util
import json
import os
import shlex
import shutil
import subprocess
import sys
import tempfile
import types

if CORRECTOR_MODE not in {"off", "train", "predict"}:
    raise ValueError('CORRECTOR_MODE는 off/train/predict 중 하나여야 합니다.')
if CORRECTOR_MODE != "off" and USE_RETRIEVAL:
    raise ValueError('첫 물리 보정 비교에서는 USE_RETRIEVAL=False로 고정해 주세요.')
if CORRECTOR_MODE == "predict" and CORRECTOR_SELECTION is None:
    raise ValueError('저장된 보정 selection.json을 CORRECTOR_SELECTION에 지정해 주세요.')
if min(CORRECTOR_EPOCHS, CORRECTOR_STEPS_PER_GROUP) < 1:
    raise ValueError('보정 학습량은 1 이상이어야 합니다.')
if VALIDATION_WINDOWS < 1:
    raise ValueError('검증 표본 수는 1 이상이어야 합니다.')
if RUN_MODE not in {"train", "predict"}:
    raise ValueError("RUN_MODE를 train/predict 중에서 지정해 주세요.")
packages = {"numpy": "numpy", "pandas": "pandas>=2.0", "pyarrow": "pyarrow", "scipy": "scipy"}
missing = [requirement for name, requirement in packages.items() if importlib.util.find_spec(name) is None]
if missing:
    if not INSTALL_MISSING_PACKAGES:
        raise RuntimeError(f"필요한 패키지를 먼저 설치해 주세요: {missing}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
import numpy as np
import pandas as pd
import torch
if tuple(int(n) for n in torch.__version__.split("+")[0].split(".")[:2]) < (2, 3):
    raise RuntimeError("PyTorch 2.3 이상이 필요합니다. Colab의 최신 GPU 런타임을 사용해 주세요.")
if int(pd.__version__.split(".")[0]) < 2:
    raise RuntimeError("pandas 2.0 이상이 필요합니다.")
torch.set_num_threads(CPU_THREADS)
WORK_ROOT = Path(WORK_ROOT).resolve()
WORK_ROOT.mkdir(parents=True, exist_ok=True)
SAVE_ROOT = Path(SAVE_ROOT).expanduser().resolve()
SAVE_ROOT.mkdir(parents=True, exist_ok=True)

def locate_release(requested, search_root=Path("/content/drive/MyDrive/dataset")):
    if requested is not None:
        p = Path(requested).expanduser()
        candidates = [p, p / "kaggle_public"]
    else:
        candidates = [p.parent for p in search_root.rglob("submission_key.csv")]
    matches = sorted({p.resolve() for p in candidates
                      if all((p / name).is_dir() for name in ("corridors", "task1", "task2"))})
    if len(matches) != 1:
        raise ValueError(f"배포본 경로를 하나로 지정해 주세요. RELEASE_ROOT 후보: {matches}")
    return matches[0]

INPUT_RELEASE = locate_release(RELEASE_ROOT)
RELEASE = WORK_ROOT / "data" / "release"
RELEASE.mkdir(parents=True, exist_ok=True)
for source in INPUT_RELEASE.iterdir():
    destination = RELEASE / source.name
    if destination.is_symlink() or destination.exists():
        if not destination.is_symlink() or destination.resolve() != source.resolve():
            raise FileExistsError(f"다른 입력이 연결돼 있습니다. 새 WORK_ROOT를 사용해 주세요: {destination}")
    else:
        destination.symlink_to(source, target_is_directory=source.is_dir())
def restore_reports(previous, target):
    previous, target = Path(previous).resolve(), Path(target).resolve()
    if not previous.is_dir():
        raise FileNotFoundError(previous)
    if previous == target or previous.is_relative_to(target) or target.is_relative_to(previous):
        raise ValueError('복원 원본과 대상 폴더는 서로 분리해 주세요.')
    if target.exists() and any(target.iterdir()):
        raise FileExistsError('현재 결과를 덮어쓸 수 없습니다. 빈 SAVE_ROOT에서 복원하거나 RESTORE_ROOT=None으로 설정해 주세요.')
    target.parent.mkdir(parents=True, exist_ok=True)
    pending = Path(tempfile.mkdtemp(prefix='.reports-restore-', dir=target.parent))
    try:
        shutil.copytree(previous, pending, dirs_exist_ok=True)
        if target.exists():
            target.rmdir()
        pending.replace(target)
    finally:
        shutil.rmtree(pending, ignore_errors=True)

RESULTS = SAVE_ROOT / "reports" / "full"
if RESTORE_ROOT is not None:
    previous = Path(RESTORE_ROOT).expanduser() / "reports"
    if not previous.is_dir():
        raise FileNotFoundError(f"이전 trafficflow_task123/reports 폴더가 필요합니다: {previous}")
    restore_reports(previous, SAVE_ROOT / 'reports')
all_panels = sorted(p.name for p in (RELEASE / "corridors").iterdir() if p.is_dir())
selected_panels = list(PANELS) if PANELS is not None else None
if selected_panels is not None and not set(selected_panels) <= set(all_panels):
    raise ValueError("PANELS에 입력에 없는 패널이 있습니다.")
if selected_panels == [] or not set(selected_panels or all_panels) - {"D12_I405_N", "D12_I405_S"}:
    raise ValueError("Task 2 대상 패널을 하나 이상 선택해 주세요.")
EFFECTIVE_BACKEND = QUEUE_BACKEND
MAGINET_DIR = RESULTS / 'maginet' / f'{MAGINET_EXPERIMENT}-current003'
RETRIEVAL_SELECTION = MAGINET_DIR / "retrieval" / "selected_retrieval.json"
QUEUE_DIR = RESULTS / "queue" / f"{QUEUE_TAG}-{EFFECTIVE_BACKEND}"
QUEUE_CHECKPOINT = QUEUE_DIR / "queue.pt"
queue_runtime = None
queue_runtime_path = QUEUE_CHECKPOINT if RUN_MODE == 'predict' else QUEUE_CHECKPOINT.with_name('queue.last.pt')
if RUN_MODE == 'predict' or RESUME and queue_runtime_path.is_file():
    queue_runtime = torch.load(queue_runtime_path, map_location='cpu', weights_only=True)
    EFFECTIVE_BACKEND = queue_runtime['model_config']['backend']
    if RUN_MODE != 'predict' and queue_runtime['training_args'].get('use_bf16', False) and not torch.cuda.is_bf16_supported():
        raise RuntimeError('저장된 Queue는 BF16을 사용합니다. BF16 지원 GPU로 바꾼 뒤 재개해 주세요.')
if RUN_MODE == "train" and EFFECTIVE_BACKEND != "scan" and not torch.cuda.is_available():
    raise RuntimeError("Queue의 Mamba 학습에는 GPU 런타임이 필요합니다.")
if EFFECTIVE_BACKEND.startswith("mamba") and importlib.util.find_spec("mamba_ssm") is None:
    if not INSTALL_MISSING_PACKAGES:
        raise RuntimeError("Mamba 패키지가 필요합니다. 설치하거나 QUEUE_BACKEND를 명시적으로 바꿔 주세요.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "einops==0.8.2", "ninja==1.13.2",
                    "apache-tvm-ffi==0.1.9"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--no-build-isolation",
                    "git+https://github.com/state-spaces/mamba.git@e9594ce1c732d97440f0332fdc43170a2294dbfa"], check=True)
CACHE_DIR = WORK_ROOT / "cache" / "maginet"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
queue_wanted_bf16 = queue_runtime['training_args'].get('use_bf16', False) if queue_runtime else torch.cuda.is_available() and torch.cuda.is_bf16_supported()
QUEUE_PRECISION = 'bf16' if queue_wanted_bf16 and torch.cuda.is_available() and torch.cuda.is_bf16_supported() else 'fp32'
if RUN_MODE == 'predict' and queue_wanted_bf16 and QUEUE_PRECISION == 'fp32':
    print('현재 장치의 BF16 미지원으로 Queue 추론은 FP32를 사용합니다.')
del queue_runtime
print({"input": str(INPUT_RELEASE), "writable_release": str(RELEASE), "saved_results": str(SAVE_ROOT), "mode": RUN_MODE,
       "panels": selected_panels or all_panels, "queue_backend": EFFECTIVE_BACKEND,
       "queue_precision": QUEUE_PRECISION, "torch": str(torch.__version__),
       "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"})


## 3. 내장 소스

기반 여섯 셀은 `cdf8882` 코드이며, `physics_corrector`는 고정된 모델의 물리 보정 모듈입니다. `maginet`, `learning`, `run`에는 이 노트북의 current003 학습·Queue 검증·후처리 개선이 들어 있습니다. 셀을 펼쳐 확인하거나 수정할 수 있습니다. 내부 import 이름을 노트북 전용 이름으로 바꾸어 메모리에 정의하며 파일을 생성하지 않습니다. 실행 소스의 SHA-256을 검사하고, 기반 원본 해시는 노트북 메타데이터의 `base_source_sha256`에 보존합니다.


In [ ]:
SOURCE_COMMIT = "cdf888214fe8a54b73302f3988b764b74b3f30a4"
SOURCE_HASHES = {'data': '9dec2c7be98fd5809abda519117f3430f4bb7be5f7990f66eb3a4c90fc5746af', 'fast_model': '7f9b603aaa44da66853a148b9d9189a965d9b84b78304772a0b10a2a4f3830e0', 'learning': '3bc768ceb88184e40e81a3d8a56c42c477860bfeb84dbfdf16aa072fab301284', 'historical_retrieval': '7089b3bac861b4aac54d6b89b615e12b965fd679ffd6f311fecc038d787555f4', 'maginet': '044d57aab84d7db256bc57586ef446fd13a9ca8fa3f690a38830dfa712f31db1', 'run': '1bb709b6ec9a424105ec341eddbd157cd1a98701c9849b127bd88a44ca5d48eb', 'physics_corrector': 'b43cad4d3fad2200efab941ac2406a41bffa3fb7de77e06cfa1d799b21bb3d67'}
MODULE_NAMES = {name: "_trafficflow_task123_" + name for name in SOURCE_HASHES}
MODULES = {}

def load_source(name, source):
    if hashlib.sha256(source.encode()).hexdigest() != SOURCE_HASHES[name]:
        raise ValueError(f"내장 원본이 바뀌었습니다. 수정 의도가 있다면 해당 해시도 갱신해 주세요: {name}")
    tree = ast.parse(source)
    for node in ast.walk(tree):
        if isinstance(node, ast.ImportFrom) and node.module in MODULE_NAMES:
            node.module = MODULE_NAMES[node.module]
    module = types.ModuleType(MODULE_NAMES[name])
    module.__source_sha256__ = SOURCE_HASHES[name]
    module.__source_sha256__ = SOURCE_HASHES[name]
    module.__file__ = str(WORK_ROOT / "src" / f"{name}.py")
    sys.modules[module.__name__] = module
    exec(compile(tree, module.__file__, "exec"), module.__dict__)
    MODULES[name] = module
    print(f"내장 코드 로드: {name} ({len(source.splitlines())}줄)")


In [ ]:
# @title
# src/data.py — 원본 cdf888214fe8a54b73302f3988b764b74b3f30a4
load_source('data', r'''"""Released panel data to tensors, with masks applied before aggregation."""

from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path
import heapq

import numpy as np
import pandas as pd


def scan_layout(link_ids: list[str], topology: pd.DataFrame) -> tuple[list[str], tuple[tuple[int, int], ...]]:
    """Topological order within each weakly connected component; cycles are stable."""
    links = set(link_ids)
    outgoing = {link: set() for link in links}
    neighbors = {link: set() for link in links}
    for row in topology.itertuples(index=False):
        target = str(row.link_id)
        if target not in links:
            continue
        for source in str(getattr(row, "incoming_link_ids", "")).split(";"):
            source = source.strip()
            if source in links and source != target:
                outgoing[source].add(target)
                neighbors[source].add(target)
                neighbors[target].add(source)
        for dest in str(getattr(row, "outgoing_link_ids", "")).split(";"):
            dest = dest.strip()
            if dest in links and dest != target:
                outgoing[target].add(dest)
                neighbors[target].add(dest)
                neighbors[dest].add(target)
    seen: set[str] = set()
    ordered: list[str] = []
    segments: list[tuple[int, int]] = []
    for root in sorted(links):
        if root in seen:
            continue
        component = {root}
        pending = [root]
        seen.add(root)
        while pending:
            for neighbor in neighbors[pending.pop()]:
                if neighbor not in seen:
                    seen.add(neighbor)
                    component.add(neighbor)
                    pending.append(neighbor)
        indegree = {link: 0 for link in component}
        for source in component:
            for target in outgoing[source] & component:
                indegree[target] += 1
        ready = [link for link in component if indegree[link] == 0]
        heapq.heapify(ready)
        part = []
        while ready:
            source = heapq.heappop(ready)
            part.append(source)
            for target in outgoing[source] & component:
                indegree[target] -= 1
                if indegree[target] == 0:
                    heapq.heappush(ready, target)
        part.extend(sorted(component - set(part)))
        start = len(ordered)
        ordered.extend(part)
        segments.append((start, len(ordered)))
    return ordered, tuple(segments)


@dataclass
class PanelInfo:
    name: str
    links: list[str]
    segments: tuple[tuple[int, int], ...]
    link_index: dict[str, int]
    free_speed: np.ndarray
    lanes: np.ndarray
    lengths: np.ndarray
    cutoffs: np.ndarray
    station_lanes: dict[str, float]
    capacity: np.ndarray | None = None
    jam_density: np.ndarray | None = None
    position: np.ndarray | None = None
    edges: np.ndarray = field(default_factory=lambda: np.empty((2, 0), dtype=np.int64))
    ramp_map: dict[str, tuple[int, int]] = field(default_factory=dict)

    @classmethod
    def load(cls, release: Path, panel: str) -> "PanelInfo":
        panel_dir = release / "corridors" / panel
        candidates = sorted((panel_dir / "train" / "mainline_states").glob("**/*.parquet"))
        if not candidates:
            raise FileNotFoundError(f"No original train observations under {panel_dir}")
        observed = pd.read_parquet(candidates[0], columns=["link_id"]).link_id.astype(str).unique().tolist()
        network = panel_dir / "network"
        topology = pd.read_csv(network / "lwr_mainline_topology.csv", dtype=str)
        # Latent mainline connectors carry no detector value, but preserve the
        # road between observed links during the spatial scan.
        links, segments = scan_layout(sorted(set(observed) | set(topology.link_id.dropna().astype(str))), topology)
        frame = pd.read_csv(network / "links.csv", dtype={"link_id": str}).drop_duplicates("link_id").set_index("link_id")
        fd_path = network / "fd_parameters.csv"
        fd = pd.read_csv(fd_path, dtype={"link_id": str, "station_id": str}) if fd_path.exists() else pd.DataFrame()
        grouped = fd.groupby("link_id").mean(numeric_only=True) if not fd.empty else pd.DataFrame()

        def parameter(column: str, default: float) -> np.ndarray:
            raw = pd.to_numeric(frame[column], errors="coerce").reindex(links) if column in frame else pd.Series(default, index=links)
            if column in grouped:
                raw = pd.to_numeric(grouped[column], errors="coerce").reindex(links).fillna(raw)
            return raw.fillna(default).to_numpy(dtype=np.float32)

        free_speed = parameter("free_speed_kmh", 105.0)
        lanes = np.clip(parameter("lanes", 1.0), 1, 16)
        lengths = parameter("length_km", 1.0)
        cutoffs = 0.6 * free_speed
        if "v_cut" in grouped:
            cutoffs = pd.to_numeric(grouped.v_cut, errors="coerce").reindex(links).fillna(pd.Series(cutoffs, index=links)).to_numpy(dtype=np.float32)
        station_lanes: dict[str, float] = {}
        if "station_id" in fd and "lanes" in fd:
            for row in fd.dropna(subset=["station_id", "lanes"]).itertuples(index=False):
                station_lanes[str(row.station_id)] = float(row.lanes)
        lookup = {link: i for i, link in enumerate(links)}
        pairs = set()
        for row in topology.itertuples(index=False):
            target = str(row.link_id)
            for source in str(getattr(row, "incoming_link_ids", "")).split(";"):
                if source in lookup and target in lookup and source != target:
                    pairs.add((lookup[source], lookup[target]))
            for dest in str(getattr(row, "outgoing_link_ids", "")).split(";"):
                if target in lookup and dest in lookup and dest != target:
                    pairs.add((lookup[target], lookup[dest]))
        positions = topology.drop_duplicates("link_id").set_index("link_id")
        position = pd.to_numeric(positions.get("order_index", pd.Series(dtype=float)), errors="coerce").reindex(links)
        position = position.fillna(pd.Series(np.arange(len(links)), index=links)).to_numpy(np.float32)
        position = (position - position.min()) / max(float(np.ptp(position)), 1.0)
        ramp_map = {}
        ramp_path = network / "ramp_attachment_map.csv"
        if ramp_path.exists():
            for row in pd.read_csv(ramp_path, dtype=str).itertuples(index=False):
                if row.nearest_mainline_link_id in lookup:
                    if row.ramp_type not in {"OR", "FR", "on", "off"}:
                        raise ValueError(f"Unknown ramp type: {row.ramp_type}")
                    ramp_map[row.ramp_link_id] = (lookup[row.nearest_mainline_link_id], int(row.ramp_type in {"FR", "off"}))
        return cls(panel, links, segments, lookup, free_speed, lanes, lengths, cutoffs, station_lanes,
                   parameter("capacity_vph", 6000.0), parameter("k_jam", 400.0), position,
                   np.array(sorted(pairs), dtype=np.int64).reshape(-1, 2).T, ramp_map)


@dataclass
class StationDay:
    timestamps: list[str]
    station_ids: list[str]
    station_links: list[str]
    station_to_link: np.ndarray
    lanes: np.ndarray
    speed: np.ndarray
    flow_lane: np.ndarray
    eligible: np.ndarray
    occupancy: np.ndarray | None = None
    quality: np.ndarray | None = None

    @classmethod
    def load(cls, path: Path, panel: PanelInfo) -> "StationDay":
        columns = ["timestamp", "station_id", "link_id", "speed_kmh", "flow_vph", "is_score_eligible"]
        # Optional published features also exist in Task 2 histories.
        import pyarrow.parquet as pq
        available = set(pq.read_schema(path).names) if path.stat().st_size else set(columns)
        columns += [c for c in ("occupancy", "pct_observed") if c in available]
        frame = pd.read_parquet(path, columns=columns)
        for column in ("timestamp", "station_id", "link_id"):
            frame[column] = frame[column].astype(str)
        if frame.duplicated(["timestamp", "station_id", "link_id"]).any():
            raise ValueError(f"Duplicate station/time rows in {path}")
        stations = frame[["station_id", "link_id"]].drop_duplicates()
        if stations.station_id.duplicated().any():
            raise ValueError(f"Station maps to multiple links in {path}")
        stations = stations.sort_values(["link_id", "station_id"])
        station_ids = stations.station_id.tolist()
        station_links = stations.link_id.tolist()
        unknown = set(station_links) - set(panel.link_index)
        if unknown:
            raise ValueError(f"Links absent from panel layout: {sorted(unknown)[:4]}")
        timestamps = sorted(frame.timestamp.unique().tolist())
        index = pd.MultiIndex.from_product([timestamps, station_ids], names=["timestamp", "station_id"])
        aligned = frame.set_index(["timestamp", "station_id"]).reindex(index)
        speed = pd.to_numeric(aligned.speed_kmh, errors="coerce").to_numpy(dtype=np.float32).reshape(len(timestamps), -1)
        flow = pd.to_numeric(aligned.flow_vph, errors="coerce").to_numpy(dtype=np.float32).reshape(len(timestamps), -1)
        lanes = np.array([panel.station_lanes.get(s, panel.lanes[panel.link_index[l]]) for s, l in zip(station_ids, station_links)], dtype=np.float32)
        lanes = np.clip(lanes, 1, 16)
        eligible = aligned.is_score_eligible.fillna(False).astype(bool).to_numpy(copy=True).reshape(len(timestamps), -1)
        eligible &= np.isfinite(speed) & np.isfinite(flow)
        occupancy = pd.to_numeric(aligned.occupancy, errors="coerce").to_numpy(np.float32).reshape(speed.shape) if "occupancy" in aligned else np.full_like(speed, np.nan)
        quality = pd.to_numeric(aligned.pct_observed, errors="coerce").fillna(0).to_numpy(np.float32).reshape(speed.shape) / 100 if "pct_observed" in aligned else eligible.astype(np.float32)
        return cls(timestamps, station_ids, station_links,
                   np.array([panel.link_index[l] for l in station_links], dtype=np.int64),
                   lanes, speed / 25.0, flow / lanes[None, :] / 600.0, eligible, occupancy, quality)








# The first nine channels retain the legacy order; v2 checkpoints store this list.
FEATURE_NAMES = ("speed", "flow_lane", "both_observed", "free_speed", "lanes", "length",
                 "time_sin", "time_cos", "weekday", "occupancy", "speed_observed", "flow_observed",
                 "occupancy_observed", "pct_observed", "speed_age", "flow_age", "occupancy_age",
                 "capacity", "road_position", "on_ramp_attached", "off_ramp_attached", "on_ramp_flow",
                 "off_ramp_flow", "on_ramp_valid", "off_ramp_valid", "prior_speed", "prior_flow_lane",
                 "prior_occupancy", "time_slot", "day_of_week", "relative_time")

QUEUE_CHANGE_NAMES = ("speed_margin",) + tuple(
    f"{channel}_{field}_{minutes}m" for minutes in (5, 15)
    for channel in ("speed", "flow_lane", "occupancy") for field in ("delta", "valid")
)


def queue_feature_names(changes=False):
    return FEATURE_NAMES + QUEUE_CHANGE_NAMES if changes else FEATURE_NAMES


QUEUE_FUTURE_NAMES = ('prior_speed', 'prior_flow_lane', 'prior_occupancy',
                      'time_sin', 'time_cos', 'weekday', 'lead')


def queue_future_features(origin, panel, profile):
    """Known calendar and fit-only priors; never read future observations."""
    if profile.shape != (2016, len(panel.links), 3):
        raise ValueError('Queue 미래 과거 평균의 형태·값이 잘못되었습니다.')
    times = pd.date_range(pd.to_datetime(origin, utc=True) + pd.Timedelta(minutes=5), periods=6, freq='5min')
    slot = np.asarray(times.hour * 12 + times.minute // 5)
    weekday = np.asarray(times.dayofweek)
    calendar = np.stack((np.sin(2 * np.pi * slot / 288), np.cos(2 * np.pi * slot / 288),
                         weekday / 6, np.arange(1, 7) / 6), -1)
    prior = profile[weekday * 288 + slot]
    if not np.isfinite(prior).all():
        raise ValueError('Queue 미래 과거 평균에 유효하지 않은 값이 있습니다.')
    return np.concatenate((prior,
                           np.broadcast_to(calendar[:, None], (6, len(panel.links), 4))), -1).astype(np.float32)



def aggregate_channels(day: StationDay, panel: PanelInfo, visible: np.ndarray | None = None):
    values = np.stack((day.speed, day.flow_lane,
                       day.occupancy if day.occupancy is not None else np.full_like(day.speed, np.nan)), -1)
    good = np.isfinite(values)
    if visible is not None:
        good &= visible[..., None] if visible.ndim == 2 else visible
    sums = np.zeros((len(day.timestamps), len(panel.links), 3), np.float32)
    counts = np.zeros_like(sums)
    quality = np.zeros(sums.shape[:2], np.float32)
    qcount = np.zeros_like(quality)
    for station, link in enumerate(day.station_to_link):
        sums[:, link] += np.where(good[:, station], values[:, station], 0)
        counts[:, link] += good[:, station]
        present = good[:, station].any(-1)
        q = day.quality[:, station] if day.quality is not None else present.astype(np.float32)
        # Observation quality remains public even when measured values are masked.
        quality[:, link] += np.where(np.isfinite(q), q, 0)
        qcount[:, link] += np.isfinite(q)
    return sums / np.maximum(counts, 1), counts > 0, quality / np.maximum(qcount, 1)


def features_v2(day: StationDay, panel: PanelInfo, profile: np.ndarray,
                ramp: np.ndarray | None = None, visible: np.ndarray | None = None):
    values, good, quality = aggregate_channels(day, panel, visible)
    steps, nodes = values.shape[:2]
    out = np.zeros((steps, nodes, len(FEATURE_NAMES)), np.float32)
    dates = pd.to_datetime(day.timestamps, utc=True, format="ISO8601")
    slots = np.asarray(dates.hour * 12 + dates.minute // 5)
    dow = dates.dayofweek.to_numpy()
    out[..., :2] = values[..., :2]
    out[..., 2] = good[..., :2].all(-1)
    out[..., 3] = panel.free_speed / 125
    out[..., 4] = panel.lanes / 5
    out[..., 5] = panel.lengths / 5
    out[..., 6] = np.sin(2 * np.pi * slots / 288)[:, None]
    out[..., 7] = np.cos(2 * np.pi * slots / 288)[:, None]
    out[..., 8] = dow[:, None] / 6
    out[..., 9] = values[..., 2]
    out[..., 10:13] = good
    out[..., 13] = quality
    elapsed = np.asarray((dates - dates[0]).total_seconds() / 300, dtype=np.float32)
    for channel in range(3):
        last = np.maximum.accumulate(np.where(good[..., channel], elapsed[:, None], -72), axis=0)
        out[..., 14 + channel] = np.minimum(elapsed[:, None] - last, 72) / 72
    out[..., 17] = (panel.capacity if panel.capacity is not None else np.full(nodes, 6000)) / 10000
    out[..., 18] = panel.position if panel.position is not None else np.arange(nodes) / max(nodes - 1, 1)
    for link, kind in panel.ramp_map.values():
        out[:, link, 19 + kind] = 1
    if ramp is not None:
        out[..., 21:23] = ramp[..., :2] / 600
        out[..., 23:25] = ramp[..., 2:]
    out[..., 25:28] = profile[dow * 288 + slots]
    out[..., 28] = slots[:, None] / 287
    out[..., 29] = dow[:, None] / 6
    out[..., 30] = np.arange(steps)[:, None] / max(steps - 1, 1)
    return out


def queue_features(day, panel, profile, ramp=None, origin=None, changes=False):
    """Build Queue inputs from the supplied past and optional released origin only."""
    x = features_v2(day, panel, profile, ramp)
    if origin is None and not changes:
        return x
    times = pd.to_datetime(day.timestamps, utc=True, format="ISO8601")
    if origin is not None:
        if origin.shape != (1, len(panel.links), len(FEATURE_NAMES)):
            raise ValueError("Unexpected released Queue origin shape")
        x = np.concatenate((x, origin), axis=0)
        times = times.append(pd.DatetimeIndex([times[-1] + pd.Timedelta(minutes=5)]))
        elapsed = np.asarray((times - times[0]).total_seconds() / 300, dtype=np.float32)
        for channel in range(3):
            last = np.maximum.accumulate(np.where(x[..., 10 + channel] > 0, elapsed[:, None], -72), axis=0)
            x[..., 14 + channel] = np.minimum(elapsed[:, None] - last, 72) / 72
        x[..., 30] = np.arange(len(x))[:, None] / (len(x) - 1)
    if not changes:
        return x
    extra = np.zeros((*x.shape[:2], len(QUEUE_CHANGE_NAMES)), np.float32)
    extra[..., 0] = np.where(x[..., 10] > 0, x[..., 0] - panel.cutoffs[None] / 25, 0)
    column = 1
    for lag in (1, 3):
        consecutive = np.asarray(times[lag:] - times[:-lag] == pd.Timedelta(minutes=5 * lag))
        for channel, position in enumerate((0, 1, 9)):
            good = (x[lag:, :, 10 + channel] > 0) & (x[:-lag, :, 10 + channel] > 0) & consecutive[:, None]
            extra[lag:, :, column] = np.where(good, x[lag:, :, position] - x[:-lag, :, position], 0)
            extra[lag:, :, column + 1] = good
            column += 2
    return np.concatenate((x, extra), axis=-1)


def slice_day(day: StationDay, index: np.ndarray) -> StationDay:
    return StationDay([day.timestamps[i] for i in index], day.station_ids, day.station_links,
                      day.station_to_link, day.lanes, day.speed[index], day.flow_lane[index], day.eligible[index],
                      None if day.occupancy is None else day.occupancy[index],
                      None if day.quality is None else day.quality[index])




def ramp_features(release: Path, panel: PanelInfo, split: str, timestamps: list[str]) -> np.ndarray:
    out = np.zeros((len(timestamps), len(panel.links), 4), np.float32)
    out[..., 2:] = 1  # A road without an attached ramp has known zero ramp flow.
    expected = np.zeros((len(panel.links), 2), np.float32)
    for link, kind in panel.ramp_map.values():
        expected[link, kind] += 1
    counts = np.zeros_like(out[..., :2])
    date = timestamps[0][:10]
    root = release / "corridors" / panel.name / split / "ramp_states"
    paths = sorted(set(root.glob(f"**/*{date}*.parquet")) | set(root.glob(f"**/*{date.replace('-', '_')}*.parquet")))
    time_index = {value: i for i, value in enumerate(timestamps)}
    for path in paths:
        frame = pd.read_parquet(path, columns=["timestamp", "ramp_link_id", "flow_vph", "pct_observed"])
        frame = frame[frame.timestamp.isin(time_index)].drop_duplicates(["timestamp", "ramp_link_id"])
        for row in frame.itertuples(index=False):
            if row.ramp_link_id in panel.ramp_map and pd.notna(row.flow_vph) and row.pct_observed >= 75:
                link, kind = panel.ramp_map[row.ramp_link_id]
                index = time_index[row.timestamp]
                out[index, link, kind] += float(row.flow_vph)
                counts[index, link, kind] += 1
    out[..., 2:] = counts >= expected[None]
    out[..., :2] *= out[..., 2:]
    return out


''')


In [ ]:
# @title
# src/fast_model.py — 원본 cdf888214fe8a54b73302f3988b764b74b3f30a4
load_source('fast_model', r'''"""FAST-style traffic models with legacy scan and official Mamba backends."""

from __future__ import annotations

import torch
from torch import nn
from torch.nn import functional as F

from data import QUEUE_FUTURE_NAMES


def temporal_attention(x: torch.Tensor, attention: nn.MultiheadAttention) -> torch.Tensor:
        batch, steps, nodes, width = x.shape
        sequence = x.transpose(1, 2).reshape(batch * nodes, steps, width)
        attended, _ = attention(sequence, sequence, sequence, need_weights=False)
        return attended.reshape(batch, nodes, steps, width).transpose(1, 2)


class SpatialScan(nn.Module):
    def __init__(self, width: int):
        super().__init__()
        self.parameters_for_step = nn.Linear(width, 3 * width)
        self.log_decay = nn.Parameter(torch.zeros(width))
        self.local = nn.Conv1d(width, width, 3, padding=1, groups=width)
        self.project = nn.Linear(width, width)

    def forward(self, x: torch.Tensor, segments: tuple[tuple[int, int], ...], reverse: bool) -> torch.Tensor:
        # Reverse inside each component, then scan all components together.
        order = None
        if reverse:
            order = torch.tensor([i for start, stop in segments for i in range(stop - 1, start - 1, -1)],
                                 device=x.device)
            x = x.index_select(2, order)
        count = x.shape[2]
        starts = torch.tensor([start for start, _ in segments], device=x.device)
        ends = torch.tensor([stop - 1 for _, stop in segments], device=x.device)
        left_mask = torch.ones(count, dtype=x.dtype, device=x.device)
        right_mask = torch.ones(count, dtype=x.dtype, device=x.device)
        left_mask[starts] = 0
        right_mask[ends] = 0

        # Equivalent to the depthwise width-3 convolution, with zeros at each
        # component boundary rather than borrowing a neighbor from another road.
        left = torch.cat((torch.zeros_like(x[:, :, :1]), x[:, :, :-1]), dim=2) * left_mask[None, None, :, None]
        right = torch.cat((x[:, :, 1:], torch.zeros_like(x[:, :, :1])), dim=2) * right_mask[None, None, :, None]
        weights = self.local.weight[:, 0]
        local = left * weights[:, 0] + x * weights[:, 1] + right * weights[:, 2] + self.local.bias

        delta, source, readout = self.parameters_for_step(x).chunk(3, -1)
        decay = torch.exp(-F.softplus(delta) * F.softplus(self.log_decay))
        emission = (1 - decay) * torch.tanh(source)
        decay = decay * left_mask[None, None, :, None]

        # Parallel prefix of affine recurrences: s_i = decay_i * s_(i-1) + emission_i.
        # A zero decay at each start resets the state between components.
        state = emission
        step = 1
        while step < count:
            next_state = state[:, :, step:] + decay[:, :, step:] * state[:, :, :-step]
            next_decay = decay[:, :, step:] * decay[:, :, :-step]
            state = torch.cat((state[:, :, :step], next_state), dim=2)
            decay = torch.cat((decay[:, :, :step], next_decay), dim=2)
            step *= 2
        scanned = state * torch.sigmoid(readout) * torch.sigmoid(local)
        if reverse:
            scanned = scanned.index_select(2, order)
        return self.project(scanned)










class MambaSpatial(nn.Module):
    """Official Mamba over independently padded road components."""
    def __init__(self, width: int, backend: str, state: int = 64):
        super().__init__()
        try:
            if backend == "mamba1":
                from mamba_ssm import Mamba
                self.ssm = Mamba(d_model=width, d_state=state, d_conv=2, expand=1)
            else:
                from mamba_ssm import Mamba2
                # The torch-convolution path also works without causal-conv1d.
                self.ssm = Mamba2(d_model=width, d_state=state, d_conv=2, expand=1,
                                  headdim=min(32, width), chunk_size=32, use_mem_eff_path=False)
        except ImportError as error:
            raise RuntimeError("Install requirements-mamba.txt in WSL; no implicit scan fallback is used") from error
        self.layouts = {}

    def forward(self, x, segments, reverse=False):
        batch, steps, nodes, width = x.shape
        key = segments, reverse, x.device
        if key not in self.layouts:
            buckets = {}
            for start, stop in segments:
                length = 1 << max((stop - start - 1).bit_length(), 0)
                indices = list(range(start, stop))
                if reverse:
                    indices.reverse()
                buckets.setdefault(length, []).append(indices)
            layouts = []
            for length, components in sorted(buckets.items()):
                gather = torch.zeros((len(components), length), dtype=torch.long, device=x.device)
                valid = torch.zeros_like(gather, dtype=torch.bool)
                for row, indices in enumerate(components):
                    gather[row, :len(indices)] = torch.tensor(indices, device=x.device)
                    valid[row, :len(indices)] = True
                layouts.append((gather, valid, gather[valid]))
            self.layouts[key] = layouts
        out = torch.zeros_like(x)
        for gather, valid, destination in self.layouts[key]:
            packed = x[:, :, gather] * valid[None, None, :, :, None]
            packed = packed.reshape(batch * steps * len(gather), gather.shape[1], width).contiguous()
            scanned = self.ssm(packed).reshape(batch, steps, *gather.shape, width)
            out = out.index_copy(2, destination, scanned[:, :, valid].to(out.dtype))
        return out


class GraphResidual(nn.Module):
    def __init__(self, width):
        super().__init__()
        self.up = nn.Linear(width, width, bias=False)
        self.down = nn.Linear(width, width, bias=False)

    def forward(self, x, edges):
        out = torch.zeros_like(x)
        if edges.numel():
            for source, target, projection in ((edges[0], edges[1], self.up), (edges[1], edges[0], self.down)):
                degree = torch.bincount(target, minlength=x.shape[2]).clamp_min(1)
                neighbor = x.index_select(2, source)
                gathered = projection(neighbor) / degree[target][None, None, :, None]
                out = out.index_add(2, target, gathered.to(out.dtype))
        return out


class TrafficBlock(nn.Module):
    def __init__(self, width, heads, dropout, backend, graph):
        super().__init__()
        self.time_a = nn.MultiheadAttention(width, heads, dropout=dropout, batch_first=True)
        self.time_b = nn.MultiheadAttention(width, heads, dropout=dropout, batch_first=True)
        self.ffn_a = nn.Sequential(nn.Linear(width, 2 * width), nn.GELU(), nn.Dropout(dropout), nn.Linear(2 * width, width))
        self.ffn_b = nn.Sequential(nn.Linear(width, 2 * width), nn.GELU(), nn.Dropout(dropout), nn.Linear(2 * width, width))
        self.norms = nn.ModuleList(nn.LayerNorm(width) for _ in range(5))
        cls = (lambda: SpatialScan(width)) if backend == "scan" else (lambda: MambaSpatial(width, backend))
        self.forward_scan, self.reverse_scan = cls(), cls()
        self.graph = GraphResidual(width) if graph else None
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, segments, edges, bidirectional):
        x = self.norms[0](x + self.dropout(temporal_attention(x, self.time_a)))
        x = self.norms[1](x + self.dropout(self.ffn_a(x)))
        spatial = self.forward_scan(x, segments, False)
        if bidirectional:
            spatial = 0.5 * (spatial + self.reverse_scan(x, segments, True))
        if self.graph is not None:
            spatial = spatial + self.graph(x, edges)
        x = self.norms[2](x + self.dropout(spatial))
        x = self.norms[3](x + self.dropout(temporal_attention(x, self.time_b)))
        return self.norms[4](x + self.dropout(self.ffn_b(x)))


def haar_split(x):
    """One orthonormal Haar level on time, replicating an odd final sample."""
    if x.shape[1] % 2:
        x = torch.cat((x, x[:, -1:]), 1)
    even, odd = x[:, ::2], x[:, 1::2]
    return (even + odd) / 2 ** 0.5, (even - odd) / 2 ** 0.5


def haar_merge(low, high, steps):
    return torch.stack(((low + high) / 2 ** 0.5, (low - high) / 2 ** 0.5), 2).flatten(1, 2)[:, :steps]


class WaveletResidual(nn.Module):
    """Learn separate trend/detail filters, then reconstruct every time step."""
    def __init__(self, width, levels):
        super().__init__()
        self.levels = levels
        self.bands = nn.ModuleList(nn.Sequential(
            nn.Conv1d(width, width, 3, padding=1, groups=width), nn.GELU(), nn.Conv1d(width, width, 1)
        ) for _ in range(levels + 1))
        self.gain = nn.Parameter(torch.tensor(-2.0))

    @staticmethod
    def filter(x, band):
        batch, steps, nodes, width = x.shape
        y = band(x.permute(0, 2, 3, 1).reshape(batch * nodes, width, steps))
        return y.reshape(batch, nodes, width, steps).permute(0, 3, 1, 2)

    def forward(self, x):
        details, lengths = [], []
        low = x
        for _ in range(self.levels):
            lengths.append(low.shape[1])
            low, high = haar_split(low)
            details.append(high)
        result = self.filter(low, self.bands[0])
        for level in reversed(range(self.levels)):
            result = haar_merge(result, self.filter(details[level], self.bands[level + 1]), lengths[level])
        return self.gain.sigmoid() * result




class TrafficEncoder(nn.Module):
    def __init__(self, features, node_count, width=64, blocks=2, heads=4,
                 dropout=0.1, backend="mamba2", graph=False, missing_aware=False, graph_mode="mean",
                 adaptive_graph_k=0, wavelet_levels=0, wavelet_input="embedded"):
        super().__init__()
        if width % heads or backend.startswith("mamba") and width % min(width, 32):
            raise ValueError("width must be divisible by attention heads and the Mamba head dimension")
        # Former default fields may still be present in existing Queue checkpoints.
        if missing_aware or graph_mode != "mean" or adaptive_graph_k:
            raise ValueError('제거된 Queue 구조 실험이 활성화된 체크포인트는 지원하지 않습니다.')
        if not 0 <= wavelet_levels <= 3:
            raise ValueError('wavelet-levels는 0~3이어야 합니다.')
        if wavelet_levels and features < 31:
            raise ValueError('Wavelet은 v2 관측·과거 평균 특징이 필요합니다.')
        if wavelet_input not in ("embedded", "signals"):
            raise ValueError("Unknown wavelet input")
        # Configs without this field retain the original checkpoint architecture.
        self.wavelet_input = wavelet_input
        self.input = nn.Linear(features, width)
        self.node = nn.Embedding(node_count + 1, width, padding_idx=0)
        self.slot = nn.Embedding(288, width)
        self.weekday = nn.Embedding(7, width)
        self.relative = nn.Embedding(72, width)
        self.blocks = nn.ModuleList(TrafficBlock(width, heads, dropout, backend, graph) for _ in range(blocks))
        self.skip = nn.Linear(width * blocks, width)
        self.dropout = nn.Dropout(dropout)
        self.wavelet = WaveletResidual(3 if wavelet_input == "signals" else width, wavelet_levels) if wavelet_levels else None
        self.wavelet_project = nn.Linear(3, width, bias=False) if wavelet_levels and wavelet_input == "signals" else None

    def project_input(self, x, fill_missing=False):
        if self.wavelet is None:
            return self.input(x)
        masks = x[..., 10:13].clamp(0, 1)
        clean = x.clone()
        for channel, position in enumerate((0, 1, 9)):
            fallback = x[..., 25 + channel] if fill_missing else 0
            clean[..., position] = torch.where(masks[..., channel] > 0, x[..., position], fallback)
        return self.input(clean)

    def forward(self, x, node_ids, segments, edges, bidirectional=True):
        steps = x.shape[1]
        if steps > 72:
            raise ValueError("The input window must contain at most 72 steps")
        slot = (x[..., 28] * 287).round().long().clamp(0, 287)
        weekday = (x[..., 29] * 6).round().long().clamp(0, 6)
        projected = self.project_input(x)
        identities = self.node(node_ids)
        embedding = (identities[None, None].to(projected.dtype) + self.slot(slot).to(projected.dtype)
                     + self.weekday(weekday).to(projected.dtype)
                     + self.relative(torch.arange(steps, device=x.device))[None, :, None].to(projected.dtype))
        hidden = self.dropout(projected + embedding)
        if self.wavelet is not None:
            if self.wavelet_input == "signals":
                # DWT sees only continuous measurements; masks and embeddings stay in the main path.
                signals = torch.where(x[..., 10:13] > 0, x[..., (0, 1, 9)], x[..., 25:28])
                hidden = hidden + self.wavelet_project(self.wavelet(signals))
            else:
                hidden = hidden + self.wavelet(self.project_input(x, fill_missing=True) + embedding)
        outputs = []
        for block in self.blocks:
            hidden = block(hidden, segments, edges, bidirectional)
            outputs.append(hidden)
        return self.skip(torch.cat(outputs, -1))




class TrafficQueueModel(nn.Module):
    def __init__(self, station_decoder="linear", queue_head="linear", queue_changes=False, queue_future_prior=False, **config):
        super().__init__()
        if queue_head not in ("linear", "pooled-mlp", "conditional"):
            raise ValueError("Unknown Queue head")
        self.queue_head = queue_head
        self.encoder = TrafficEncoder(**config)
        width = config.get("width", 64)
        def head():
            return nn.Sequential(nn.Linear(2 * width, width), nn.GELU(),
                                 nn.Dropout(config.get("dropout", 0.1)), nn.Linear(width, 12))
        self.head = (nn.Linear(width, 12) if queue_head == 'linear' else
                     nn.ModuleList([head(), head()]) if queue_head == 'conditional' else head())
        if queue_head == 'conditional':
            self.head[1].load_state_dict(self.head[0].state_dict())
        self.queue_future_prior = queue_future_prior
        if queue_future_prior:
            self.future_head = nn.Sequential(nn.Linear(len(QUEUE_FUTURE_NAMES), width), nn.GELU(), nn.Linear(width, 2))
            nn.init.zeros_(self.future_head[-1].weight)
            nn.init.zeros_(self.future_head[-1].bias)

    def forward(self, x, node_ids, segments, edges, bidirectional=True, condition=None, future=None):
        history = self.encoder(x, node_ids, segments, edges, bidirectional)
        hidden = history[:, -1]
        if self.queue_head in ('pooled-mlp', 'conditional'):
            hidden = torch.cat((hidden, history.mean(1)), dim=-1)
        if self.queue_head == 'conditional':
            if condition is None or condition.shape != (len(x),) or ((condition < 0) | (condition > 1)).any():
                raise ValueError('Queue 조건은 창별 onset=0/ongoing=1이어야 합니다.')
            choices = torch.stack([head(hidden) for head in self.head], 1)
            output = choices[torch.arange(len(x), device=x.device), condition.long()]
        else:
            output = self.head(hidden)
        output = output.reshape(hidden.shape[0], hidden.shape[1], 6, 2).transpose(1, 2)
        if self.queue_future_prior:
            if future is None or future.shape != (*output.shape[:3], len(QUEUE_FUTURE_NAMES)):
                raise ValueError('Queue 미래 시간·과거 평균 입력이 잘못되었습니다.')
            output = output + self.future_head(future)
        return output[..., 0], F.softplus(output[..., 1])
''')


In [ ]:
# @title
# 내장 learning.py — cdf8882 기반 Colab Task 2·3 개선본
load_source('learning', r'''"""Queue training, proxy validation, calibration, and template prediction."""
from __future__ import annotations

from concurrent.futures import ThreadPoolExecutor
from functools import lru_cache
import hashlib
from types import SimpleNamespace
import itertools
import json
from pathlib import Path
import re
import sys
import time

import numpy as np
import pandas as pd
import torch
from torch.nn import functional as F

from data import (PanelInfo, StationDay, aggregate_channels,
                  features_v2, queue_features, queue_feature_names, queue_future_features, QUEUE_FUTURE_NAMES,
                  ramp_features, slice_day)
from fast_model import TrafficQueueModel



# 공식 공개 설정 확인일: 2026-10-28. 관측 속도 라벨을 쓰는 로컬 대리 검증입니다.
QUEUE_PUBLIC_CONTRACT = dict(minimum_history_coverage=0.7, maximum_persistence_iou=0.9,
                             minimum_spacing_minutes=360, requires_queue_in_horizon=True)


def checkpoint_bf16(checkpoint, device, strict=False):
    wanted = bool(checkpoint['training_args'].get('use_bf16', False))
    supported = device == 'cuda' and torch.cuda.is_bf16_supported()
    if strict and wanted and not supported:
        raise ValueError('이 Queue 체크포인트의 BF16 재개에는 BF16 지원 GPU가 필요합니다.')
    return wanted and supported


def queue_condition(day, panel, nodes=None):
    nodes = np.arange(len(panel.links)) if nodes is None else nodes
    return 'ongoing' if ((day.speed[:, nodes] * 25 <= panel.cutoffs[None, nodes]) & day.eligible[:, nodes]).any() else 'onset'


def queue_decisions(probability, threshold, valid, force_nonempty=False):
    probability, valid = np.asarray(probability), np.asarray(valid, bool)
    if probability.shape != valid.shape or not np.isfinite(probability).all() or not 0 <= threshold <= 1:
        raise ValueError('Queue 확률·임계값·대상 마스크가 잘못되었습니다.')
    predicted = (probability >= threshold) & valid
    if force_nonempty and valid.any() and not predicted.any():
        best = np.where(valid, probability, -np.inf).argmax()
        predicted.flat[best] = True
    return predicted


def stable_seed(*parts):
    return int.from_bytes(hashlib.blake2b("|".join(map(str, parts)).encode(), digest_size=8).digest(), "big")


@lru_cache(maxsize=64)
def file_index(release: Path, name: str):
    result = {}
    for path in sorted((release / "corridors" / name / "train" / "mainline_states").glob("**/*.parquet")):
        match = re.search(r"\d{4}[-_]\d{2}[-_]\d{2}", path.name)
        date = match.group().replace("_", "-") if match else str(pd.read_parquet(path, columns=["timestamp"]).timestamp.iloc[0])[:10]
        if date in result:
            raise ValueError(f"Duplicate day: {name} {date}")
        result[date] = path
    return result


def period_files(release, name, period, maximum=None):
    limits = {"fit": ("2030-06-01", "2030-12-31"), "select": ("2031-01-01", "2031-01-31"),
              "holdout": ("2031-02-01", "2031-02-28")}
    low, high = limits[period]
    paths = [path for date, path in file_index(release, name).items() if low <= date <= high]
    return paths[:maximum] if maximum else paths


def signature(paths):
    return hashlib.sha256(json.dumps([(p.name, p.stat().st_size, p.stat().st_mtime_ns) for p in paths]).encode()).hexdigest()


def fit_profile(release, panel, paths):
    """Fit-only weekday/slot means; the checkpoint embeds the resulting array."""
    key = hashlib.sha256(("eligible-v2|" + signature(paths) + json.dumps(panel.links)).encode()).hexdigest()[:16]
    folder = release.parent / "processed_v2"
    folder.mkdir(parents=True, exist_ok=True)
    cache = folder / f"{panel.name}-profile-{key}.npy"
    if cache.exists():
        return np.load(cache, allow_pickle=False)
    print(f"Preparing fit-only means: {panel.name} days={len(paths)}", flush=True)
    sums = np.zeros((2016, len(panel.links), 3), np.float64)
    counts = np.zeros_like(sums)
    for number, path in enumerate(paths, 1):
        day = StationDay.load(path, panel)
        values, valid, _ = aggregate_channels(day, panel, day.eligible)
        dates = pd.to_datetime(day.timestamps, utc=True)
        slots = np.asarray(dates.dayofweek * 288 + dates.hour * 12 + dates.minute // 5)
        np.add.at(sums, slots, values * valid)
        np.add.at(counts, slots, valid)
        if number % 50 == 0:
            print(f"fit-only means {panel.name}: {number}/{len(paths)}", flush=True)
    global_mean = sums.sum((0, 1)) / np.maximum(counts.sum((0, 1)), 1)
    link_mean = np.divide(sums.sum(0), counts.sum(0), out=np.broadcast_to(global_mean, sums.shape[1:]).copy(), where=counts.sum(0) > 0)
    result = np.divide(sums, counts, out=np.broadcast_to(link_mean, sums.shape).copy(), where=counts > 0).astype(np.float32)
    temporary = cache.with_suffix(".pending.npy")
    np.save(temporary, result)
    temporary.replace(cache)
    return result






@lru_cache(maxsize=32)
def masked_file_index(release, name, split):
    folder = release / "corridors" / name / split / "mainline_states_masked"
    if not folder.is_dir():
        raise FileNotFoundError(f"Released Queue origin needs masked inputs: {folder}")
    result = {}
    for path in sorted(folder.glob("**/*.parquet")):
        match = re.search(r"\d{4}[-_]\d{2}[-_]\d{2}", path.name)
        if not match:
            raise ValueError(f"Masked file has no date: {path}")
        date = match.group().replace("_", "-")
        if date in result:
            raise ValueError(f"Duplicate masked day: {name} {split} {date}")
        result[date] = path
    return result


def released_origin_features(context, timestamp, ramp, cache):
    """Read exactly T from the released masked layer, never fill it from truth."""
    panel, date = context["panel"], timestamp[:10]
    missing = np.full((1, len(panel.links)), np.nan, np.float32)
    origin = StationDay([timestamp], panel.links, panel.links, np.arange(len(panel.links)), panel.lanes,
                        missing, missing.copy(), np.zeros_like(missing, bool), missing.copy(), np.zeros_like(missing))
    path = context["origin_paths"].get(date)
    if path is not None:
        if date not in cache:
            cache[date] = StationDay.load(path, panel)
        day = cache[date]
        if timestamp in day.timestamps:
            origin = slice_day(day, np.array([day.timestamps.index(timestamp)]))
    return features_v2(origin, panel, context["profile"], ramp)


def queue_label_probability(speed, cutoffs, margin):
    if not np.isfinite(margin) or margin <= 0:
        raise ValueError('Queue 확률 라벨의 완화 폭은 양수여야 합니다.')
    return np.clip(.5 + (cutoffs - speed) / (2 * margin), 0, 1).astype(np.float32)


def choose_queue_origins(candidates, args, name, path, period, epoch):
    public = period != 'fit' and getattr(args, 'queue_public_contract', False)
    seed_parts = (args.seed, name, path.name, 'queue')
    if period == 'fit' and getattr(args, 'queue_resample', False):
        seed_parts += (epoch,)
    rng = np.random.default_rng(stable_seed(*seed_parts))
    condition_gap = getattr(args, 'queue_condition_gap', False) if period == 'fit' else (public or getattr(args, 'queue_validation_spread', False))
    spacing = QUEUE_PUBLIC_CONTRACT['minimum_spacing_minutes'] if public else args.window_gap_minutes
    chosen = []
    for kind in ('onset', 'ongoing', 'normal'):
        limit = len(candidates[kind]) if public else (args.normal_windows if kind == 'normal' else args.windows_per_day)
        for origin in rng.permutation(candidates[kind]):
            if sum(k == kind for k, _ in chosen) >= limit:
                break
            if all(abs(int(origin) - old) * 5 >= spacing
                   for other_kind, old in chosen if not condition_gap or other_kind == kind):
                chosen.append((kind, int(origin)))
    return chosen


def queue_examples(context, path, args, period, epoch=0, selection_only=False, selected=None):
    panel = context["panel"]
    day = StationDay.load(path, panel)
    steps, nodes = len(day.timestamps), len(panel.links)
    values, _, _ = aggregate_channels(day, panel, day.eligible)
    speed = values[..., 0]
    minimum = np.full((steps, nodes), np.inf, np.float32)
    valid = np.zeros_like(minimum, bool)
    for station, link in enumerate(day.station_to_link):
        good = day.eligible[:, station]
        minimum[:, link] = np.minimum(minimum[:, link], np.where(good, day.speed[:, station] * 25, np.inf))
        valid[:, link] |= good
    queues = (minimum <= panel.cutoffs) & valid  # Any eligible station, not mean speed.
    public_contract = period != "fit" and getattr(args, "queue_public_contract", False)
    target_sampling = getattr(args, "queue_sampling_version", 2) >= 3
    detector_links = np.unique(day.station_to_link)
    target_nodes = context.get('queue_target_nodes', detector_links)
    target_mask = np.zeros((6, nodes), dtype=bool)
    target_mask[:, target_nodes] = True
    coverage_nodes = np.intersect1d(detector_links, target_nodes)
    if (public_contract or target_sampling) and not len(coverage_nodes):
        raise ValueError('Queue 대상에 관측 링크가 없습니다.')
    candidates = {"onset": [], "ongoing": [], "normal": []}
    for origin in range(12, steps - 6):
        history, future = queues[origin - 12:origin], queues[origin + 1:origin + 7]
        if public_contract:
            if valid[origin - 12:origin, coverage_nodes].mean() < QUEUE_PUBLIC_CONTRACT['minimum_history_coverage'] or not (future & target_mask).any():
                continue
            persistence = np.broadcast_to(queues[origin - 1], future.shape)
            if iou(persistence, future, valid[origin + 1:origin + 7] & target_mask) > QUEUE_PUBLIC_CONTRACT['maximum_persistence_iou']:
                continue
            kind = "onset" if not history[:, target_nodes].any() else "ongoing"
        elif target_sampling:
            if valid[origin - 12:origin, coverage_nodes].mean() < args.history_coverage or valid[origin + 1:origin + 7, coverage_nodes].mean() < args.future_coverage:
                continue
            kind = ("onset" if not history[:, target_nodes].any() else "ongoing") if future[:, target_nodes].any() else "normal"
        else:
            # Preserve the sampling of checkpoints created before version 3.
            if valid[origin - 12:origin].mean() < args.history_coverage or valid[origin + 1:origin + 7].mean() < args.future_coverage:
                continue
            kind = ("onset" if not history.any() else "ongoing" if history[-1].any() else None) if future.any() else "normal"
        if kind is not None:
            candidates[kind].append(origin)
    if selection_only:
        return candidates
    chosen = choose_queue_origins(candidates, args, panel.name, path, period, epoch) if selected is None else selected
    if any(kind not in candidates or origin not in candidates[kind] for kind, origin in chosen):
        raise ValueError('Queue 균등 표본이 현재 입력의 적격 창과 다릅니다.')
    ramp = ramp_features(context["release"], panel, "train", day.timestamps)
    samples, origin_cache = {}, {}
    for kind, origin in dict.fromkeys(chosen):
        index = np.arange(origin - 12, origin)
        future = slice(origin + 1, origin + 7)
        released = (released_origin_features(context, day.timestamps[origin], ramp[origin:origin + 1], origin_cache)
                    if getattr(args, "queue_origin", False) else None)
        x = queue_features(slice_day(day, index), panel, context["profile"], ramp[index], released,
                           getattr(args, "queue_changes", False))
        hard = queues[future].astype(np.float32)
        weight = valid[future].astype(np.float32)
        sample = {"x": x, "truth": hard, "soft": hard, "speed": speed[future],
                        "mask": valid[future] & target_mask if public_contract or target_sampling else valid[future],
                        "target_mask": target_mask, "weight": weight,
                        "baseline": np.broadcast_to(queues[origin - 1], hard.shape).copy(),
                        "condition": kind, "date": day.timestamps[0][:10], "origin": origin,
                        "forecast_origin": day.timestamps[origin],
                        "branch": np.array(int(queues[origin - 12:origin, target_nodes].any()), np.int64)}
        if getattr(args, 'queue_future_prior', False):
            sample['future'] = queue_future_features(day.timestamps[origin], panel, context['profile'])
        mode = getattr(args, 'queue_label_mode', 'hard')
        if mode != 'hard':
            label = hard.copy()
            if period == 'fit':
                probability = queue_label_probability(minimum[future], panel.cutoffs, args.queue_label_margin)
                label_rng = np.random.default_rng(stable_seed(args.seed, panel.name, path.name, origin, epoch, 'queue-label'))
                label = ((label_rng.random(hard.shape) < probability).astype(np.float32)
                         if mode == 'sample' else probability)
            sample['train_target'] = label
        samples[kind, origin] = sample
    return [samples[kind, origin] for kind, origin in chosen]


def tensor_batch(samples, keys, device):
    out = {}
    for key in keys:
        value = torch.from_numpy(np.stack([sample[key] for sample in samples]))
        out[key] = value.pin_memory().to(device, non_blocking=True) if device == "cuda" else value
    return out






def queue_loss(logits, speeds, batch, conditions, args, condition_counts=None):
    logits, speeds = logits.float(), speeds.float()
    weights = batch["weight"] * batch["mask"]
    target = batch.get('train_target', batch['soft'])
    bce = (F.binary_cross_entropy_with_logits(logits, target, reduction="none") * weights).sum((1, 2)) / weights.sum((1, 2)).clamp_min(1)
    probability = logits.sigmoid() * batch["mask"]
    truth = batch.get('train_target', batch['truth']) * batch["mask"]
    confidence = batch['weight'] if 'train_target' in batch else torch.ones_like(truth)
    intersection = (confidence * probability * truth).sum((1, 2))
    union = (confidence * (probability + truth - probability * truth)).sum((1, 2))
    soft_iou = (1 - (intersection + 1e-6) / (union + 1e-6)) * (truth.sum((1, 2)) > 0)
    speed = (F.smooth_l1_loss(speeds, batch["speed"], reduction="none") * batch["mask"]).sum((1, 2)) / batch["mask"].sum((1, 2)).clamp_min(1)
    losses = bce + args.soft_iou_weight * soft_iou + args.speed_weight * speed
    if getattr(args, 'queue_balanced', False):
        denominator = sum(condition_counts.values()) if condition_counts is not None else len(losses)
        return losses.sum() / max(denominator, 1)
    groups = []
    for kind in ("onset", "ongoing", "normal"):
        indices = [i for i, condition in enumerate(conditions) if condition == kind]
        if indices:
            value = losses[indices].mean() if condition_counts is None else losses[indices].sum() / condition_counts[kind]
            groups.append(value * (args.normal_weight if kind == "normal" else 1.0))
    return torch.stack(groups).mean() if condition_counts is None else torch.stack(groups).sum() / len(condition_counts)


def family_mean(scores):
    families = {}
    for panel, score in scores.items():
        families.setdefault(panel.rsplit("_", 1)[0], []).append(score)
    return float(np.mean([np.mean(values) for values in families.values()])) if families else 0.0




def iou(predicted, truth, mask):
    predicted, truth = predicted & mask, truth.astype(bool) & mask
    union = np.logical_or(predicted, truth).sum()
    return float(np.logical_and(predicted, truth).sum() / union) if union else 0.0


def queue_metrics(records, threshold, baseline=False):
    groups = {}
    normal = []
    for row in records:
        score = row["persistence_iou"] if baseline else row["ious"][str(threshold)]
        if row["condition"] == "normal":
            normal.append(row["persistence_false_positive"] if baseline else row["normal_false_positive"][str(threshold)])
        else:
            groups.setdefault((row["panel"], row["condition"]), []).append(score)
    panels = {row["panel"] for row in records}
    by_condition = {kind: {panel: float(np.mean(groups.get((panel, kind), [0]))) for panel in panels} for kind in ("onset", "ongoing")}
    scores = {panel: 0.5 * (by_condition["onset"][panel] + by_condition["ongoing"][panel]) for panel in panels}
    return {"proxy_IoU": family_mean(scores), "onset": family_mean(by_condition["onset"]),
            "ongoing": family_mean(by_condition["ongoing"]), "panels": scores,
            "condition_counts": {kind: sum(row["condition"] == kind for row in records) for kind in ("onset", "ongoing", "normal")},
            "normal_false_positive": float(np.mean(normal)) if normal else None}


def write_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    pending = path.with_suffix(path.suffix + ".pending")
    pending.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    pending.replace(path)


def event(path, **values):
    values = {"time_utc": pd.Timestamp.now(tz="UTC").isoformat(), **values}
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(values, ensure_ascii=False, allow_nan=False) + "\n")
    print(json.dumps(values, ensure_ascii=False), flush=True)


def model_from_checkpoint(checkpoint, device):
    if checkpoint.get('format_version') != 2 or checkpoint.get('task') != 'queue':
        raise ValueError('현재 v2 Queue 체크포인트만 지원합니다.')
    expected = queue_feature_names(checkpoint['model_config'].get('queue_changes', False))
    if tuple(checkpoint['feature_names']) != expected:
        raise ValueError('Queue 체크포인트의 과거 입력 특징이 다릅니다.')
    if checkpoint['model_config'].get('queue_future_prior', False) and tuple(checkpoint.get('queue_future_names', ())) != QUEUE_FUTURE_NAMES:
        raise ValueError('Queue 미래 시간·과거 평균 계약이 다릅니다.')
    model = TrafficQueueModel(**checkpoint['model_config']).to(device)
    model.load_state_dict(checkpoint['weights'])
    return model


def initialize_queue(model, checkpoint):
    """Clone an existing pooled head into both branches; new priors start at zero."""
    if checkpoint.get('format_version') != 2 or checkpoint.get('task') != 'queue':
        raise ValueError('초기화에는 v2 Queue 체크포인트가 필요합니다.')
    source = checkpoint['model_config'].get('queue_head', 'linear')
    if source != model.queue_head and (source, model.queue_head) != ('pooled-mlp', 'conditional'):
        raise ValueError('Queue 출력층 초기화는 같은 구조 또는 pooled-mlp → conditional만 지원합니다.')
    weights = checkpoint['weights']
    def part(prefix):
        return {key[len(prefix):]: value for key, value in weights.items() if key.startswith(prefix)}
    model.encoder.load_state_dict(part('encoder.'))
    if source == 'pooled-mlp' and model.queue_head == 'conditional':
        for head in model.head:
            head.load_state_dict(part('head.'))
    else:
        model.head.load_state_dict(part('head.'))
    if model.queue_future_prior and checkpoint['model_config'].get('queue_future_prior', False):
        if tuple(checkpoint.get('queue_future_names', ())) != QUEUE_FUTURE_NAMES:
            raise ValueError('초기 Queue 미래 과거 평균 계약이 다릅니다.')
        model.future_head.load_state_dict(part('future_head.'))


def check_model_requests(args, config, queue=True):
    for key in ("graph",):
        if getattr(args, key, False) and not config.get(key, False):
            raise ValueError(f"The checkpoint has {key}=False; use a new checkpoint to change the model")
    for key, default in (("wavelet_levels", 0),):
        requested = getattr(args, key, default)
        if requested != default and requested != config.get(key, default):
            raise ValueError(f"The checkpoint has a different {key}; use a new checkpoint to change the model")
    if getattr(args, "wavelet_input", None) is not None and args.wavelet_input != config.get("wavelet_input", "embedded"):
        raise ValueError("The checkpoint has a different wavelet_input; use a new checkpoint to change the model")
    if queue:
        for key, default in (("queue_head", "linear"), ("queue_changes", False), ('queue_future_prior', False)):
            requested = getattr(args, key, default)
            if requested != default and requested != config.get(key, default):
                raise ValueError(f"The checkpoint has a different {key}; use a new checkpoint to change the model")




def contexts(args, task, checkpoint=None):
    if task != 'queue':
        raise ValueError('Task 1은 MagiNet을 사용합니다. 이 경로는 Queue 전용입니다.')
    release = args.release_root.resolve()
    available = sorted(path.name for path in (release / "corridors").iterdir() if path.is_dir())
    names = sorted(set(args.panel or (checkpoint["trained_panels"] if checkpoint else available)))
    if set(names) - set(available):
        raise ValueError(f"Unknown panels: {set(names) - set(available)}")
    names = [name for name in names if name not in {"D12_I405_N", "D12_I405_S"}]
    if not names:
        raise ValueError("No panels for this task")
    result = {}
    for name in names:
        panel = PanelInfo.load(release, name)
        fit_paths = period_files(release, name, "fit", args.max_days)
        if not fit_paths:
            raise ValueError(f"No fit days for {name}")
        if checkpoint:
            if name not in checkpoint["profiles"]:
                raise ValueError(f"Checkpoint has no fit-only profile for {name}")
            profile = checkpoint["profiles"][name].cpu().numpy()
            if checkpoint["links"][name] != panel.links:
                raise ValueError(f"Road layout changed: {name}")
        else:
            profile = fit_profile(release, panel, fit_paths)
        context = {"panel": panel, "release": release, "profile": profile, "fit_paths": fit_paths}
        if task == 'queue' and (getattr(args, 'queue_public_contract', False)
                                or getattr(args, 'queue_sampling_version', 2) >= 3):
            template = release / 'task2' / name / 'validation/sample_submission_queue.csv'
            if template.is_file():
                targets = set(pd.read_csv(template, usecols=['link_id'], dtype=str).link_id)
                if not targets or not targets <= set(panel.links):
                    raise ValueError(f'알 수 없는 Queue 대상 링크: {name}')
                context['queue_target_nodes'] = np.array([panel.link_index[link] for link in sorted(targets)], dtype=np.int64)
        if task == "queue" and getattr(args, "queue_origin", False):
            context["origin_paths"] = masked_file_index(release, name, "train")
            fit_set = set(fit_paths)
            context["origin_fit_paths"] = [context["origin_paths"][date] for date, path in sorted(file_index(release, name).items())
                                           if path in fit_set and date in context["origin_paths"]]
        result[name] = context
    return result


def schedule(context, period, args, epoch):
    rng = np.random.default_rng(stable_seed(args.seed, epoch, period))
    names = list(context)
    if period == "fit":
        rng.shuffle(names)
    paths = []
    for name in names:
        days = period_files(args.release_root, name, period, args.max_days)
        if period == "fit":
            rng.shuffle(days)
        elif getattr(args, "queue_validation_spread", False) and days:
            # Visit evenly spaced dates first; run_period takes at most one window per condition/day.
            first = np.linspace(0, len(days) - 1, min(len(days), args.validation_windows)).round().astype(int).tolist()
            chosen = set(first)
            days = [days[i] for i in first] + [path for i, path in enumerate(days) if i not in chosen]
        paths.append([(name, path) for path in days])
    return [item for row in itertools.zip_longest(*paths) for item in row if item is not None]


def balanced_queue_items(context, args, epoch):
    """Cache only candidate positions, then sample equal positive-condition quotas."""
    paths = [(name, path) for name in context for path in context[name]['fit_paths']]
    network = [p for name in context for p in (args.release_root / 'corridors' / name / 'network').glob('*.csv')]
    config = dict(version='queue-balanced-v1', source=globals().get('__source_sha256__', 'local'),
                  files=[[name, str(path), path.stat().st_size, path.stat().st_mtime_ns] for name, path in paths],
                  network=signature(sorted(network)), history=args.history_coverage, future=args.future_coverage,
                  targets={name: list(map(int, item.get('queue_target_nodes', []))) for name, item in context.items()},
                  sampling=args.queue_sampling_version)
    key = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:20]
    marker = args.release_root.parent / 'processed_v2' / f'queue-candidates-{key}.json'
    if marker.is_file():
        saved = json.loads(marker.read_text())
        if saved['config'] != config:
            raise ValueError('Queue 후보 창 캐시가 현재 데이터와 다릅니다.')
        days = saved['days']
    else:
        days = []
        for index, (name, path) in enumerate(paths, 1):
            days.append(dict(panel=name, path=str(path), candidates=queue_examples(
                context[name], path, args, 'fit', selection_only=True)))
            if index == 1 or index % 100 == 0 or index == len(paths):
                print(f'Queue 균등 학습 후보 준비: {index}/{len(paths)}일', flush=True)
        write_json(marker, dict(config=config, days=days))
    groups = {(name, kind): [] for name in context for kind in ('onset', 'ongoing')}
    for day in days:
        for kind, origin in choose_queue_origins(day['candidates'], args, day['panel'], Path(day['path']), 'fit', epoch):
            if kind in ('onset', 'ongoing'):
                groups[day['panel'], kind].append((day['path'], origin))
    missing = [key for key, values in groups.items() if not values]
    if missing:
        raise ValueError(f'균등 학습에 필요한 fit 정체 창이 없습니다: {missing}')
    unit = args.update_windows or 48
    quota = max(unit, int(np.ceil(sum(map(len, groups.values())) / len(groups) / unit)) * unit)
    rng = np.random.default_rng(stable_seed(args.seed, epoch, 'queue-balanced'))
    by_day = {}
    for (name, kind), values in groups.items():
        for index in rng.choice(len(values), quota, replace=len(values) < quota):
            path, origin = values[int(index)]
            by_day.setdefault((name, path), []).append((kind, origin))
    items = [(name, Path(path), selected) for (name, path), selected in by_day.items()]
    rng.shuffle(items)
    print({'queue_windows_per_panel_condition': quota, 'groups': len(groups), 'unique_days': len(items)}, flush=True)
    return items


def prepared_days(context, items, args, task, period, epoch=0):
    def prepare(item):
        name, path = item[:2]
        start = time.perf_counter()
        samples = (queue_examples(context[name], path, args, period, epoch,
                                  selected=item[2] if len(item) > 2 else None))
        return name, path, samples, time.perf_counter() - start
    if args.workers == 0:
        for item in items:
            yield prepare(item)
    else:
        # Bounded queue: do not retain the whole dataset in WSL's small RAM.
        with ThreadPoolExecutor(max_workers=args.workers) as pool:
            iterator = iter(items)
            pending = [pool.submit(prepare, item) for item in itertools.islice(iterator, args.workers + 1)]
            while pending:
                future = pending.pop(0)
                result = future.result()
                item = next(iterator, None)
                if item is not None:
                    pending.append(pool.submit(prepare, item))
                yield result


def batch_size(args, name, nodes, task):
    overrides = dict(value.split("=", 1) for value in args.panel_batch)
    if name in overrides:
        size = int(overrides[name])
    elif args.batch_size:
        size = args.batch_size
    else:
        size = 48 if nodes <= 128 else 12
    if size < 1:
        raise ValueError("Batch size must be positive")
    return size


def update_size(args, name, nodes, task):
    requested = getattr(args, "update_windows", None)
    if requested is None:
        return batch_size(args, name, nodes, task)
    if requested < 0:
        raise ValueError("--update-windows must be nonnegative")
    return requested or (48)


def device_layout(context, vocabulary, device):
    for name, item in context.items():
        panel = item["panel"]
        item["node_ids"] = torch.tensor([vocabulary.get(f"{name}|{link}", 0) for link in panel.links], device=device)
        item["edges"] = torch.as_tensor(panel.edges, device=device)


def run_period(model, context, vocabulary, args, task, period, epoch, device, optimizer=None, log=None, global_step=0, queue_store=None):
    if task != 'queue':
        raise ValueError('Task 1은 MagiNet을 사용합니다. 이 경로는 Queue 전용입니다.')
    training = optimizer is not None
    model.train(training)
    records = []
    pending = {name: [] for name in context}
    totals = torch.zeros(2, device=device)
    seen_conditions = {name: {kind: 0 for kind in ("onset", "ongoing", "normal")} for name in context}
    accepted_origins = {}
    fixed_policy = getattr(args, 'fixed_policy', None)
    thresholds = tuple(getattr(args, 'queue_thresholds', (.1, .2, .3, .4, .5, .6, .7, .8, .9)))
    if period == "holdout" or fixed_policy is not None:
        thresholds = (args.fixed_threshold,)
    policy_rows = []
    started = time.perf_counter()
    preparation_seconds = 0.0
    windows = 0
    updates_by_panel = {name: 0 for name in context}
    training_groups, label_changes = {}, 0
    device_layout(context, vocabulary, device)

    def process_batch(name, samples, condition_counts=None):
        nonlocal windows, label_changes
        panel = context[name]["panel"]
        if log and windows == 0:
            event(log, event="first_batch", epoch=epoch + 1, period=period, panel=name,
                  batch_size=len(samples), backend=args.model_config["backend"],
                  note="The first CUDA call may compile Triton kernels")
        keys = ['x', 'truth', 'soft', 'speed', 'mask', 'weight']
        if args.model_config.get('queue_head') == 'conditional':
            keys.append('branch')
        if args.model_config.get('queue_future_prior', False):
            keys.append('future')
        if getattr(args, 'queue_label_mode', 'hard') != 'hard':
            keys.append('train_target')
        batch = tensor_batch(samples, keys, device)
        if training:
            for sample in samples:
                key = f"{name}:{sample['condition']}"
                training_groups[key] = training_groups.get(key, 0) + 1
            if 'train_target' in batch:
                label_changes += int(((batch['train_target'] != batch['truth']) & batch['mask']).sum())
        with torch.set_grad_enabled(training):
            with torch.autocast(device, dtype=torch.bfloat16, enabled=args.use_bf16):
                logits, speeds = model(batch['x'], context[name]['node_ids'], panel.segments,
                                       context[name]['edges'], args.bidirectional,
                                       condition=batch.get('branch'), future=batch.get('future'))
            loss = (queue_loss(logits, speeds, batch, [s["condition"] for s in samples], args, condition_counts))
            if training:
                loss.backward()
        windows += len(samples)
        if not training:
            probabilities, future_speed = logits.float().sigmoid().cpu().numpy(), speeds.float().cpu().numpy()
            for sample, probability, predicted_speed in zip(samples, probabilities, future_speed):
                record = {"panel": name, "date": sample["date"], "condition": sample["condition"], "origin": sample["origin"],
                          "persistence_iou": iou(sample["baseline"], sample["truth"], sample["mask"]),
                          "persistence_false_positive": float((sample["baseline"] & sample["mask"]).sum() / max(sample["mask"].sum(), 1)),
                          "ious": {str(t): iou(probability >= t, sample["truth"], sample["mask"]) for t in thresholds},
                          "normal_false_positive": {str(t): float(((probability >= t) & sample["mask"]).sum() / max(sample["mask"].sum(), 1)) for t in thresholds},
                          "speed_sse_kmh": float((((predicted_speed - sample["speed"]) * 25) ** 2)[sample["mask"]].sum()),
                          "count": int(sample["mask"].sum())}
                records.append(record)
                policy_rows.append(dict(panel=name, condition=sample['condition'], date=sample['date'],
                                        origin=sample['forecast_origin'], probability=probability,
                                        truth=sample['truth'].astype(bool), valid=sample['mask'].astype(bool),
                                        target_mask=sample['target_mask'], baseline=sample['baseline'].astype(bool)))
                if queue_store is not None:
                    queue_store.append(policy_rows[-1])
        return loss.detach()

    def process(name, samples):
        nonlocal global_step
        count = len(samples)
        size = batch_size(args, name, len(context[name]["panel"].links), task) if training else count
        accumulated = training and getattr(args, "update_windows", None) is not None
        conditions = {kind: sum(sample["condition"] == kind for sample in samples)
                      for kind in {sample["condition"] for sample in samples}} if accumulated and task == "queue" else None
        if training:
            optimizer.zero_grad(set_to_none=True)
        total_loss = torch.zeros((), device=device)
        for start in range(0, count, size):
            part = samples[start:start + size]
            total_loss += process_batch(name, part, conditions)
        if training:
            torch.nn.utils.clip_grad_norm_(model.parameters(), args.grad_clip)
            optimizer.step()
            global_step += 1
            updates_by_panel[name] += 1
        totals[0] += total_loss * count
        totals[1] += count
        if log and training and global_step % args.log_every == 0:
            values = totals.cpu().tolist()
            event(log, event="progress", epoch=epoch + 1, period=period, panel=name, step=global_step,
                  windows=windows, mean_loss=values[0] / max(values[1], 1), elapsed_seconds=time.perf_counter() - started)

    items = (balanced_queue_items(context, args, epoch)
             if training and task == 'queue' and getattr(args, 'queue_balanced', False)
             else schedule(context, period, args, epoch))
    if not items:
        raise ValueError(f"No days in {period}")
    for number, (name, path, samples, prep) in enumerate(prepared_days(context, items, args, task, period, epoch), 1):
        preparation_seconds += prep
        if not training and task == "queue":
            selected = []
            daily_seen = set()
            for sample in samples:
                kind = sample["condition"]
                if (seen_conditions[name][kind] < args.validation_windows
                        and (not getattr(args, "queue_validation_spread", False) or kind not in daily_seen)):
                    if getattr(args, 'queue_public_contract', False):
                        origin_time = pd.Timestamp(sample['forecast_origin'])
                        previous = accepted_origins.setdefault((name, kind), [])
                        if any(abs((origin_time - old).total_seconds()) < 60 * QUEUE_PUBLIC_CONTRACT['minimum_spacing_minutes'] for old in previous):
                            continue
                        previous.append(origin_time)
                    selected.append(sample)
                    daily_seen.add(kind)
                    seen_conditions[name][kind] += 1
            samples = selected
        pending[name].extend(samples)
        size = (update_size if training else batch_size)(args, name, len(context[name]["panel"].links), task)
        while len(pending[name]) >= size:
            process(name, pending[name][:size])
            del pending[name][:size]
        if log and (number == 1 or number % 50 == 0 or number == len(items)):
            event(log, event="data_progress", epoch=epoch + 1, period=period, days=number, total_days=len(items), panel=name, windows=windows)
    for name, samples in pending.items():
        if samples:
            process(name, samples)
    values = totals.cpu().tolist()
    if not values[1] or not np.isfinite(values).all():
        raise RuntimeError(f"No usable windows or non-finite loss in {period}")
    elapsed = time.perf_counter() - started
    report = {"task": task, "period": period, "loss": values[0] / values[1], "windows": windows,
              "elapsed_seconds": elapsed, "preparation_seconds_sum": preparation_seconds,
              "windows_per_second": windows / elapsed, "daily_records": records,
              "optimizer_updates": sum(updates_by_panel.values()), "updates_by_panel": updates_by_panel}
    if training and task == 'queue':
        report.update(training_group_counts=training_groups, sampled_label_changes=label_changes)
    if not training:
        chosen = max(thresholds, key=lambda t: queue_metrics(records, t)["proxy_IoU"])
        policy = fixed_policy
        if policy is None and period == 'select':
            positive = [row for row in policy_rows if row['condition'] != 'normal']
            candidates = queue_policy_candidates(positive, thresholds, chosen)
            policy = max(candidates, key=lambda row: row['metrics']['proxy_IoU'])['policy']
        if policy is None:
            policy = dict(name='saved', thresholds=dict(onset=chosen, ongoing=chosen), force_nonempty=False)
        for record, row in zip(records, policy_rows):
            threshold = policy['thresholds'].get(row['condition'], chosen)
            predicted = queue_decisions(row['probability'], threshold, row['target_mask'], policy['force_nonempty'])
            record['ious']['policy'] = iou(predicted, row['truth'], row['valid'])
            record['normal_false_positive']['policy'] = float((predicted & row['valid']).sum() / max(row['valid'].sum(), 1))
        report.update(queue_metrics(records, 'policy'))
        report['policy'] = policy
        report['selection_thresholds'] = list(thresholds)
        report["threshold"] = chosen
        report["baseline"] = queue_metrics(records, chosen, True)
        report["future_speed_rmse_kmh"] = float(np.sqrt(sum(row["speed_sse_kmh"] for row in records) / max(sum(row["count"] for row in records), 1)))
        keys = sorted((row["panel"], row["date"], row["condition"], row["origin"]) for row in records)
        report["window_signature"] = hashlib.sha256(json.dumps(keys).encode()).hexdigest()
        report["validation_spread"] = getattr(args, "queue_validation_spread", False)
    return report, global_step


def save_checkpoint(path, model, optimizer, scheduler, context, vocabulary, args, task, epoch, best, bad_epochs, step, report):
    payload = {"format_version": 2, "task": task, "weights": {k: v.detach().cpu() for k, v in model.state_dict().items()},
               "model_config": args.model_config,
               "feature_names": list(queue_feature_names(args.model_config.get("queue_changes", False))),
               "node_vocab": vocabulary,
               "trained_panels": list(context), "links": {name: item["panel"].links for name, item in context.items()},
               "profiles": {name: torch.from_numpy(item["profile"]) for name, item in context.items()},
               "fit_signature": {name: signature(item["fit_paths"]) for name, item in context.items()},
               "training_args": {key: str(value) if isinstance(value, Path) else value for key, value in vars(args).items()},
               "epoch": epoch + 1, "global_step": step, "best_metric": best, "bad_epochs": bad_epochs,
               "optimizer": optimizer.state_dict(), "scheduler": scheduler.state_dict(),
               "rng_cpu": torch.get_rng_state(), "rng_cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else [],
               "threshold": report.get("threshold", 0.5), "bidirectional": args.bidirectional,
               "metrics": {key: value for key, value in report.items() if key != "daily_records"},
               "environment": {"python": sys.version, "torch": str(torch.__version__), "cuda": torch.version.cuda}}
    if task == 'queue' and args.model_config.get('queue_future_prior', False):
        payload['queue_future_names'] = list(QUEUE_FUTURE_NAMES)
    if getattr(args, "queue_origin", False):
        payload["origin_fit_signature"] = {name: signature(item["origin_fit_paths"]) for name, item in context.items()}
    temporary = path.with_suffix(path.suffix + ".pending")
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(payload, temporary)
    temporary.replace(path)


def train(args, task):
    if task != 'queue':
        raise ValueError('Task 1은 MagiNet을 사용합니다. 이 경로는 Queue 전용입니다.')
    for key, default in (('queue_future_prior', False), ('queue_label_mode', 'hard'),
                         ('queue_label_margin', 5.), ('queue_balanced', False), ('init_queue', None)):
        if not hasattr(args, key):
            setattr(args, key, default)
    if args.init_queue and args.resume:
        raise ValueError('--init-queue는 Queue 새 실험에서 단독으로 사용해야 합니다.')
    if args.queue_label_mode != 'hard' and (args.queue_label_margin <= 0 or not np.isfinite(args.queue_label_margin)):
        raise ValueError('확률 라벨의 margin은 유한한 양수여야 합니다.')
    if not args.queue_thresholds or not all(np.isfinite(t) and 0 < t < 1 for t in args.queue_thresholds):
        raise ValueError('Queue 임계값 후보는 0과 1 사이의 유한한 값이어야 합니다.')
    if min(args.epochs, args.cpu_threads, args.log_every) < 1 or args.grad_clip <= 0:
        raise ValueError("Epochs, threads, log interval, and gradient clipping must be positive")
    if min(args.normal_weight, args.soft_iou_weight, args.speed_weight, args.patience) < 0:
        raise ValueError("Loss weights and early-stopping patience must be nonnegative")
    if not 0 <= args.history_coverage <= 1 or not 0 <= args.future_coverage <= 1:
        raise ValueError("Coverage thresholds must lie in [0, 1]")
    if args.update_windows is not None and args.update_windows < 0:
        raise ValueError("--update-windows must be nonnegative")
    if not 0 <= args.wavelet_levels <= 3:
        raise ValueError('--wavelet-levels는 0~3이어야 합니다.')
    if min(args.validation_windows, args.windows_per_day) < 1 or args.normal_windows < 0 or args.window_gap_minutes < 0:
        raise ValueError("Queue window limits must be positive, and normal windows/gap nonnegative")
    torch.set_num_threads(args.cpu_threads)
    torch.manual_seed(args.seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    args.use_bf16 = device == "cuda" and (args.bf16 or args.precision == "bf16" or args.precision == "auto" and torch.cuda.is_bf16_supported())
    if (args.bf16 or args.precision == "bf16") and (device != "cuda" or not torch.cuda.is_bf16_supported()):
        raise ValueError("BF16 training requires a supported CUDA GPU")
    if args.backend != "scan" and device != "cuda":
        raise ValueError("Official Mamba training requires CUDA; use --backend scan only for CPU contract tests")
    resumed = torch.load(args.resume, map_location="cpu", weights_only=True) if args.resume else None
    if resumed and any(resumed['training_args'].get(key, 0) != 0 for key in ('label_temperature', 'uncertainty_margin')):
        raise ValueError('제거된 라벨 보정을 사용한 모델은 재개할 수 없습니다. 새 실험의 --init-queue로 사용해 주세요.')
    initial = torch.load(args.init_queue, map_location='cpu', weights_only=True) if args.init_queue else None
    if initial:
        outputs = (args.checkpoint, args.checkpoint.with_name(args.checkpoint.stem + '.last.pt'))
        if any(args.init_queue.resolve() == path.resolve() for path in outputs):
            raise ValueError('초기 Queue와 best·last 출력 경로를 분리해 주세요.')
        if initial.get('format_version') != 2 or initial.get('task') != 'queue':
            raise ValueError('--init-queue에는 v2 Queue 체크포인트가 필요합니다.')
        if tuple(initial['feature_names']) != queue_feature_names(args.queue_changes):
            raise ValueError('초기 Queue와 과거 입력 특징이 다릅니다.')
    if resumed and resumed["task"] != task:
        raise ValueError("Resume checkpoint task mismatch")
    args.queue_sampling_version = (resumed['training_args'].get('queue_sampling_version', 2) if resumed else
        4 if args.queue_head == 'conditional' or args.queue_future_prior or args.queue_balanced or args.queue_label_mode != 'hard' else 3)
    if resumed:
        check_model_requests(args, resumed["model_config"])
        for key in ("queue_origin", "queue_resample", "queue_condition_gap", "queue_validation_spread", "queue_public_contract"):
            saved = resumed["training_args"].get(key, False)
            if getattr(args, key, False) and not saved:
                raise ValueError(f"Resume must retain {key}; use a new checkpoint to change Queue inputs/sampling")
            setattr(args, key, saved)
        args.queue_head = resumed["model_config"].get("queue_head", "linear")
        args.queue_changes = resumed["model_config"].get("queue_changes", False)
        args.queue_future_prior = resumed['model_config'].get('queue_future_prior', False)
        for key, default in (('queue_label_mode', 'hard'), ('queue_label_margin', 5.), ('queue_balanced', False)):
            value = resumed['training_args'].get(key, default)
            if getattr(args, key) != default and getattr(args, key) != value:
                raise ValueError(f'Resume must retain {key}; use --init-queue for a new experiment')
            setattr(args, key, value)
        # Restore settings that determine the data/loss; only the requested epoch budget and paths may change.
        for key in ("seed", "bidirectional", "windows_per_day", "normal_windows", "window_gap_minutes", "validation_windows",
                    "history_coverage", "future_coverage", "normal_weight",
                    "soft_iou_weight", "speed_weight", "max_days", "precision", "bf16",
                    "batch_size", "panel_batch", "grad_clip"):
            setattr(args, key, resumed["training_args"][key])
        args.use_bf16 = checkpoint_bf16(resumed, device, strict=True)
        # Older v2 checkpoints keep their original per-microbatch updates.
        args.update_windows = resumed["training_args"].get("update_windows")
    if args.queue_balanced and args.update_windows is None:
        args.update_windows = 48
    context = contexts(args, task, resumed or initial)
    if resumed:
        if set(context) != set(resumed["trained_panels"]):
            raise ValueError("Resume must retain the trained panel set")
        for name, item in context.items():
            if signature(item["fit_paths"]) != resumed["fit_signature"][name]:
                raise ValueError(f"Fit dataset changed since the checkpoint: {name}")
            if args.queue_origin and signature(item["origin_fit_paths"]) != resumed["origin_fit_signature"][name]:
                raise ValueError(f"Released Queue origin dataset changed since the checkpoint: {name}")
        vocabulary = resumed["node_vocab"]
        args.model_config = resumed["model_config"]
        model = model_from_checkpoint(resumed, device)
    else:
        vocabulary = {f"{name}|{link}": i + 1 for i, (name, link) in enumerate((name, link) for name in context for link in context[name]["panel"].links)}
        args.model_config = {"node_count": len(vocabulary), "width": args.width,
                             "blocks": args.blocks, "heads": args.heads, "dropout": args.dropout,
                             "backend": args.backend, "graph": args.graph,
                             "wavelet_levels": args.wavelet_levels, "wavelet_input": args.wavelet_input or "signals"}
        queue_config = {"queue_head": args.queue_head, "queue_changes": args.queue_changes,
                        'queue_future_prior': args.queue_future_prior,
                        "features": len(queue_feature_names(args.queue_changes))}
        args.model_config.update(queue_config)
        if initial:
            check_model_requests(args, initial['model_config'], queue=False)
            if set(context) != set(initial['trained_panels']):
                raise ValueError('초기 Queue의 전체 패널을 사용해 주세요.')
            for name, item in context.items():
                if signature(item['fit_paths']) != initial['fit_signature'][name]:
                    raise ValueError(f'초기 Queue의 fit 데이터가 다릅니다: {name}')
            vocabulary = initial['node_vocab']
            args.bidirectional = initial['bidirectional']
            args.model_config = {**initial['model_config'], **queue_config}
        model = (TrafficQueueModel)(**args.model_config).to(device)
        if initial:
            initialize_queue(model, initial)
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, fused=device == "cuda")
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", patience=args.scheduler_patience, factor=0.5)
    start_epoch, best, bad_epochs, step = 0, -float("inf"), 0, 0
    if resumed:
        optimizer.load_state_dict(resumed["optimizer"])
        scheduler.load_state_dict(resumed["scheduler"])
        start_epoch, best, bad_epochs, step = resumed["epoch"], resumed["best_metric"], resumed["bad_epochs"], resumed["global_step"]
        torch.set_rng_state(resumed["rng_cpu"])
        if device == "cuda" and resumed["rng_cuda"]:
            torch.cuda.set_rng_state_all(resumed["rng_cuda"])
    if args.epochs <= start_epoch:
        raise ValueError(f"--epochs is the total budget and must exceed completed epoch {start_epoch}")
    log = args.checkpoint.with_suffix(".jsonl")
    event(log, event="start", task=task, device=device, model=args.model_config, bf16=args.use_bf16,
          panels=list(context), epochs=args.epochs, start_epoch=start_epoch, parameters=sum(p.numel() for p in model.parameters()),
          update_windows=args.update_windows,
          effective_update_windows={name: update_size(args, name, len(item["panel"].links), task) for name, item in context.items()})
    if resumed:
        previous = torch.load(args.checkpoint, map_location='cpu', weights_only=True) if args.checkpoint.is_file() else resumed
        if ('policy' not in previous.get('metrics', {})
                or previous['metrics'].get('selection_thresholds') != list(args.queue_thresholds)):
            # 구형 선택 점수와 새 정책 점수를 직접 비교하지 않습니다.
            with torch.random.fork_rng(devices=[torch.cuda.current_device()] if device == 'cuda' else []):
                old_model = model_from_checkpoint(previous, device)
                report, _ = run_period(old_model, context, vocabulary, args, task, 'select', 0, device)
                del old_model
            best, bad_epochs = report['proxy_IoU'], 0
            scheduler.best, scheduler.num_bad_epochs, scheduler.cooldown_counter = best, 0, 0
            previous.update(best_metric=best, threshold=report['threshold'],
                            metrics={key: value for key, value in report.items() if key != 'daily_records'})
            pending = args.checkpoint.with_suffix('.pending')
            torch.save(previous, pending)
            pending.replace(args.checkpoint)
    for epoch in range(start_epoch, args.epochs):
        fit, step = run_period(model, context, vocabulary, args, task, "fit", epoch, device, optimizer, log, step)
        select, _ = run_period(model, context, vocabulary, args, task, "select", epoch, device, log=log, global_step=step)
        metric = select["proxy_IoU"]
        improved = metric > best
        best = max(best, metric)
        bad_epochs = 0 if improved else bad_epochs + 1
        scheduler.step(metric)
        event(log, event="epoch", epoch=epoch + 1, fit_loss=fit["loss"], select_metric=metric, best_metric=best,
              learning_rate=optimizer.param_groups[0]["lr"], fit_seconds=fit["elapsed_seconds"],
              fit_windows_per_second=fit["windows_per_second"], improved=improved,
              fit_windows=fit["windows"], optimizer_updates=fit["optimizer_updates"], updates_by_panel=fit["updates_by_panel"])
        select['training_group_counts'] = fit.get('training_group_counts', {})
        select['sampled_label_changes'] = fit.get('sampled_label_changes', 0)
        write_json(args.checkpoint.with_suffix(f".epoch{epoch + 1}.json"), select)
        save_checkpoint(args.checkpoint.with_name(args.checkpoint.stem + ".last.pt"), model, optimizer, scheduler, context, vocabulary, args, task, epoch, best, bad_epochs, step, select)
        if improved:
            save_checkpoint(args.checkpoint, model, optimizer, scheduler, context, vocabulary, args, task, epoch, best, bad_epochs, step, select)
        if args.patience and bad_epochs >= args.patience:
            event(log, event="early_stop", epoch=epoch + 1, patience=args.patience)
            break
    print(f"Best checkpoint: {args.checkpoint}; latest/resume: {args.checkpoint.stem}.last.pt", flush=True)


def evaluate(args):
    checkpoint = torch.load(args.checkpoint, map_location="cpu", weights_only=True)
    if checkpoint.get("format_version") != 2 or checkpoint.get("task") != "queue":
        raise ValueError("evaluate는 v2 Queue 체크포인트만 지원합니다.")
    for key, value in checkpoint["training_args"].items():
        if not hasattr(args, key) or getattr(args, key) is None:
            setattr(args, key, value)
    if args.validation_windows is None:
        args.validation_windows = checkpoint["training_args"]["validation_windows"]
    if args.validation_windows < 1:
        raise ValueError("--validation-windows must be positive")
    args.queue_validation_spread = bool(getattr(args, "queue_validation_spread", False))
    args.seed = checkpoint["training_args"]["seed"]
    args.fixed_threshold = checkpoint["threshold"]
    args.fixed_policy = checkpoint.get('metrics', {}).get('policy') or dict(
        name='saved', thresholds=dict(onset=args.fixed_threshold, ongoing=args.fixed_threshold), force_nonempty=False)
    args.bidirectional = checkpoint["bidirectional"]
    device = "cuda" if torch.cuda.is_available() else "cpu"
    torch.set_num_threads(args.cpu_threads)
    args.use_bf16 = checkpoint_bf16(checkpoint, device)
    context = contexts(args, checkpoint["task"], checkpoint)
    model = model_from_checkpoint(checkpoint, device).eval()
    report, _ = run_period(model, context, checkpoint["node_vocab"], args, checkpoint["task"], args.period, 0, device)
    if args.bootstrap:
        rows = report["daily_records"]
        dates = sorted({row["date"] for row in rows})
        by_date = {date: [row for row in rows if row["date"] == date] for date in dates}
        rng = np.random.default_rng(args.seed)
        differences = []
        for _ in range(args.bootstrap):
            sample = [row for date in rng.choice(dates, len(dates), replace=True) for row in by_date[date]]
            differences.append(queue_metrics(sample, 'policy')["proxy_IoU"] - queue_metrics(sample, 'policy', True)["proxy_IoU"])
        report["paired_day_bootstrap_delta_95pct"] = np.percentile(differences, [2.5, 97.5]).tolist()
    report["checkpoint_epoch"] = checkpoint["epoch"]
    write_json(args.output, report)
    print(json.dumps({key: value for key, value in report.items() if key != "daily_records"}, ensure_ascii=False, indent=2))


def inference_context(checkpoint, release, name, device):
    panel = PanelInfo.load(release, name)
    if name not in checkpoint["profiles"] or checkpoint["links"][name] != panel.links:
        raise ValueError(f"Checkpoint does not cover this panel/layout: {name}")
    return panel, checkpoint["profiles"][name].numpy(), torch.tensor(
        [checkpoint["node_vocab"].get(f"{name}|{link}", 0) for link in panel.links], device=device), torch.as_tensor(panel.edges, device=device)




def history_day(frame, panel):
    times = sorted(frame.timestamp.unique().tolist())
    if len(times) != 12 or frame.duplicated(["timestamp", "link_id"]).any():
        raise ValueError("Task 2 history must have twelve unique time/link cells")
    if not set(frame.link_id).issubset(panel.link_index):
        raise ValueError("Task 2 history has unknown links")
    index = pd.MultiIndex.from_product([times, panel.links], names=["timestamp", "link_id"])
    aligned = frame.set_index(["timestamp", "link_id"]).reindex(index)
    shape = len(times), len(panel.links)
    speed = pd.to_numeric(aligned.speed_kmh, errors="coerce").to_numpy(np.float32).reshape(shape) / 25
    flow = pd.to_numeric(aligned.flow_vph, errors="coerce").to_numpy(np.float32).reshape(shape) / panel.lanes / 600
    occupancy = pd.to_numeric(aligned.occupancy, errors="coerce").to_numpy(np.float32).reshape(shape)
    quality = pd.to_numeric(aligned.pct_observed, errors="coerce").fillna(0).to_numpy(np.float32).reshape(shape) / 100
    eligible = aligned.is_score_eligible.fillna(False).astype(bool).to_numpy().reshape(shape)
    return StationDay(times, panel.links, panel.links, np.arange(len(panel.links)), panel.lanes,
                      speed, flow, eligible & np.isfinite(speed) & np.isfinite(flow), occupancy, quality)


def predict_queue_v2(model, checkpoint, release, split, output, selected=None, probabilities=False, policy=None, probability_output=None, baseline_output=None):
    policy = policy if policy is not None else checkpoint.get('metrics', {}).get('policy')
    device = next(model.parameters()).device.type
    parts, probability_parts, baseline_parts = [], [], []
    for name in selected or checkpoint["trained_panels"]:
        root = release / "task2" / name / split
        if not (root / "sample_submission_queue.csv").exists():
            continue
        panel, profile, node_ids, edges = inference_context(checkpoint, release, name, device)
        include_origin = checkpoint["training_args"].get("queue_origin", False)
        changes = checkpoint.get("model_config", {}).get("queue_changes", False)
        origin_context = {"panel": panel, "profile": profile}
        if include_origin:
            origin_context["origin_paths"] = masked_file_index(release, name, split)
        origin_cache = {}
        template = pd.read_csv(root / "sample_submission_queue.csv", dtype=str)
        index = pd.read_csv(root / "window_index.csv", dtype=str).set_index("window_id")
        history = pd.read_parquet(root / "window_history.parquet")
        for col in ("window_id", "timestamp", "link_id"):
            history[col] = history[col].astype(str)
        histories = {key: frame for key, frame in history.groupby("window_id")}
        inputs, targets = [], []
        for window_id, target in template.groupby("window_id", sort=False):
            if window_id not in histories or window_id not in index.index:
                raise ValueError(f"No history/index for {window_id}")
            day = history_day(histories[window_id], panel)
            origin = pd.Timestamp(index.loc[window_id, "forecast_origin"])
            expected = pd.date_range(origin - pd.Timedelta(minutes=60), periods=12, freq="5min").strftime("%Y-%m-%dT%H:%M:%SZ").tolist()
            future = pd.date_range(origin + pd.Timedelta(minutes=5), periods=6, freq="5min").strftime("%Y-%m-%dT%H:%M:%SZ").tolist()
            if day.timestamps != expected or sorted(target.timestamp.unique()) != future:
                raise ValueError(f"Unexpected Task 2 time boundary: {window_id}")
            if not set(target.link_id).issubset(panel.link_index):
                raise ValueError("Unknown Queue target link")
            timestamp = origin.strftime("%Y-%m-%dT%H:%M:%SZ")
            ramp_times = day.timestamps + ([timestamp] if include_origin else [])
            ramp = ramp_features(release, panel, split, ramp_times)
            released = released_origin_features(origin_context, timestamp, ramp[-1:], origin_cache) if include_origin else None
            inputs.append({"x": queue_features(day, panel, profile, ramp[:12], released, changes)})
            public_kind = queue_condition(day, panel, np.array([panel.link_index[link] for link in target.link_id.unique()]))
            if (policy is not None or checkpoint['model_config'].get('queue_head') == 'conditional') and 'condition' in index.columns:
                published = str(index.loc[window_id, 'condition']).removeprefix('queue_')
                if published not in ('onset', 'ongoing'):
                    raise ValueError(f'알 수 없는 공개 Queue 조건: {published}')
                public_kind = published
            if checkpoint['model_config'].get('queue_head') == 'conditional':
                inputs[-1]['branch'] = np.array(int(public_kind == 'ongoing'), np.int64)
            if checkpoint['model_config'].get('queue_future_prior', False):
                inputs[-1]['future'] = queue_future_features(timestamp, panel, profile)
            targets.append((target, future, public_kind))
        size = 48 if len(panel.links) <= 128 else 12
        for start in range(0, len(inputs), size):
            keys = ['x']
            if checkpoint['model_config'].get('queue_head') == 'conditional':
                keys.append('branch')
            if checkpoint['model_config'].get('queue_future_prior', False):
                keys.append('future')
            batch = tensor_batch(inputs[start:start + size], keys, device)
            with torch.inference_mode(), torch.autocast(device, dtype=torch.bfloat16,
                enabled=checkpoint_bf16(checkpoint, device)):
                logits, _ = model(batch['x'], node_ids, panel.segments, edges, checkpoint['bidirectional'],
                                  condition=batch.get('branch'), future=batch.get('future'))
                values = logits.float().sigmoid().cpu().numpy()
            if not np.isfinite(values).all():
                raise RuntimeError("Non-finite Queue prediction")
            for probability, (target, future, public_kind) in zip(values, targets[start:start + size]):
                result = target.copy()
                ti = result.timestamp.map({time: i for i, time in enumerate(future)}).to_numpy(int)
                li = result.link_id.map(panel.link_index).to_numpy(int)
                target_probability = probability[ti, li]
                valid = np.ones_like(target_probability, dtype=bool)
                threshold = policy['thresholds'][public_kind] if policy else checkpoint['threshold']
                predicted = queue_decisions(target_probability, threshold, valid, policy['force_nonempty'] if policy else False)
                result['queue_pred'] = target_probability if probabilities else predicted.astype(int)
                parts.append(result[['window_id', 'timestamp', 'link_id', 'queue_pred']])
                if probability_output is not None:
                    probability_parts.append(result[['window_id', 'timestamp', 'link_id']].assign(queue_probability=target_probability))
                if baseline_output is not None:
                    baseline_parts.append(result[['window_id', 'timestamp', 'link_id']].assign(
                        queue_pred=(target_probability >= checkpoint['threshold']).astype(int)))
    if not parts:
        raise ValueError("No Queue predictions")
    result = pd.concat(parts, ignore_index=True)
    if result[['window_id', 'timestamp', 'link_id']].duplicated().any():
        raise ValueError('Duplicate Queue keys')
    if probability_output is not None:
        atomic_queue_csv(pd.concat(probability_parts, ignore_index=True), probability_output)
    if baseline_output is not None:
        atomic_queue_csv(pd.concat(baseline_parts, ignore_index=True), baseline_output)
    atomic_queue_csv(result, output)
def queue_file_digest(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def atomic_queue_csv(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    pending = path.with_suffix(path.suffix + '.pending')
    try:
        frame.to_csv(pending, index=False)
        pending.replace(path)
    finally:
        pending.unlink(missing_ok=True)


def check_queue_release_contract(release):
    release = Path(release)
    candidates = [release / 'config/synthetic_release_v1.json', release / 'synthetic_release_v1.json',
                  release.parent / 'config/synthetic_release_v1.json']
    for path in candidates:
        if path.is_file():
            config = json.loads(path.read_text())['task2']
            if any(config.get(key) != value for key, value in QUEUE_PUBLIC_CONTRACT.items()):
                raise ValueError(f'연결된 배포본의 Task 2 설정이 확인한 공개 설정과 다릅니다: {path}')
            return str(path)
    print('배포본 설정 파일이 없어 2026-10-28 확인한 공식 공개 조건을 사용합니다. 관측 라벨 기반 대리 검증입니다.')
    return 'official-public-config-2026-10-28'


def collect_queue_probabilities(checkpoint_path, release, period, folder, windows=5, seed=42, cpu_threads=2):
    if period not in ('select', 'holdout') or windows < 1:
        raise ValueError('Queue 검증은 select/holdout과 양수 window 수가 필요합니다.')
    checkpoint_path, release, folder = Path(checkpoint_path), Path(release), Path(folder)
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
    if checkpoint.get('format_version') != 2 or checkpoint['task'] != 'queue':
        raise ValueError('v2 Queue 체크포인트가 필요합니다.')
    contract_source = check_queue_release_contract(release)
    args = SimpleNamespace(**checkpoint['training_args'])
    args.release_root, args.period, args.panel = release, period, None
    args.seed, args.max_days, args.workers, args.cpu_threads = seed, None, 0, cpu_threads
    args.queue_public_contract, args.queue_validation_spread, args.validation_windows = True, True, windows
    args.fixed_threshold, args.model_config = checkpoint['threshold'], checkpoint['model_config']
    file_index.cache_clear()
    masked_file_index.cache_clear()
    files = []
    for panel in checkpoint['trained_panels']:
        files.extend(period_files(release, panel, period))
        template = release / 'task2' / panel / 'validation/sample_submission_queue.csv'
        if template.is_file():
            files.append(template)
        files.extend((release / 'corridors' / panel / 'network').glob('*.csv'))
        files.extend((release / 'corridors' / panel / 'train/mainline_states_masked').rglob('*.parquet'))
        files.extend((release / 'corridors' / panel / 'train/ramp_states').rglob('*.parquet'))
        originals = sorted((release / 'corridors' / panel / 'train/mainline_states').rglob('*.parquet'))
        if originals:
            files.append(originals[0])
    inputs = [(str(p.relative_to(release)), p.stat().st_size, p.stat().st_mtime_ns) for p in sorted(set(files))]
    config = dict(selector_version='queue-public-observed-v2', checkpoint_sha256=queue_file_digest(checkpoint_path), period=period, seed=seed, windows=windows,
                  contract=QUEUE_PUBLIC_CONTRACT, contract_source=contract_source,
                  input_signature=hashlib.sha256(json.dumps(inputs).encode()).hexdigest())
    cache_key = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:20]
    destination = folder / 'probabilities' / period / cache_key
    marker = destination / 'manifest.json'
    rows = []
    if marker.is_file():
        manifest = json.loads(marker.read_text())
        if manifest['config'] != config:
            raise ValueError('Queue 확률 캐시 설정이 다릅니다.')
        for panel, digest in manifest['files'].items():
            path = destination / f'{panel}.npz'
            if queue_file_digest(path) != digest:
                raise ValueError(f'Queue 확률 캐시가 손상됐습니다. 해당 캐시 폴더를 지운 뒤 다시 생성해 주세요: {path}')
            with np.load(path, allow_pickle=False) as data:
                for i in range(len(data['origin'])):
                    rows.append(dict(panel=panel, condition=str(data['condition'][i]), date=str(data['date'][i]),
                                     origin=str(data['origin'][i]), **{key: data[key][i].copy() for key in
                                     ('probability', 'truth', 'valid', 'baseline', 'target_mask')}))
        return rows, manifest
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    args.use_bf16 = checkpoint_bf16(checkpoint, device)
    torch.set_num_threads(cpu_threads)
    context = contexts(args, 'queue', checkpoint)
    model = model_from_checkpoint(checkpoint, device).eval()
    try:
        run_period(model, context, checkpoint['node_vocab'], args, 'queue', period, 0, device, queue_store=rows)
    finally:
        del model
        if device == 'cuda':
            torch.cuda.empty_cache()
    rows.sort(key=lambda row: (row['panel'], row['origin'], row['condition']))
    missing = [(panel, kind) for panel in checkpoint['trained_panels'] for kind in ('onset', 'ongoing')
               if not any(row['panel'] == panel and row['condition'] == kind for row in rows)]
    if missing:
        raise ValueError(f'공개 조건의 Queue 대리 검증 표본이 없습니다: {missing}')
    signature_rows = [(row['panel'], row['origin'], row['condition'],
                       hashlib.sha256(row['target_mask'].tobytes() + row['valid'].tobytes() + row['truth'].tobytes()).hexdigest()) for row in rows]
    manifest = dict(config=config, proxy_only=True,
                    window_signature=hashlib.sha256(json.dumps(signature_rows).encode()).hexdigest(),
                    window_counts={f'{panel}:{kind}': sum(row['panel'] == panel and row['condition'] == kind for row in rows)
                                   for panel in checkpoint['trained_panels'] for kind in ('onset', 'ongoing')}, files={})
    insufficient = {key: count for key, count in manifest['window_counts'].items() if count < windows}
    if insufficient:
        print('공개 관측으로 만든 대리 검증의 표본 수가 요청보다 적습니다:', insufficient)
    destination.mkdir(parents=True, exist_ok=True)
    for panel in sorted({row['panel'] for row in rows}):
        part = [row for row in rows if row['panel'] == panel]
        path = destination / f'{panel}.npz'
        pending = path.with_suffix('.pending.npz')
        np.savez_compressed(pending, **{key: np.stack([row[key] for row in part]) for key in
                            ('probability', 'truth', 'valid', 'baseline', 'target_mask')},
                            **{key: np.array([row[key] for row in part]) for key in ('condition', 'date', 'origin')})
        pending.replace(path)
        manifest['files'][panel] = queue_file_digest(path)
    write_json(marker, manifest)
    return rows, manifest


def queue_policy_metrics(rows, policy):
    records, details, leads = [], [], []
    for row in rows:
        threshold = policy['thresholds'][row['condition']]
        predicted = queue_decisions(row['probability'], threshold, row['target_mask'], policy['force_nonempty'])
        truth, valid = row['truth'], row['valid']
        score = iou(predicted, truth, valid)
        records.append(dict(panel=row['panel'], condition=row['condition'], ious={'0.5': score}))
        base = dict(panel=row['panel'], condition=row['condition'], date=row['date'], origin=row['origin'])
        counts = lambda p, t, m: dict(tp=int((p & t & m).sum()), fp=int((p & ~t & m).sum()),
                                      fn=int((~p & t & m).sum()), predicted=int((p & m).sum()),
                                      truth=int((t & m).sum()), eligible=int(m.sum()))
        details.append(dict(**base, IoU=score, threshold=threshold, empty_prediction=not predicted.any(),
                            empty_before=not ((row['probability'] >= threshold) & row['target_mask']).any(),
                            persistence_IoU=iou(row['baseline'], truth, valid), **counts(predicted, truth, valid)))
        for lead in range(6):
            leads.append(dict(**base, minutes=5 * (lead + 1), **counts(predicted[lead], truth[lead], valid[lead])))
    return queue_metrics(records, 0.5), pd.DataFrame(details), pd.DataFrame(leads)


def queue_policy_candidates(rows, thresholds, saved):
    thresholds = sorted(set(float(value) for value in thresholds))
    if not thresholds or not all(np.isfinite(value) and 0 < value < 1 for value in thresholds):
        raise ValueError('Queue 임계값 후보는 0과 1 사이의 유한한 값이어야 합니다.')
    candidates = []
    def add(name, values, force):
        policy = dict(name=name, thresholds=values, force_nonempty=force)
        metrics, _, _ = queue_policy_metrics(rows, policy)
        candidates.append(dict(policy=policy, metrics=metrics))
    add('saved', dict(onset=saved, ongoing=saved), False)
    add('saved_nonempty', dict(onset=saved, ongoing=saved), True)
    for force in (False, True):
        for threshold in sorted(set(thresholds + [saved])):
            add(f'global_{threshold:g}_nonempty_{force}', dict(onset=threshold, ongoing=threshold), force)
        selected = {}
        for kind in ('onset', 'ongoing'):
            part = [row for row in rows if row['condition'] == kind]
            selected[kind] = max(sorted(set(thresholds + [saved])), key=lambda value:
                queue_policy_metrics(part, dict(thresholds={kind: value}, force_nonempty=force))[0]['proxy_IoU'])
        add(f'conditional_nonempty_{force}', selected, force)
    return candidates


def calibrate_queue(rows, manifest, checkpoint_path, folder, thresholds):
    if manifest['config']['period'] != 'select':
        raise ValueError('Queue 임계값은 1월 select 구간에서만 선택합니다.')
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
    if manifest['config']['checkpoint_sha256'] != queue_file_digest(checkpoint_path):
        raise ValueError('확률 캐시와 체크포인트가 다릅니다.')
    candidates = queue_policy_candidates(rows, thresholds, float(checkpoint['threshold']))
    best = max(candidates, key=lambda row: row['metrics']['proxy_IoU'])
    folder = Path(folder)
    comparison = [dict(candidate=row['policy']['name'], force_nonempty=row['policy']['force_nonempty'],
                       threshold_onset=row['policy']['thresholds']['onset'], threshold_ongoing=row['policy']['thresholds']['ongoing'],
                       **{key: row['metrics'][key] for key in ('proxy_IoU', 'onset', 'ongoing')}) for row in candidates]
    atomic_queue_csv(pd.DataFrame(comparison), folder / 'january_policy_comparison.csv')
    for label, candidate in (('saved', candidates[0]), ('selected', best)):
        _, windows, leads = queue_policy_metrics(rows, candidate['policy'])
        atomic_queue_csv(windows, folder / f'january_{label}_windows.csv')
        atomic_queue_csv(leads, folder / f'january_{label}_leads.csv')
    selection = dict(format='queue-calibration-v1', proxy_only=True, selected_on='2031-01',
                     checkpoint_sha256=manifest['config']['checkpoint_sha256'],
                     window_signature=manifest['window_signature'], contract=QUEUE_PUBLIC_CONTRACT,
                     validation_config=manifest['config'], policy=best['policy'], metrics=best['metrics'])
    write_json(folder / 'selection.json', selection)
    return selection


def load_queue_selection(path, checkpoint_path):
    selection = json.loads(Path(path).read_text())
    if (selection.get('format') != 'queue-calibration-v1' or selection.get('selected_on') != '2031-01'
            or selection.get('checkpoint_sha256') != queue_file_digest(checkpoint_path)
            or selection.get('contract') != QUEUE_PUBLIC_CONTRACT):
        raise ValueError('Queue 선택 파일의 체크포인트·선택 구간·공개 조건이 다릅니다.')
    policy = selection['policy']
    if (set(policy['thresholds']) != {'onset', 'ongoing'} or type(policy['force_nonempty']) is not bool
            or not all(np.isfinite(v) and 0 < v < 1 for v in policy['thresholds'].values())):
        raise ValueError('Queue 후처리 설정이 잘못되었습니다.')
    return selection


def predict_queue_calibrated(checkpoint_path, release, split, output, selection_path=None):
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=True)
    policy = load_queue_selection(selection_path, checkpoint_path)['policy'] if selection_path else None
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = model_from_checkpoint(checkpoint, device).eval()
    output = Path(output)
    try:
        predict_queue_v2(model, checkpoint, release, split, output, policy=policy,
                         probability_output=output.with_name(output.stem + '-probabilities.csv'),
                         baseline_output=output.with_name(output.stem + '-baseline.csv'))
    finally:
        del model
        if device == 'cuda':
            torch.cuda.empty_cache()
''')


In [10]:
# @title
# src/historical_retrieval.py — 원본 cdf888214fe8a54b73302f3988b764b74b3f30a4
load_source('historical_retrieval', r'''"""Fit-only historical patterns for optional frozen MagiNet inference."""
import hashlib
import json
from pathlib import Path

import numpy as np


CONFIG = dict(neighbors=4, window=36, stride=18, shifts=[-2, 0, 2], min_observations=8)
METHODS = ('global', 'local', 'local_gated')


class HistoricalRetrieval:
    def __init__(self, paths):
        paths = sorted(map(Path, paths))
        if not paths or len({p.stem for p in paths}) != len(paths):
            raise ValueError('중복 없는 fit 날짜별 NPZ가 필요합니다.')
        self.dates = [p.stem for p in paths]
        residuals, valid = [], []
        for path in paths:
            with np.load(path, allow_pickle=False) as day:
                active = day['active'].astype(bool)
                residual = np.where(active[..., None], day['truth'] - day['aux'][..., 8:10], 0.)
                if residual.shape != (*active.shape, 2) or not np.isfinite(residual).all():
                    raise ValueError(f'잘못된 fit 관측 배열입니다: {path}')
                residuals.append(residual.astype(np.float32))
                valid.append(active)
        # ponytail: keep one fit memory in RAM; persist only if rebuilding becomes a bottleneck.
        self.residuals, self.valid = np.stack(residuals), np.stack(valid)
        digest = hashlib.sha256(json.dumps(self.dates).encode())
        digest.update(self.residuals.tobytes())
        digest.update(self.valid.tobytes())
        self.sha256 = digest.hexdigest()

    def candidates(self, day, network, methods=METHODS):
        if not set(methods) <= set(METHODS):
            raise ValueError('지원하지 않는 과거 검색 방법입니다.')
        values, visible, prior = day['values'][..., :2], day['visible'][..., :2], day['aux'][..., 8:10]
        if (values.shape != prior.shape or visible.shape != prior.shape or network.shape != prior.shape
                or prior.shape != self.residuals.shape[1:]):
            raise ValueError('과거 검색의 시각·관측소·채널 구성이 다릅니다.')
        if self.dates[-1] >= str(day['timestamps'][0])[:10]:
            raise ValueError('과거 검색은 추론 날짜보다 이른 fit 자료만 사용해야 합니다.')
        visible = visible.astype(bool) & np.isfinite(values)
        residual = np.where(visible, values - prior, 0.)
        results = {}
        if 'global' in methods:
            estimate = np.zeros_like(prior)
            for channel in (0, 1):
                estimate[..., channel], _ = self._retrieve(residual, visible, 0, len(prior), channel, [0])
            results['global'] = np.maximum(prior + estimate, 0.)
        if set(methods) & {'local', 'local_gated'}:
            sums = np.zeros_like(prior, dtype=np.float64)
            gated = np.zeros_like(sums)
            weights = np.zeros((len(prior), 1, 1), dtype=np.float64)
            width = min(CONFIG['window'], len(prior))
            starts = sorted(set(range(0, len(prior) - width + 1, CONFIG['stride'])) | {len(prior) - width})
            taper = np.hanning(width + 2)[1:-1, None]
            for start in starts:
                stop = start + width
                for channel in (0, 1):
                    estimate, confidence = self._retrieve(residual, visible, start, stop, channel, CONFIG['shifts'])
                    candidate = np.maximum(prior[start:stop, :, channel] + estimate, 0.)
                    sums[start:stop, :, channel] += taper * candidate
                    gated[start:stop, :, channel] += taper * (
                        network[start:stop, :, channel] + confidence * (candidate - network[start:stop, :, channel]))
                weights[start:stop] += taper[..., None]
            if 'local' in methods:
                results['local'] = (sums / weights).astype(np.float32)
            if 'local_gated' in methods:
                results['local_gated'] = (gated / weights).astype(np.float32)
        return results

    def _retrieve(self, residual, visible, start, stop, channel, shifts):
        known = visible[start:stop, :, channel]
        query = residual[start:stop, :, channel]
        if known.sum() < CONFIG['min_observations']:
            return np.zeros_like(query), 0.
        candidates, masks, scores = [], [], []
        indices = np.arange(start, stop)
        for shift in shifts:
            shifted = indices + shift
            inside = (shifted >= 0) & (shifted < self.residuals.shape[1])
            indices_safe = shifted.clip(0, self.residuals.shape[1] - 1)
            patterns = self.residuals[..., channel][:, indices_safe]
            valid = self.valid[:, indices_safe] & inside[None, :, None]
            shared = valid & known[None]
            count = shared.sum(axis=(1, 2))
            error = np.where(shared, patterns - query, 0.)
            distance = np.square(error).sum(axis=(1, 2), dtype=np.float64) / np.maximum(count, 1)
            distance[count < max(CONFIG['min_observations'], known.sum() // 2)] = np.inf
            candidates.append(patterns)
            masks.append(valid)
            scores.append(distance)
        scores = np.stack(scores)
        # One shift per historical day prevents the same day occupying all nearest slots.
        best_shift = scores.argmin(axis=0)
        distance = scores[best_shift, np.arange(len(self.dates))]
        neighbors = np.argsort(distance, kind='stable')[:CONFIG['neighbors']]
        neighbors = neighbors[np.isfinite(distance[neighbors])]
        if not len(neighbors):
            return np.zeros_like(query), 0.
        patterns = np.stack([candidates[best_shift[n]][n] for n in neighbors])
        valid = np.stack([masks[best_shift[n]][n] for n in neighbors])
        estimate = np.where(valid, patterns, 0.).sum(axis=0) / np.maximum(valid.sum(axis=0), 1)
        # Compare only public observations; target truth/active are never read here.
        reference = np.square(query[known]).mean(dtype=np.float64)
        mismatch = np.square((estimate - query)[known]).mean(dtype=np.float64)
        uncertainty = np.where(valid, np.square(patterns - estimate), 0.).sum(dtype=np.float64) / max(valid.sum(), 1)
        confidence = np.clip(1 - mismatch / (reference + 1e-8), 0, 1) * reference / (reference + uncertainty + 1e-8)
        return estimate, float(confidence)

    def apply(self, day, network, method, weights):
        candidate = self.candidates(day, network, (method,))[method]
        return (network + np.asarray(weights, np.float32) * (candidate - network)).astype(np.float32)
''')


내장 코드 로드: historical_retrieval (115줄)


In [ ]:
# @title
# 내장 maginet.py — 최종 보간 이후 보정 경로 포함
load_source('maginet', r'''"""Local MagiNet Task 1/3 runner, adapted from task13_maginet_colab.ipynb.

The channel graph architecture and current003 loss are fixed.
Saved settings retain checkpoint identity and data contracts.
"""
from __future__ import annotations

import argparse
from types import SimpleNamespace

from dataclasses import dataclass, field
from pathlib import Path, PurePosixPath
from functools import lru_cache, partial
import hashlib, heapq, json, random, re, shutil, tempfile, time, zipfile

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F

PROJECT_ROOT = Path(__file__).resolve().parents[1]
RELEASE_ROOT = PROJECT_ROOT / 'data' / 'kaggle_public'
INPUT_NORMALIZATION = 'standard'  # 'fixed': 기존 /25·/600, 'standard': fit 관측으로 입력 표준화
EXPERIMENT = 'temporal_direct_dn003_channel_graph'
TEMPORAL_DIRECT, MASK_RESAMPLE, LEARNED_GRAPH = True, False, True
SPATIAL_MODE = 'channel'
REPORT_BASE = PROJECT_ROOT / 'reports' / 'maginet'
REPORT_DIR = REPORT_BASE / EXPERIMENT
CACHE_STORE_DIR = PROJECT_ROOT / 'data' / 'maginet_cache'
CACHE_METADATA_DIR = CACHE_STORE_DIR / 'metadata'
SAVED_CACHE_ZIP = None
CACHE_DIR = CACHE_STORE_DIR  # ZIP을 한 번 풀 로컬 폴더
CACHE_MODE = 'build'             # 'reuse': 누락 시 중단, 'build': 없는 날짜만 원본에서 생성
CACHE_PROFILE_DIR = CACHE_STORE_DIR / 'legacy_profiles'  # 이전 config.json·fit_profile-*.npy
PANELS = None                    # None: 전체 패널 공동 학습
MAX_TRAIN_DAYS = None            # 실행 점검 시 12 이상으로 줄일 수 있습니다.
DAYS_PER_REGIME = None          # 점검 시 각 기간·regime의 날짜 수를 제한합니다.
DATE_OVERRIDES = {}
SEED = 42
WINDOW, STRIDE = 72, 36
WIDTH, BLOCKS, HEADS, CHEB_K = 64, 3, 4, 3
GRAPH_DIM = 128
DROPOUT = 0.05
MGL_THRESHOLD = 0.5              # 공식 구현의 missing graph 판정 시작값
BATCH_SIZE = 2
BATCH_POLICY = 'fixed'           # adaptive: 관측소 128개 이하 8, 초과 4
PANEL_BATCH = {}                  # 명시한 패널의 배치가 policy보다 우선합니다.
UPDATES_PER_GROUP = 24           # epoch당 패널×regime 업데이트
EPOCHS, PATIENCE = 50, 8
LR, WEIGHT_DECAY = 1e-3, 1e-4
WARMUP_EPOCHS, PHYSICS_RAMP_EPOCHS = 5, 5
LOSS_WEIGHTS = dict(N=0.05, dN=0.03, FD=0.0)
# 공식 Task 1·3 비중을 참고한 대리 선택식입니다. 공식 Physics 점수가 아닙니다.
SELECT_WEIGHTS = dict(State=0.70, FD_proxy=0.10, LWR_proxy=0.20)
CHECKPOINT_METRIC = 'State'
ALPHAS = (0.0, 0.25, 0.5, 0.75, 1.0)
RESUME = False                   # last_maginet.pt에서 재개; EPOCHS를 마지막 epoch보다 크게 설정
LWR_MODES = {}                   # 공개 기본값 A; 패널별 B/C 실험 시 명시

REGIMES = ('R1', 'R2', 'R3')
KEYS = ['panel', 'timestamp', 'station_id', 'link_id', 'mask_regime']
SCALE = np.array([25., 600.], dtype=np.float32)
PERIODS = dict(fit=('2030-06-01', '2030-12-31'), early=('2031-01-01', '2031-01-15'),
               blend=('2031-01-16', '2031-01-31'), february=('2031-02-01', '2031-02-28'))
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
AMP_DTYPE = torch.bfloat16 if USE_AMP and torch.cuda.is_bf16_supported() else torch.float16
assert tuple(map(int, re.match(r'(\d+)\.(\d+)', torch.__version__).groups())) >= (2, 3)
assert 12 <= WINDOW <= 288 and 1 <= STRIDE <= WINDOW
assert WIDTH % HEADS == 0 and WIDTH % 2 == 0 and CHEB_K >= 1 and BLOCKS >= 1
assert BATCH_SIZE >= 1 and UPDATES_PER_GROUP >= 1 and EPOCHS >= 1
assert INPUT_NORMALIZATION in ('fixed', 'standard') and CACHE_MODE in ('reuse', 'build')
assert all(v >= 0 for v in LOSS_WEIGHTS.values()) and sum(SELECT_WEIGHTS.values()) > 0
assert all(v >= 0 for v in SELECT_WEIGHTS.values())
assert WARMUP_EPOCHS >= 0 and PHYSICS_RAMP_EPOCHS >= 0
assert len(set(ALPHAS)) == len(ALPHAS) and all(0 <= a <= 1 for a in ALPHAS) and 0.0 in ALPHAS and 1.0 in ALPHAS

def reset_seed(seed=None):
    seed = SEED if seed is None else seed
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def scan_layout(link_ids: list[str], topology: pd.DataFrame) -> list[str]:
    """Topological order within each weakly connected component; cycles are stable."""
    links = set(link_ids)
    outgoing = {link: set() for link in links}
    neighbors = {link: set() for link in links}
    for row in topology.itertuples(index=False):
        target = str(row.link_id)
        if target not in links:
            continue
        for source in str(getattr(row, "incoming_link_ids", "")).split(";"):
            source = source.strip()
            if source in links and source != target:
                outgoing[source].add(target)
                neighbors[source].add(target)
                neighbors[target].add(source)
        for dest in str(getattr(row, "outgoing_link_ids", "")).split(";"):
            dest = dest.strip()
            if dest in links and dest != target:
                outgoing[target].add(dest)
                neighbors[target].add(dest)
                neighbors[dest].add(target)
    seen: set[str] = set()
    ordered: list[str] = []
    for root in sorted(links):
        if root in seen:
            continue
        component = {root}
        pending = [root]
        seen.add(root)
        while pending:
            for neighbor in neighbors[pending.pop()]:
                if neighbor not in seen:
                    seen.add(neighbor)
                    component.add(neighbor)
                    pending.append(neighbor)
        indegree = {link: 0 for link in component}
        for source in component:
            for target in outgoing[source] & component:
                indegree[target] += 1
        ready = [link for link in component if indegree[link] == 0]
        heapq.heapify(ready)
        part = []
        while ready:
            source = heapq.heappop(ready)
            part.append(source)
            for target in outgoing[source] & component:
                indegree[target] -= 1
                if indegree[target] == 0:
                    heapq.heappush(ready, target)
        part.extend(sorted(component - set(part)))
        ordered.extend(part)
    return ordered

@dataclass
class PanelInfo:
    name: str
    links: list[str]
    link_index: dict[str, int]
    free_speed: np.ndarray
    lanes: np.ndarray
    lengths: np.ndarray
    station_lanes: dict[str, float]
    capacity: np.ndarray | None = None
    jam_density: np.ndarray | None = None
    edges: np.ndarray = field(default_factory=lambda: np.empty((2, 0), dtype=np.int64))
    ramp_map: dict[str, tuple[int, int]] = field(default_factory=dict)

    @classmethod
    def load(cls, release: Path, panel: str) -> "PanelInfo":
        panel_dir = release / "corridors" / panel
        candidates = sorted((panel_dir / "train" / "mainline_states").glob("**/*.parquet"))
        if not candidates:
            raise FileNotFoundError(f"No original train observations under {panel_dir}")
        observed = pd.read_parquet(candidates[0], columns=["link_id"]).link_id.astype(str).unique().tolist()
        network = panel_dir / "network"
        topology = pd.read_csv(network / "lwr_mainline_topology.csv", dtype=str)
        # Latent mainline connectors carry no detector value, but preserve the
        # road between observed links during the spatial scan.
        links = scan_layout(sorted(set(observed) | set(topology.link_id.dropna().astype(str))), topology)
        frame = pd.read_csv(network / "links.csv", dtype={"link_id": str}).drop_duplicates("link_id").set_index("link_id")
        fd_path = network / "fd_parameters.csv"
        fd = pd.read_csv(fd_path, dtype={"link_id": str, "station_id": str}) if fd_path.exists() else pd.DataFrame()
        grouped = fd.groupby("link_id").mean(numeric_only=True) if not fd.empty else pd.DataFrame()

        def parameter(column: str, default: float) -> np.ndarray:
            raw = pd.to_numeric(frame[column], errors="coerce").reindex(links) if column in frame else pd.Series(default, index=links)
            if column in grouped:
                raw = pd.to_numeric(grouped[column], errors="coerce").reindex(links).fillna(raw)
            return raw.fillna(default).to_numpy(dtype=np.float32)

        free_speed = parameter("free_speed_kmh", 105.0)
        lanes = np.clip(parameter("lanes", 1.0), 1, 16)
        lengths = parameter("length_km", 1.0)
        station_lanes: dict[str, float] = {}
        if "station_id" in fd and "lanes" in fd:
            for row in fd.dropna(subset=["station_id", "lanes"]).itertuples(index=False):
                station_lanes[str(row.station_id)] = float(row.lanes)
        lookup = {link: i for i, link in enumerate(links)}
        pairs = set()
        for row in topology.itertuples(index=False):
            target = str(row.link_id)
            for source in str(getattr(row, "incoming_link_ids", "")).split(";"):
                if source in lookup and target in lookup and source != target:
                    pairs.add((lookup[source], lookup[target]))
            for dest in str(getattr(row, "outgoing_link_ids", "")).split(";"):
                if target in lookup and dest in lookup and dest != target:
                    pairs.add((lookup[target], lookup[dest]))
        ramp_map = {}
        ramp_path = network / "ramp_attachment_map.csv"
        if ramp_path.exists():
            for row in pd.read_csv(ramp_path, dtype=str).itertuples(index=False):
                if row.nearest_mainline_link_id in lookup:
                    if row.ramp_type not in {"OR", "FR", "on", "off"}:
                        raise ValueError(f"Unknown ramp type: {row.ramp_type}")
                    ramp_map[row.ramp_link_id] = (lookup[row.nearest_mainline_link_id], int(row.ramp_type in {"FR", "off"}))
        return cls(panel, links, lookup, free_speed, lanes, lengths, station_lanes,
                   parameter("capacity_vph", 6000.0), np.maximum(parameter("k_jam", np.nan), 1.05 * parameter("capacity_vph", 6000.0) / np.maximum(free_speed, 1)),
                   np.array(sorted(pairs), dtype=np.int64).reshape(-1, 2).T, ramp_map)

@dataclass
class StationDay:
    timestamps: list[str]
    station_ids: list[str]
    station_links: list[str]
    station_to_link: np.ndarray
    lanes: np.ndarray
    speed: np.ndarray
    flow_lane: np.ndarray
    eligible: np.ndarray
    occupancy: np.ndarray | None = None
    quality: np.ndarray | None = None

    @classmethod
    def load(cls, path: Path, panel: PanelInfo) -> "StationDay":
        columns = ["timestamp", "station_id", "link_id", "speed_kmh", "flow_vph", "is_score_eligible"]
        # Optional published features also exist in Task 2 histories.
        import pyarrow.parquet as pq
        available = set(pq.read_schema(path).names) if path.stat().st_size else set(columns)
        columns += [c for c in ("occupancy", "pct_observed") if c in available]
        frame = pd.read_parquet(path, columns=columns)
        for column in ("timestamp", "station_id", "link_id"):
            frame[column] = frame[column].astype(str)
        if frame.duplicated(["timestamp", "station_id", "link_id"]).any():
            raise ValueError(f"Duplicate station/time rows in {path}")
        stations = frame[["station_id", "link_id"]].drop_duplicates()
        if stations.station_id.duplicated().any():
            raise ValueError(f"Station maps to multiple links in {path}")
        stations = stations.sort_values(["link_id", "station_id"])
        station_ids = stations.station_id.tolist()
        station_links = stations.link_id.tolist()
        unknown = set(station_links) - set(panel.link_index)
        if unknown:
            raise ValueError(f"Links absent from panel layout: {sorted(unknown)[:4]}")
        timestamps = sorted(frame.timestamp.unique().tolist())
        index = pd.MultiIndex.from_product([timestamps, station_ids], names=["timestamp", "station_id"])
        aligned = frame.set_index(["timestamp", "station_id"]).reindex(index)
        speed = pd.to_numeric(aligned.speed_kmh, errors="coerce").to_numpy(dtype=np.float32).reshape(len(timestamps), -1)
        flow = pd.to_numeric(aligned.flow_vph, errors="coerce").to_numpy(dtype=np.float32).reshape(len(timestamps), -1)
        lanes = np.array([panel.station_lanes.get(s, panel.lanes[panel.link_index[l]]) for s, l in zip(station_ids, station_links)], dtype=np.float32)
        lanes = np.clip(lanes, 1, 16)
        eligible = aligned.is_score_eligible.fillna(False).astype(bool).to_numpy(copy=True).reshape(len(timestamps), -1)
        eligible &= np.isfinite(speed) & np.isfinite(flow)
        occupancy = pd.to_numeric(aligned.occupancy, errors="coerce").to_numpy(np.float32).reshape(speed.shape) if "occupancy" in aligned else np.full_like(speed, np.nan)
        quality = pd.to_numeric(aligned.pct_observed, errors="coerce").fillna(0).to_numpy(np.float32).reshape(speed.shape) / 100 if "pct_observed" in aligned else eligible.astype(np.float32)
        return cls(timestamps, station_ids, station_links,
                   np.array([panel.link_index[l] for l in station_links], dtype=np.int64),
                   lanes, speed / 25.0, flow / lanes[None, :] / 600.0, eligible, occupancy, quality)

def aggregate_channels(day: StationDay, panel: PanelInfo, visible: np.ndarray | None = None):
    values = np.stack((day.speed, day.flow_lane,
                       day.occupancy if day.occupancy is not None else np.full_like(day.speed, np.nan)), -1)
    good = np.isfinite(values)
    if visible is not None:
        good &= visible[..., None] if visible.ndim == 2 else visible
    sums = np.zeros((len(day.timestamps), len(panel.links), 3), np.float32)
    counts = np.zeros_like(sums)
    quality = np.zeros(sums.shape[:2], np.float32)
    qcount = np.zeros_like(quality)
    for station, link in enumerate(day.station_to_link):
        sums[:, link] += np.where(good[:, station], values[:, station], 0)
        counts[:, link] += good[:, station]
        present = good[:, station].any(-1)
        q = day.quality[:, station] if day.quality is not None else present.astype(np.float32)
        # Observation quality remains public even when measured values are masked.
        quality[:, link] += np.where(np.isfinite(q), q, 0)
        qcount[:, link] += np.isfinite(q)
    return sums / np.maximum(counts, 1), counts > 0, quality / np.maximum(qcount, 1)

def ramp_files(release, panel, split, date):
    root = release / 'corridors' / panel.name / split / 'ramp_states'
    return sorted(set(root.glob(f"**/*{date}*.parquet"))
                  | set(root.glob(f"**/*{date.replace('-', '_')}*.parquet")))


def ramp_features(release: Path, panel: PanelInfo, split: str, timestamps: list[str]) -> np.ndarray:
    out = np.zeros((len(timestamps), len(panel.links), 4), np.float32)
    out[..., 2:] = 1  # A road without an attached ramp has known zero ramp flow.
    expected = np.zeros((len(panel.links), 2), np.float32)
    for link, kind in panel.ramp_map.values():
        expected[link, kind] += 1
    counts = np.zeros_like(out[..., :2])
    time_index = {value: i for i, value in enumerate(timestamps)}
    for path in ramp_files(release, panel, split, timestamps[0][:10]):
        frame = pd.read_parquet(path, columns=["timestamp", "ramp_link_id", "flow_vph", "pct_observed", "is_score_eligible"])
        frame = frame[frame.timestamp.isin(time_index)].drop_duplicates(["timestamp", "ramp_link_id"])
        for row in frame.itertuples(index=False):
            if row.ramp_link_id in panel.ramp_map and np.isfinite(row.flow_vph) and row.pct_observed >= 75 and bool(row.is_score_eligible):
                link, kind = panel.ramp_map[row.ramp_link_id]
                index = time_index[row.timestamp]
                out[index, link, kind] += float(row.flow_vph)
                counts[index, link, kind] += 1
    out[..., 2:] = counts >= expected[None]
    out[..., :2] *= out[..., 2:]
    return out

def dated_files(root):
    result = {}
    for path in sorted(Path(root).rglob("*.parquet")):
        match = re.search(r"\d{4}[-_]\d{2}[-_]\d{2}", path.name)
        if match is None:
            raise ValueError(f"Date missing from filename: {path}")
        date = match.group().replace("_", "-")
        if date in result:
            raise ValueError(f"Duplicate date: {date}")
        result[date] = path
    if not result:
        raise FileNotFoundError(f"No parquet files: {root}")
    return result

def fit_prior(panel, paths):
    """Only fit dates contribute to weekday/slot means and fallbacks."""
    sums = np.zeros((2016, len(panel.links), 3), np.float64)
    counts = np.zeros_like(sums)
    for number, path in enumerate(paths, 1):
        if number == 1 or number % 30 == 0 or number == len(paths):
            print(f"과거 평균 계산 {panel.name}: {number}/{len(paths)} days", flush=True)
        day = StationDay.load(path, panel)
        values, valid, _ = aggregate_channels(day, panel, day.eligible)
        dates = pd.to_datetime(day.timestamps, utc=True)
        slot = np.asarray(dates.dayofweek * 288 + dates.hour * 12 + dates.minute // 5)
        np.add.at(sums, slot, values * valid)
        np.add.at(counts, slot, valid)
    mean = sums.sum((0, 1)) / np.maximum(counts.sum((0, 1)), 1)
    link_mean = np.divide(sums.sum(0), counts.sum(0),
                          out=np.broadcast_to(mean, sums.shape[1:]).copy(),
                          where=counts.sum(0) > 0)
    return np.divide(sums, counts, out=np.broadcast_to(link_mean, sums.shape).copy(),
                     where=counts > 0).astype(np.float32)

class TargetIndex:
    def __init__(self, release, name, day, split='train'):
        path = release / 'task1' / name / split / 'sample_submission_state.csv'
        self.stations, self.links = day.station_ids, day.station_links
        self.archive, self.regimes = {}, {}
        station_lookup = {s: i for i, s in enumerate(self.stations)}
        parts = {}
        for frame in pd.read_csv(path, usecols=KEYS, dtype=str, chunksize=200_000):
            positions = frame.station_id.map(station_lookup)
            if positions.isna().any() or set(frame.panel) != {name}:
                raise ValueError(f'템플릿의 panel/station이 잘못되었습니다: {path}')
            if not np.array_equal(np.asarray(self.links)[positions.to_numpy(int)], frame.link_id.to_numpy()):
                raise ValueError(f'템플릿 station/link 연결이 다릅니다: {path}')
            times = pd.to_datetime(frame.timestamp, utc=True, format='ISO8601')
            if ((times.dt.minute % 5 != 0) | (times.dt.second != 0)).any():
                raise ValueError('템플릿은 5분 시각이어야 합니다.')
            slots = times.dt.hour * 12 + times.dt.minute // 5
            frame = frame.assign(_date=frame.timestamp.str[:10],
                                 _cell=(slots * len(self.stations) + positions).astype(np.uint32))
            for date, part in frame.groupby('_date', sort=False):
                regimes = part.mask_regime.unique()
                if len(regimes) != 1 or regimes[0] not in REGIMES:
                    raise ValueError(f'날짜별 regime이 잘못되었습니다: {date}')
                if date in self.regimes and self.regimes[date] != regimes[0]:
                    raise ValueError(f'날짜별 regime이 바뀝니다: {date}')
                self.regimes[date] = regimes[0]
                parts.setdefault(date, []).append(part._cell.to_numpy())
        if not parts:
            raise ValueError(f'빈 템플릿: {path}')
        self.archive = {d: np.concatenate(p) for d, p in parts.items()}
        if any(len(np.unique(a)) != len(a) for a in self.archive.values()):
            raise ValueError(f'중복 대상 셀: {path}')

    def get(self, day, require_eligible=True):
        date = day.timestamps[0][:10]
        if (day.station_ids, day.station_links) != (self.stations, self.links) or date not in self.archive:
            raise ValueError(f'대상 관측소 구성이 달라졌습니다: {date}')
        if any(t[:10] != date for t in day.timestamps):
            raise ValueError('하루 파일에 여러 날짜가 있습니다.')
        slots = {int(t[11:13]) * 12 + int(t[14:16]) // 5: i for i, t in enumerate(day.timestamps)}
        mask = np.zeros(day.speed.shape, bool)
        cells = self.archive[date]
        mask[[slots[int(s)] for s in cells // len(self.stations)], cells % len(self.stations)] = True
        if require_eligible and (mask & ~day.eligible).any():
            raise ValueError(f'정답이 없는 대상 셀: {date}')
        return self.regimes[date], mask

def available_panels(release):
    release = Path(release).expanduser().resolve()
    if not (release / 'corridors').is_dir() or not (release / 'task1').is_dir():
        raise FileNotFoundError(f'RELEASE_ROOT에 corridors/와 task1/이 필요합니다: {release}')
    available = sorted(p.name for p in (release / 'corridors').iterdir() if p.is_dir())
    names = available if PANELS is None else sorted(set(PANELS))
    if not names or set(names) - set(available):
        raise ValueError(f'잘못된 PANELS: {names}')
    return names

def station_graph(panel, day):
    # 무관측 connector를 따라 실제 도로에서 가장 가까운 관측 링크를 찾습니다.
    observed = set(day.station_to_link.tolist())
    neighbours = [set() for _ in panel.links]
    for a, b in panel.edges.T:
        neighbours[a].add(int(b))
        neighbours[b].add(int(a))
    links = np.eye(len(panel.links), dtype=bool)
    for origin in observed:
        pending, seen = list(neighbours[origin]), {origin}
        while pending:
            node = pending.pop()
            if node in seen:
                continue
            seen.add(node)
            if node in observed:
                links[origin, node] = True
            else:
                pending.extend(neighbours[node] - seen)
    return links[day.station_to_link[:, None], day.station_to_link[None, :]]

def prepare_panel(release, name, code_id, station_offset):
    started = time.perf_counter()
    panel = PanelInfo.load(release, name)
    files = dated_files(release / 'corridors' / name / 'train' / 'mainline_states')
    masked = dated_files(release / 'corridors' / name / 'train' / 'mainline_states_masked')
    dates = {p: [d for d in files if lo <= d <= hi] for p, (lo, hi) in PERIODS.items()}
    if name in DATE_OVERRIDES:
        saved_dates = DATE_OVERRIDES[name]
        if any(not set(saved_dates[p]) <= set(dates[p]) for p in PERIODS):
            raise ValueError(f'저장된 날짜가 로컬 데이터 기간과 다릅니다: {name}')
        dates = {p: list(saved_dates[p]) for p in PERIODS}
    if MAX_TRAIN_DAYS is not None:
        if MAX_TRAIN_DAYS < 1:
            raise ValueError('MAX_TRAIN_DAYS는 양수 또는 None이어야 합니다.')
        indices = np.linspace(0, len(dates['fit']) - 1, min(MAX_TRAIN_DAYS, len(dates['fit'])), dtype=int)
        dates['fit'] = [dates['fit'][i] for i in indices]
    if any(not dates[p] for p in ('fit', 'early', 'blend')):
        raise ValueError(f'학습·조기 종료·선택 구간이 모두 필요합니다: {name}')
    first = StationDay.load(files[dates['fit'][0]], panel)
    observed_links = np.unique(first.station_to_link)
    for attribute, fallback in (('free_speed', 100.), ('capacity', 1800. * panel.lanes), ('lengths', 1.)):
        values = getattr(panel, attribute)
        invalid = ~np.isfinite(values) | ((values < 0) if attribute == 'lengths' else (values <= 0))
        if invalid[observed_links].any():
            raise ValueError(f'관측 링크의 {attribute}가 잘못되었습니다: {name}')
        # 무관측 connector의 placeholder 값이 손실 연산에 NaN을 전파하지 않도록 합니다.
        setattr(panel, attribute, np.where(invalid, fallback, values).astype(np.float32))
    panel.jam_density = np.where(np.isfinite(panel.jam_density), panel.jam_density, 100 * panel.lanes)
    panel.jam_density = np.maximum(panel.jam_density, 1.05 * panel.capacity / panel.free_speed)
    target_index = TargetIndex(release, name, first)
    if DAYS_PER_REGIME is not None:
        for period, candidates in dates.items():
            chosen = []
            for regime in REGIMES:
                pool = [d for d in candidates if target_index.regimes.get(d) == regime]
                indices = np.linspace(0, len(pool) - 1, min(DAYS_PER_REGIME, len(pool)), dtype=int)
                chosen.extend(pool[i] for i in indices)
            dates[period] = sorted(chosen)
    for p in ('fit', 'early', 'blend'):
        if {target_index.regimes.get(d) for d in dates[p]} != set(REGIMES):
            raise ValueError(f'{name}/{p}의 R1·R2·R3가 부족합니다. 날짜 수를 늘려 주세요.')
        if not set(dates[p]) <= set(masked):
            raise ValueError(f'masked 날짜가 누락되었습니다: {name}/{p}')
    mode = LWR_MODES.get(name, 'A').upper()
    if mode not in ('A', 'B', 'C'):
        raise ValueError(f'지원하지 않는 LWR mode: {mode}')
    out = dict(release=release, panel=panel, files=files, masked=masked, dates=dates,
               targets=target_index, profile=load_fit_profile(panel, target_index, dates['fit'], files),
               panel_code=code_id, station_codes=np.arange(station_offset, station_offset + len(first.station_ids)),
               mapping=first.station_to_link, lanes=first.lanes, station_graph=station_graph(panel, first),
               mode=mode)
    topology = pd.read_csv(release / 'corridors' / name / 'network' / 'lwr_mainline_topology.csv', dtype=str)
    out['topology_nodes'] = {panel.link_index[l] for l in topology.link_id.dropna().astype(str)}
    out['preparation_seconds'] = time.perf_counter() - started
    print({'panel': name, 'stations': len(first.station_ids), 'days': {p: len(ds) for p, ds in dates.items()},
           'preparation_seconds': round(out['preparation_seconds'], 1)}, flush=True)
    return out

def link_mean_numpy(values, active, mapping, nodes):
    sums = np.zeros((len(values), nodes, values.shape[-1]), np.float64)
    counts = np.zeros((len(values), nodes), np.float64)
    np.add.at(sums.transpose(1, 0, 2), mapping, np.where(active[..., None], values, 0).transpose(1, 0, 2))
    np.add.at(counts.T, mapping, active.T)
    return sums / np.maximum(counts[..., None], 1), counts

def nearest_features(values, prior, elapsed):
    good = np.isfinite(values)
    t = np.arange(len(values))[:, None]
    left = np.maximum.accumulate(np.where(good, t, -1), axis=0)
    right = np.minimum.accumulate(np.where(good, t, len(values))[::-1], axis=0)[::-1]
    lv, rv = left >= 0, right < len(values)
    ls, rs = np.clip(left, 0, len(values) - 1), np.clip(right, 0, len(values) - 1)
    a, b = np.take_along_axis(values, ls, axis=0), np.take_along_axis(values, rs, axis=0)
    a = np.where(lv, a, np.where(rv, b, prior))
    b = np.where(rv, b, a)
    la, ra = elapsed[:, None] - elapsed[ls], elapsed[rs] - elapsed[:, None]
    fraction = np.divide(la, la + ra, out=np.zeros_like(la), where=(la + ra) > 0).clip(0, 1)
    base = a + fraction * (b - a)
    la, ra = np.where(lv, la, np.nan), np.where(rv, ra, np.nan)
    return base, la, ra

def macro_mean(panel_values):
    families = {}
    for name, value in panel_values.items():
        families.setdefault(name.rsplit('_', 1)[0], []).append(value)
    return float(np.mean([np.mean(values) for values in families.values()]))

AUX_NAMES = ('quality', 'speed_visible', 'flow_visible', 'occupancy_visible',
             'speed_left_age', 'speed_right_age', 'flow_left_age', 'flow_right_age',
             'profile_speed', 'profile_flow', 'profile_occupancy',
             'lanes', 'free_speed', 'capacity_per_lane', 'length',
             'ramp_in', 'ramp_out', 'ramp_in_valid', 'ramp_out_valid', 'on_attached', 'off_attached')


def restore_input_cache(names, archive_path, destination):
    destination = Path(destination).expanduser()
    destination.mkdir(parents=True, exist_ok=True)
    if archive_path is not None:
        archive_path = Path(archive_path).expanduser()
        if not archive_path.is_file():
            raise FileNotFoundError(f'저장 캐시 ZIP이 없습니다: {archive_path}. 새 캐시를 만들 때는 SAVED_CACHE_ZIP=None, CACHE_MODE="build"로 설정해 주세요.')
        with zipfile.ZipFile(archive_path) as archive:
            members = archive.infolist()
            for member in members:
                relative = PurePosixPath(member.filename)
                if (relative.is_absolute() or '..' in relative.parts or '\\' in member.filename or ':' in member.filename
                        or not (destination / member.filename).resolve().is_relative_to(destination.resolve())
                        or (member.external_attr >> 16) & 0o170000 == 0o120000):
                    raise ValueError(f'안전하지 않은 ZIP 경로: {member.filename}')
            fingerprint = hashlib.sha256(json.dumps(
                [(m.filename, m.file_size, m.CRC) for m in members], sort_keys=True).encode()).hexdigest()
            marker = destination / '.cache_source.json'
            if marker.exists() and json.loads(marker.read_text())['fingerprint'] != fingerprint:
                raise ValueError('다른 ZIP을 푼 폴더입니다. CACHE_DIR를 새 로컬 경로로 지정해 주세요.')
            pending = [m for m in members if not m.is_dir() and
                       (not marker.exists() or not (destination / m.filename).is_file())]
            if pending:
                needed = sum(m.file_size for m in pending)
                if shutil.disk_usage(destination).free < needed + 2 * 2**30:
                    raise RuntimeError(f'캐시 ZIP 복원에 {needed / 2**30:.1f} GiB + 여유 2 GiB가 필요합니다.')
                began = time.perf_counter()
                for number, member in enumerate(pending, 1):
                    target = destination / member.filename
                    target.parent.mkdir(parents=True, exist_ok=True)
                    temporary = target.with_suffix(target.suffix + '.tmp')
                    with archive.open(member) as src, temporary.open('wb') as dst:
                        shutil.copyfileobj(src, dst)
                    temporary.replace(target)
                    if number == 1 or number % 100 == 0 or number == len(pending):
                        print(f'저장 ZIP 복원: {number}/{len(pending)}파일', flush=True)
                marker_temporary = marker.with_suffix('.json.tmp')
                marker_temporary.write_text(json.dumps(dict(fingerprint=fingerprint, archive=str(archive_path))))
                marker_temporary.replace(marker)
                print({'cache_restore_seconds': round(time.perf_counter() - began, 1)})
            else:
                print('같은 ZIP이 이미 복원되어 로컬 NPZ를 재사용합니다.')
    roots = {p.parent.parent.parent for p in destination.rglob('*.npz')
             if p.parent.name in PERIODS and p.parent.parent.name in names
             and re.fullmatch(r'\d{4}-\d{2}-\d{2}\.npz', p.name)}
    if len(roots) > 1:
        raise ValueError('ZIP에 캐시 루트가 여러 개입니다. 필요한 폴더를 CACHE_DIR로 지정하고 SAVED_CACHE_ZIP=None으로 설정해 주세요.')
    if not roots and CACHE_MODE == 'reuse':
        raise ValueError('캐시의 <패널>/fit|early|blend/YYYY-MM-DD.npz 구조를 찾지 못했습니다. 다른 모델의 NPZ는 사용할 수 없습니다.')
    return next(iter(roots)) if roots else destination, destination


def source_signature(paths):
    records = [(Path(p).relative_to(RELEASE_ROOT).as_posix(), file_digest(p)) for p in sorted(set(paths))]
    return hashlib.sha256(json.dumps(records, sort_keys=True).encode()).hexdigest()


def profile_source_signature(panel, fit_dates, files):
    network = RELEASE_ROOT / 'corridors' / panel.name / 'network'
    return source_signature([files[d] for d in fit_dates] + list(network.glob('*.csv')))


def day_source_signature(context, date):
    paths = [context['files'][date], context['masked'][date]]
    paths += ramp_files(context['release'], context['panel'], 'train', date)
    digest = hashlib.sha256(source_signature(paths).encode())
    digest.update(np.sort(context['targets'].archive[date]).astype('<u4').tobytes())
    digest.update(context['targets'].regimes[date].encode())
    return digest.hexdigest()


def cache_is_current(path, source):
    marker = path.with_suffix('.sources.json')
    return marker.is_file() and json.loads(marker.read_text(encoding='utf-8')) == source


def write_cache_source(path, source):
    marker = path.with_suffix('.sources.json')
    temporary = marker.with_suffix('.json.tmp')
    temporary.write_text(json.dumps(source), encoding='utf-8')
    temporary.replace(marker)


def load_fit_profile(panel, targets, fit_dates, files):
    metadata = dict(format='maginet-input-v1', panel=panel.name, links=panel.links,
                    stations=targets.stations, station_links=targets.links,
                    fit_dates=fit_dates, aux_names=list(AUX_NAMES), scale=SCALE.tolist())
    key = hashlib.sha256(json.dumps(metadata, sort_keys=True).encode()).hexdigest()[:16]
    folder = CACHE_METADATA_DIR / 'profiles' / panel.name / key
    folder.mkdir(parents=True, exist_ok=True)
    profile_path, metadata_path = folder / 'fit_profile.npy', folder / 'cache_metadata.json'
    source = profile_source_signature(panel, fit_dates, files)
    legacy = CACHE_ROOT / panel.name
    legacy_metadata = legacy / 'cache_metadata.json'
    if legacy_metadata.exists() and json.loads(legacy_metadata.read_text()) != metadata:
        raise ValueError(f'이전 캐시의 학습 날짜·관측소·입력 형식이 다릅니다: {panel.name}')
    if not metadata_path.exists() and legacy_metadata.exists():
        legacy_profile = legacy / 'fit_profile.npy'
        if not legacy_profile.is_file():
            raise FileNotFoundError(f'이전 캐시 과거 평균이 누락되었습니다: {legacy_profile}')
        profile = np.load(legacy_profile, allow_pickle=False)
        if profile.shape != (2016, len(panel.links), 3) or not np.isfinite(profile).all():
            raise ValueError(f'이전 캐시 과거 평균의 형태·값이 잘못되었습니다: {panel.name}')
        temporary = profile_path.with_suffix('.npy.tmp')
        shutil.copyfile(legacy_profile, temporary)
        temporary.replace(profile_path)
        metadata_temporary = metadata_path.with_suffix('.json.tmp')
        metadata_temporary.write_text(json.dumps(metadata, ensure_ascii=False, sort_keys=True))
        metadata_temporary.replace(metadata_path)
    if metadata_path.exists():
        if json.loads(metadata_path.read_text()) != metadata:
            raise ValueError(f'저장 과거 평균의 학습 날짜·관측소·입력 형식이 다릅니다: {metadata_path}')
    if profile_path.is_file() and metadata_path.is_file() and cache_is_current(profile_path, source):
        profile = np.load(profile_path, allow_pickle=False)
    else:
        if CACHE_MODE == 'reuse':
            raise ValueError(f'{panel.name}: 과거 평균의 원본이 변경됐거나 출처 기록이 없습니다. --cache-mode build로 한 번 실행해 주세요.')
        profile = fit_prior(panel, [files[d] for d in fit_dates])
        candidates = [CACHE_PROFILE_DIR / f'fit_profile-{panel.name}.npy']
        candidates += sorted(CACHE_CONTAINER.rglob(f'fit_profile-{panel.name}.npy'))
        for candidate in dict.fromkeys(candidates):
            config_path = candidate.parent / 'config.json'
            if not candidate.is_file() or not config_path.is_file():
                continue
            saved = json.loads(config_path.read_text(encoding='utf-8')).get('panels', {}).get(panel.name, {})
            legacy_profile = np.load(candidate, allow_pickle=False)
            if (saved.get('stations') != targets.stations or saved.get('links') != targets.links
                    or saved.get('dates', {}).get('fit') != fit_dates
                    or hashlib.sha256(legacy_profile.tobytes()).hexdigest() != saved.get('profile_hash')
                    or not np.array_equal(legacy_profile, profile)):
                raise ValueError(f'이전 과거 평균이 현재 원본·관측소·fit 날짜와 다릅니다: {candidate}')
            break
        if profile.shape != (2016, len(panel.links), 3) or not np.isfinite(profile).all():
            raise ValueError(f'저장 과거 평균의 형태·값이 잘못되었습니다: {panel.name}')
        if not profile_path.is_file() or not np.array_equal(np.load(profile_path, allow_pickle=False), profile):
            temporary = profile_path.with_suffix('.npy.tmp')
            with temporary.open('wb') as stream:
                np.save(stream, profile, allow_pickle=False)
            temporary.replace(profile_path)
        metadata_temporary = metadata_path.with_suffix('.json.tmp')
        metadata_temporary.write_text(json.dumps(metadata, ensure_ascii=False, sort_keys=True))
        metadata_temporary.replace(metadata_path)
        write_cache_source(profile_path, source)
    if profile.shape != (2016, len(panel.links), 3) or not np.isfinite(profile).all():
        raise ValueError(f'캐시 과거 평균의 형태·값이 잘못되었습니다: {panel.name}')
    return profile.astype(np.float32, copy=False)


def validate_cache_targets(data, context, date):
    required = {'values', 'visible', 'aux', 'slot', 'weekday', 'base', 'truth', 'target',
                'active', 'observed', 'timestamps', 'ti', 'si', 'regime', 'ramp_valid', 'consecutive'}
    if required - set(data):
        raise ValueError(f'이 MagiNet 노트북의 NPZ 형식이 아닙니다: {date}, 누락 {sorted(required - set(data))}')
    target, timestamps = data['target'], data['timestamps']
    stations = len(context['targets'].stations)
    if (target.dtype != np.bool_ or target.ndim != 2 or target.shape != (len(timestamps), stations)
            or len(set(timestamps.tolist())) != len(timestamps)
            or any(str(t)[:10] != date for t in timestamps)):
        raise ValueError(f'캐시 날짜·관측소 형태가 다릅니다: {date}')
    ti, si = np.nonzero(target)
    cells = np.array([int(str(t)[11:13]) * 12 + int(str(t)[14:16]) // 5
                      for t in timestamps], np.int64)[ti] * stations + si
    expected = context['targets'].archive[date]
    if (not np.array_equal(np.sort(cells), np.sort(expected))
            or not np.array_equal(ti, data['ti']) or not np.array_equal(si, data['si'])
            or int(data['regime']) != REGIMES.index(context['targets'].regimes[date])):
        raise ValueError(f'캐시 대상 셀·regime이 템플릿과 다릅니다: {date}')
    if 'profile' in context:
        slot = data['weekday'] * 288 + data['slot']
        prior = context['profile'][slot[:, None], context['mapping'][None, :]].clip(-10, 10)
        if not np.allclose(data['aux'][..., 8:11], prior, atol=1e-6, rtol=1e-6):
            raise ValueError(f'캐시의 fit 과거 평균 특징이 현재 학습 구간과 다릅니다: {date}')
    return int(data['regime']), len(ti)


def fit_input_normalization(entries, cache_folder, mode=None, legacy_folder=None):
    mode = INPUT_NORMALIZATION if mode is None else mode
    fit = sorted((e for e in entries if e['period'] == 'fit'), key=lambda e: (e['panel'], e['date']))
    if not fit or mode not in ('fixed', 'standard'):
        raise ValueError('표준화에는 fit 캐시와 fixed/standard 모드가 필요합니다.')
    provenance = []
    for entry in fit:
        # NPZ의 배열 CRC는 ZIP 복원·경로 이동·수정 시각 변경 후에도 같습니다.
        with zipfile.ZipFile(entry['path']) as archive:
            arrays = [(name, archive.getinfo(name).file_size, archive.getinfo(name).CRC)
                      for name in ('values.npy', 'visible.npy')]
        provenance.append((entry['panel'], entry['date'], arrays))
    key = hashlib.sha256(json.dumps(dict(format='fit-visible-crc-v1', mode=mode, files=provenance),
                                   sort_keys=True).encode()).hexdigest()
    path = Path(cache_folder) / f'input_normalization-{key[:16]}.json'
    sources = ([Path(legacy_folder) / path.name] if legacy_folder is not None else [])
    sources.append(REPORT_BASE / 'full' / path.name)
    for source in sources:
        if path.exists():
            break
        if not source.is_file():
            continue
        saved = json.loads(source.read_text())
        if saved.get('source_signature') != key or saved.get('mode') != mode:
            raise ValueError('이전 실행의 입력 표준화 설정이 다릅니다.')
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_suffix('.json.tmp')
        temporary.write_text(json.dumps(saved, indent=2, ensure_ascii=False))
        temporary.replace(path)
    if path.exists():
        saved = json.loads(path.read_text())
        if saved.get('source_signature') != key or saved.get('mode') != mode:
            raise ValueError('저장된 입력 표준화 설정이 다릅니다.')
        return saved
    count, total, square = np.zeros(3, np.int64), np.zeros(3, np.float64), np.zeros(3, np.float64)
    if mode == 'standard':
        for number, entry in enumerate(fit, 1):
            with np.load(entry['path'], allow_pickle=False) as data:
                values, visible = data['values'], data['visible']
                if (values.shape != visible.shape or visible.dtype != np.bool_
                        or values.ndim != 3 or values.shape[-1] != 3):
                    raise ValueError(f'표준화 입력 형태가 다릅니다: {entry["path"]}')
                if not np.isfinite(values[visible]).all():
                    raise ValueError(f'관측된 입력에 NaN/Inf가 있습니다: {entry["path"]}')
                observed = np.where(visible, values, 0).astype(np.float64)
                count += visible.sum((0, 1))
                total += observed.sum((0, 1))
                square += np.square(observed).sum((0, 1))
            if number == 1 or number % 100 == 0 or number == len(fit):
                print(f'fit 관측 입력 표준화: {number}/{len(fit)}일', flush=True)
    mean = total / np.maximum(count, 1)
    std = np.sqrt(np.maximum(square / np.maximum(count, 1) - mean**2, 0))
    std = np.where(std > 1e-6, std, 1.)
    saved = dict(mode=mode, mean=mean.tolist(), std=std.tolist(), count=count.tolist(),
                 source_signature=key, fitted_period='fit', channels=['speed/25', 'flow_per_lane/600', 'occupancy'])
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix('.json.tmp')
    temporary.write_text(json.dumps(saved, indent=2, ensure_ascii=False))
    temporary.replace(path)
    return saved


def attach_vocabulary(contexts):
    families = sorted({n.rsplit('_', 1)[0] for n in contexts})
    for name, c in contexts.items():
        family, direction = name.rsplit('_', 1)
        if direction not in ('N', 'S', 'E', 'W'):
            raise ValueError(f'알 수 없는 방향: {name}')
        c['family_code'] = families.index(family)
        c['direction_code'] = ('N', 'S', 'E', 'W').index(direction)


def chebyshev_supports(adjacency, order=None):
    order = CHEB_K if order is None else order
    adjacency = np.asarray(adjacency, np.float64)
    if adjacency.ndim != 2 or adjacency.shape[0] != adjacency.shape[1] or not np.allclose(adjacency, adjacency.T):
        raise ValueError('Chebyshev 그래프는 대칭 인접행렬이어야 합니다.')
    degree = adjacency.sum(-1)
    inv = 1 / np.sqrt(np.maximum(degree, 1))
    identity = np.eye(len(adjacency))
    laplacian = identity - inv[:, None] * adjacency * inv[None, :]
    largest = float(np.linalg.eigvalsh(laplacian).max())
    scaled = 2 * laplacian / largest - identity if largest > 1e-10 else -identity
    supports = [identity]
    if order > 1:
        supports.append(scaled)
    for _ in range(2, order):
        supports.append(2 * scaled @ supports[-1] - supports[-2])
    return np.stack(supports).astype(np.float32)


def input_arrays(day, context, split='train'):
    panel, mapping = context['panel'], day.station_to_link
    values = np.stack((day.speed, day.flow_lane, day.occupancy), -1)
    visible = np.isfinite(values)
    dates = pd.to_datetime(day.timestamps, utc=True, format='ISO8601')
    elapsed = np.asarray((dates - dates[0]).total_seconds() / 60, np.float32)
    if len(elapsed) < 2 or (np.diff(elapsed) <= 0).any():
        raise ValueError('하루 시각은 서로 다른 오름차순이어야 합니다.')
    slot, weekday = np.asarray(dates.hour * 12 + dates.minute // 5), np.asarray(dates.dayofweek)
    prior = context['profile'][(weekday * 288 + slot)[:, None], mapping[None, :]]
    baseline, ages = [], []
    for channel, data in enumerate((day.speed, day.flow_lane)):
        base, left_age, right_age = nearest_features(data, prior[..., channel], elapsed)
        baseline.append(base)
        ages += [np.nan_to_num(age / 360, nan=4.).clip(0, 4) for age in (left_age, right_age)]
    ramps = ramp_features(context['release'], panel, split, day.timestamps)
    r = ramps[:, mapping]
    attached = np.zeros((len(panel.links), 2), np.float32)
    for link, kind in panel.ramp_map.values():
        attached[link, kind] = 1
    spread = lambda v: np.broadcast_to(v, day.speed.shape)
    aux = np.stack((day.quality, visible[..., 0], visible[..., 1], visible[..., 2], *ages,
                    prior[..., 0], prior[..., 1], prior[..., 2], spread(day.lanes / 5),
                    spread(panel.free_speed[mapping] / 25), spread(panel.capacity[mapping] / panel.lanes[mapping] / 600),
                    spread(panel.lengths[mapping]), r[..., 0] / 600, r[..., 1] / 600, r[..., 2], r[..., 3],
                    spread(attached[mapping, 0]), spread(attached[mapping, 1])), -1)
    if aux.shape[-1] != len(AUX_NAMES) or np.isinf(values).any():
        raise ValueError('입력 배열이 잘못되었습니다.')
    # 수치상 placeholder 0은 mask로 제외되며 관측값 임베딩으로 사용되지 않습니다.
    values = np.where(visible, values, 0).astype(np.float32)
    aux = np.nan_to_num(aux, nan=0., posinf=10., neginf=-10.).clip(-10, 10).astype(np.float32)
    base = np.stack(baseline, -1).astype(np.float32)
    if not np.isfinite(base).all():
        raise ValueError('기준 보간이 유한하지 않습니다.')
    return dict(values=values, visible=visible, aux=aux, slot=slot.astype(np.int64),
                weekday=weekday.astype(np.int64), base=base,
                ramp_valid=(ramps[..., 2] > 0) & (ramps[..., 3] > 0))


def consecutive_five_minutes(timestamps):
    times = pd.to_datetime(timestamps, utc=True, format='ISO8601')
    return np.asarray((times[1:] - times[:-1]) == pd.Timedelta(minutes=5))


def make_day(context, date, split='train', target_index=None, input_files=None):
    panel = context['panel']
    path = context['masked'][date] if split == 'train' else input_files[date]
    day = StationDay.load(path, panel)
    if split == 'train':
        truth = StationDay.load(context['files'][date], panel)
        if (day.timestamps, day.station_ids, day.station_links) != (truth.timestamps, truth.station_ids, truth.station_links):
            raise ValueError(f'정답과 masked 입력이 다릅니다: {date}')
        regime, target = context['targets'].get(truth)
    else:
        truth = None
        regime, target = target_index.get(day, require_eligible=False)
    if (day.station_ids, day.station_links) != (context['targets'].stations, context['targets'].links):
        raise ValueError(f'관측소 구성이 다릅니다: {date}')
    path_regimes = [p.name.split('=', 1)[1] for p in path.parents if p.name.startswith('mask_regime=')]
    if path_regimes and path_regimes[0] != regime:
        raise ValueError(f'입력과 템플릿의 regime이 다릅니다: {date}')
    if np.isfinite(day.speed[target]).any() or np.isfinite(day.flow_lane[target]).any():
        raise ValueError(f'대상 정답이 입력에 보입니다: {date}')
    d = input_arrays(day, context, split)
    observed_values = np.stack((day.speed * 25, day.flow_lane * 600 * day.lanes), -1)
    observed = np.isfinite(observed_values).all(-1)
    active = (observed & (truth.eligible if truth is not None else day.eligible)) | target
    labels = np.stack((truth.speed, truth.flow_lane), -1) if truth is not None else np.zeros((*day.speed.shape, 2), np.float32)
    ti, si = np.nonzero(target)
    d.update(truth=np.nan_to_num(labels), target=target, active=active,
             observed=np.nan_to_num(observed_values), timestamps=np.asarray(day.timestamps), ti=ti, si=si,
             regime=np.array(REGIMES.index(regime), np.int32),
             consecutive=consecutive_five_minutes(day.timestamps))
    return d




def cache_context(context, cache_root, date_overrides=None, periods=('fit', 'early', 'blend')):
    started, entries = time.perf_counter(), []
    for period in periods:
        dates = context['dates'][period] if date_overrides is None else date_overrides[period]
        folder = cache_root / context['panel'].name / period
        folder.mkdir(parents=True, exist_ok=True)
        for number, date in enumerate(dates, 1):
            path = folder / f'{date}.npz'
            source = day_source_signature(context, date)
            if not path.exists() or not cache_is_current(path, source):
                if CACHE_MODE == 'reuse':
                    if path.exists():
                        raise ValueError(f'캐시 원본이 변경됐거나 출처 기록이 없습니다: {path}. --cache-mode build로 실행해 주세요.')
                    raise FileNotFoundError(f'저장 캐시가 누락되었습니다: {path}. 원본에서 생성하려면 CACHE_MODE="build"로 변경해 주세요.')
                if shutil.disk_usage(cache_root).free < 2 * 2**30:
                    raise RuntimeError('임시 캐시 디스크 여유 공간이 2 GiB 미만입니다.')
                cache_action = '생성'
                d = make_day(context, date)
                regime, targets = validate_cache_targets(d, context, date)
                same = False
                if path.exists():
                    with np.load(path, allow_pickle=False) as saved:
                        same = set(saved.files) == set(d) and all(np.array_equal(saved[k], v) for k, v in d.items())
                if not same:
                    temporary = path.with_suffix('.npz.tmp')
                    with temporary.open('wb') as stream:
                        np.savez_compressed(stream, **d)
                    temporary.replace(path)
                write_cache_source(path, source)
                load_day.cache_clear()
            else:
                cache_action = '재사용 확인'
                with np.load(path, allow_pickle=False) as d:
                    regime, targets = validate_cache_targets(d, context, date)
            entries.append(dict(panel=context['panel'].name, date=date, period=period, regime=regime,
                                targets=targets, path=str(path)))
            if number == 1 or number % 20 == 0 or number == len(dates):
                print(f"입력 캐시 {cache_action} {context['panel'].name}/{period}: {number}/{len(dates)}일", flush=True)
    context['cache_seconds'] = context.get('cache_seconds', 0.) + time.perf_counter() - started
    return entries


@lru_cache(maxsize=2)
def load_day(path):
    with np.load(path, allow_pickle=False) as d:
        result = {k: d[k] for k in d.files}
    values, visible, base, aux = (result[k] for k in ('values', 'visible', 'base', 'aux'))
    if (values.ndim != 3 or values.shape[-1] != 3 or visible.dtype != np.bool_
            or visible.shape != values.shape or base.shape != values.shape[:2] + (2,)
            or aux.shape != values.shape[:2] + (len(AUX_NAMES),)
            or not np.isfinite(values[visible]).all() or not np.isfinite(base).all()
            or not np.isfinite(aux).all() or (base < 0).any()
            or visible[result['target'], :2].any()):
        raise ValueError(f'캐시 배열 형태·관측 마스크·보간값이 잘못되었습니다: {path}')
    return result




def window_starts(length):
    if length <= WINDOW:
        return [0]
    return sorted(set(range(0, length - WINDOW + 1, STRIDE)) | {length - WINDOW})

def masked_softmax(scores, valid):
    valid = valid.to(dtype=torch.bool)
    weights = scores.float().masked_fill(~valid, torch.finfo(torch.float32).min).softmax(-1)
    weights = torch.where(valid, weights, 0.)
    return weights / weights.sum(-1, keepdim=True).clamp_min(1e-8)


class AMSTEncoder(nn.Module):
    def __init__(self, station_count, panel_count, family_count):
        super().__init__()
        self.value_weight = nn.Parameter(torch.empty(3, WIDTH))
        self.value_bias = nn.Parameter(torch.zeros(3, WIDTH))
        self.missing_token = nn.Parameter(torch.empty(3, WIDTH))
        self.slot = nn.Embedding(288, WIDTH)
        self.weekday = nn.Embedding(7, WIDTH)
        self.relative = nn.Embedding(288, WIDTH)
        self.station = nn.Embedding(station_count, WIDTH)
        self.panel = nn.Embedding(panel_count, WIDTH)
        self.family = nn.Embedding(family_count, WIDTH)
        self.direction = nn.Embedding(4, WIDTH)
        self.regime = nn.Embedding(3, WIDTH)
        self.aux = nn.Linear(len(AUX_NAMES), WIDTH)
        self.base_embedding = nn.Linear(2, WIDTH)
        self.norm = nn.LayerNorm(WIDTH)
        self.dropout = nn.Dropout(DROPOUT)
        nn.init.xavier_uniform_(self.value_weight)
        nn.init.normal_(self.missing_token, std=0.02)

    def forward(self, values, visible, aux, base, slot, weekday, regime, ct):
        # values/visible: B×T×station×3; 결측 placeholder는 여기서 제외합니다.
        tokens = values[..., None] * self.value_weight + self.value_bias
        tokens = torch.where(visible[..., None], tokens, self.missing_token)
        h = tokens.sum(-2) / np.sqrt(3.)
        t = len(values[0])
        time_embedding = self.slot(slot) + self.weekday(weekday) + self.relative(torch.arange(t, device=values.device))[None]
        node_embedding = (self.station(ct['station_codes']) + self.panel(ct['panel_code'])
                          + self.family(ct['family_code']) + self.direction(ct['direction_code']))
        h = h + self.aux(aux) + self.base_embedding(base) + time_embedding[:, :, None] + node_embedding[None, None] + self.regime(regime)[:, None, None]
        return self.dropout(self.norm(h)).permute(0, 2, 1, 3), node_embedding


class MaskTemporalAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.qkv = nn.Linear(WIDTH, 3 * WIDTH, bias=False)
        self.project = nn.Linear(WIDTH, WIDTH)
        self.norm = nn.LayerNorm(WIDTH)
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, h, visible, residual_attention):
        b, s, t, d = h.shape
        q, k, v = self.qkv(h).reshape(b, s, t, 3, HEADS, d // HEADS).permute(3, 0, 1, 4, 2, 5).unbind(0)
        scores = (q.float() @ k.float().transpose(-1, -2)) / np.sqrt(d // HEADS)
        if residual_attention is not None:
            scores = scores + residual_attention
        key_valid = visible.any(-1).permute(0, 2, 1)[:, :, None, None, :]
        weights = masked_softmax(scores, key_valid)
        attended = (weights.to(v.dtype) @ v).transpose(2, 3).reshape(b, s, t, d)
        return self.norm(h + self.dropout(self.project(attended))), scores


class MaskSpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.summary = nn.Linear(2 * WIDTH, GRAPH_DIM)
        self.position = nn.Linear(WIDTH, GRAPH_DIM)
        self.qk = nn.Linear(GRAPH_DIM, 2 * CHEB_K * 16, bias=False)
        self.mask_pair = nn.Linear(GRAPH_DIM, 2 * CHEB_K * 8, bias=False)
        self.norm = nn.LayerNorm(GRAPH_DIM)

    def forward(self, h, visible, node_embedding):
        b, s, t, d = h.shape
        valid = visible.any(-1).permute(0, 2, 1).float()
        observed_summary = (h * valid[..., None]).sum(2) / valid.sum(2).clamp_min(1)[..., None]
        summary = self.summary(torch.cat((observed_summary, h.mean(2)), -1))
        # MagiNet의 관측 문맥 기반 missing spatial graph를 유지합니다.
        similarity = F.relu(summary.float() @ summary.float().transpose(-1, -2))
        missing_graph = (similarity.softmax(-1) > MGL_THRESHOLD).float()
        positioned = self.norm(summary + self.position(node_embedding)[None])
        q, k = self.qk(positioned).reshape(b, s, 2, CHEB_K, 16).permute(2, 0, 3, 1, 4).unbind(0)
        scores = (q.float() @ k.float().transpose(-1, -2)) / 4.
        a, z = self.mask_pair(positioned).reshape(b, s, 2, CHEB_K, 8).permute(2, 0, 3, 1, 4).unbind(0)
        pair_mask = torch.tanh((a.float() @ z.float().transpose(-1, -2)) / np.sqrt(8.))
        reliability = 0.25 + 0.75 * valid.mean(2)
        scores = scores + missing_graph[:, None] * pair_mask + reliability.log()[:, None, None, :]
        return scores.softmax(-1)


class MaskAwareChebConv(nn.Module):
    def __init__(self):
        super().__init__()
        self.theta = nn.Parameter(torch.empty(CHEB_K, WIDTH, WIDTH))
        nn.init.xavier_uniform_(self.theta)

    def forward(self, h, spatial_attention, supports):
        weights = spatial_attention * supports[None]
        output = torch.einsum('bkij,bjtd,kdf->bitf', weights.to(h.dtype), h, self.theta.to(h.dtype))
        return F.relu(output)


def observation_context(visible, aux, neighbors):
    """Public observations only; physical neighbors exclude the station itself."""
    with torch.autocast(device_type=visible.device.type, enabled=False):
        joint = visible[..., :2].all(-1).float()
        counts = torch.einsum('ij,btj->bti', neighbors.float(), joint)
        degree = neighbors.float().sum(-1)[None, None]
        features = torch.cat((visible.float(), (counts / (1 + counts))[..., None],
                              (counts / degree.clamp_min(1))[..., None],
                              (degree > 0).expand_as(counts)[..., None].float(),
                              aux[..., 4:8].float().clamp(0, 4) / 4), -1)
    return features.permute(0, 2, 1, 3)


def gather_visible_spatial(h, spatial_attention, visible):
    with torch.autocast(device_type=h.device.type, enabled=False):
        weights = spatial_attention.float().mean(1)
        valid = visible.any(-1).permute(0, 2, 1).float()
        numerator = torch.einsum('bij,bjtd->bitd', weights, h.float() * valid[..., None])
        denominator = torch.einsum('bij,bjt->bit', weights, valid)
        return (numerator / denominator.clamp_min(1e-8)[..., None]).to(h.dtype)




class ChannelSpatialReadout(nn.Module):
    """Channel-specific spatial residuals; the shared backbone has no extra learned graph."""
    def __init__(self):
        super().__init__()
        self.attention = MaskSpatialAttention()
        self.project = nn.Linear(WIDTH, WIDTH, bias=False)
        self.correction = nn.Linear(WIDTH, 2)
        self.gate = nn.Linear(10, 2)
        nn.init.zeros_(self.correction.weight)
        nn.init.zeros_(self.correction.bias)
        nn.init.zeros_(self.gate.weight)
        nn.init.constant_(self.gate.bias, -3.)

    def forward(self, h, visible, node_embedding, context):
        attention = self.attention(h, visible, node_embedding)
        gathered = gather_visible_spatial(h, attention, visible)
        correction = self.correction(F.gelu(self.project(gathered))).float()
        # 관측소 전체가 비어 있는 시각에는 bias만으로 공간 보정을 만들지 않습니다.
        available = visible.any(-1).any(-1)[:, None, :, None]
        return correction * self.gate(context).float().sigmoid() * available


class GatedTemporalUnit(nn.Module):
    def __init__(self, kernel):
        super().__init__()
        self.conv = nn.Conv2d(WIDTH, 2 * WIDTH, kernel_size=(1, kernel), padding=(0, kernel // 2))

    def forward(self, h):
        a, b = self.conv(h).chunk(2, dim=1)
        return a.tanh() * b.sigmoid()


class MASTBlock(nn.Module):
    def __init__(self):
        super().__init__()
        self.temporal_attention = MaskTemporalAttention()
        self.spatial_attention = MaskSpatialAttention()
        self.spatial_conv = MaskAwareChebConv()
        self.gates = nn.ModuleList(GatedTemporalUnit(k) for k in (3, 5, 7))
        self.time_mix = nn.Conv2d(3 * WIDTH, WIDTH, kernel_size=1)
        self.norm = nn.LayerNorm(WIDTH)
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, h, visible, node_embedding, supports, residual_attention, context=None):
        attended, residual_attention = self.temporal_attention(h, visible, residual_attention)
        spatial_attention = self.spatial_attention(attended, visible, node_embedding)
        signal = attended
        spatial = self.spatial_conv(signal, spatial_attention, supports)
        channels = spatial.permute(0, 3, 1, 2)
        gated = self.time_mix(torch.cat([gate(channels) for gate in self.gates], dim=1)).permute(0, 2, 3, 1)
        output = self.norm(h + self.dropout(F.relu(spatial + gated)))
        return output, residual_attention


class MagiNetTask13(nn.Module):
    def __init__(self, station_count, panel_count, family_count, normalization):
        super().__init__()
        mean = torch.as_tensor(normalization['mean'], dtype=torch.float32)
        std = torch.as_tensor(normalization['std'], dtype=torch.float32)
        if mean.shape != (3,) or std.shape != (3,) or not torch.isfinite(mean).all() or not torch.isfinite(std).all() or (std <= 0).any():
            raise ValueError('입력 표준화의 평균·표준편차가 잘못되었습니다.')
        self.register_buffer('input_mean', mean)
        self.register_buffer('input_std', std)
        self.AMSTenc = AMSTEncoder(station_count, panel_count, family_count)
        self.MASTdec = nn.ModuleList(MASTBlock() for _ in range(BLOCKS))
        self.fuse = nn.Linear((BLOCKS + 1) * WIDTH, WIDTH)
        self.readout = nn.GRU(WIDTH, WIDTH // 2, batch_first=True, bidirectional=True)
        self.speed_head = nn.Sequential(nn.Linear(WIDTH, WIDTH // 2), nn.GELU(), nn.Linear(WIDTH // 2, 1))
        self.flow_head = nn.Sequential(nn.Linear(WIDTH, WIDTH // 2), nn.GELU(), nn.Linear(WIDTH // 2, 1))
        for head in (self.speed_head, self.flow_head):
            # 보정량은 양·음 모두 허용하며 시작 출력은 기존 보간과 같습니다.
            nn.init.zeros_(head[-1].weight)
            nn.init.zeros_(head[-1].bias)
        self.channel_spatial = ChannelSpatialReadout()

    def forward(self, values, visible, aux, base, slot, weekday, regime, ct):
        normalized = torch.where(visible, (values.float() - self.input_mean) / self.input_std, 0.)
        normalized_base = (base.float() - self.input_mean[:2]) / self.input_std[:2]
        normalized_aux = aux.float().clone()
        normalized_aux[..., 8:11] = (normalized_aux[..., 8:11] - self.input_mean) / self.input_std
        h, node_embedding = self.AMSTenc(normalized, visible, normalized_aux, normalized_base, slot, weekday, regime, ct)
        context = observation_context(visible, aux, ct['neighbors'])
        outputs, residual_attention = [h], None
        for block in self.MASTdec:
            h, residual_attention = block(h, visible, node_embedding, ct['supports'], residual_attention, context)
            outputs.append(h)
        h = F.gelu(self.fuse(torch.cat(outputs, dim=-1)))
        b, s, t, d = h.shape
        # cuDNN의 BF16 RNN 지원 여부와 무관하게 시간 readout은 FP32로 계산합니다.
        with torch.autocast(device_type=h.device.type, enabled=False):
            h, _ = self.readout(h.float().reshape(b * s, t, d))
        h = h.reshape(b, s, t, d).permute(0, 2, 1, 3)
        correction = torch.cat((self.speed_head(h), self.flow_head(h)), -1).float()
        correction = correction + self.channel_spatial(h.permute(0, 2, 1, 3), visible,
                                                       node_embedding, context).permute(0, 2, 1, 3)
        # 입력 표준화와 별개로 출력·State 손실은 /25·/600 단위를 유지합니다.
        return (base.float() + correction).clamp_min(0.)


def tensor_context(context, device=None):
    device = DEVICE if device is None else device
    panel = context['panel']
    mapping = torch.as_tensor(context['mapping'], dtype=torch.long, device=device)
    neighbors = np.asarray(context['station_graph'], dtype=np.float32).copy()
    np.fill_diagonal(neighbors, 0)
    return dict(station_codes=torch.as_tensor(context['station_codes'], dtype=torch.long, device=device),
                panel_code=torch.tensor(context['panel_code'], dtype=torch.long, device=device),
                family_code=torch.tensor(context['family_code'], dtype=torch.long, device=device),
                direction_code=torch.tensor(context['direction_code'], dtype=torch.long, device=device),
                supports=torch.as_tensor(chebyshev_supports(context['station_graph']), device=device),
                neighbors=torch.as_tensor(neighbors, device=device),
                matrix=F.one_hot(mapping, num_classes=len(panel.links)).float(),
                lanes=torch.as_tensor(context['lanes'], device=device),
                link_lanes=torch.as_tensor(panel.lanes, device=device),
                lengths=torch.as_tensor(panel.lengths, device=device), free=torch.as_tensor(panel.free_speed, device=device),
                capacity=torch.as_tensor(panel.capacity, device=device), jam=torch.as_tensor(panel.jam_density, device=device))


def window_batch(d, starts, device=None):
    device = DEVICE if device is None else device
    lengths = [min(WINDOW, len(d['values']) - start) for start in starts]
    if len(set(lengths)) != 1:
        raise ValueError('한 batch의 시간 길이가 다릅니다.')
    return {key: torch.as_tensor(np.stack([d[key][start:start+length] for start, length in zip(starts, lengths)]),
                                 device=device, dtype=dtype)
            for key, dtype in (('values', torch.float32), ('visible', torch.bool), ('aux', torch.float32),
                                ('base', torch.float32), ('slot', torch.long), ('weekday', torch.long))} | {
               'regime': torch.full((len(starts),), int(d['regime']), dtype=torch.long, device=device)}


def panel_batch_size(context):
    name = context['panel'].name
    if name in PANEL_BATCH:
        return PANEL_BATCH[name]
    if BATCH_POLICY == 'adaptive':
        return 8 if len(context['mapping']) <= 128 else 4
    return BATCH_SIZE


@torch.inference_mode()
def maginet_predict(model, d, ct, batch_size=None):
    batch_size = BATCH_SIZE if batch_size is None else batch_size
    model.eval()
    device = next(model.parameters()).device
    sums = np.zeros_like(d['base'], np.float64)
    weights = np.zeros((len(d['base']), 1, 1), np.float64)
    starts = window_starts(len(d['values']))
    for offset in range(0, len(starts), batch_size):
        chunk = starts[offset:offset+batch_size]
        batch = window_batch(d, chunk, device)
        with torch.autocast(device_type=device.type, dtype=AMP_DTYPE, enabled=USE_AMP and device.type == 'cuda'):
            outputs = model(**batch, ct=ct).float().cpu().numpy()
        for start, output in zip(chunk, outputs):
            w = np.hanning(len(output) + 2)[1:-1, None, None]
            sums[start:start+len(output)] += output * w
            weights[start:start+len(output)] += w
    if (weights == 0).any():
        raise ValueError('겹치는 창 예측에 누락 시각이 있습니다.')
    result = (sums / weights).astype(np.float32)
    if not np.isfinite(result).all() or (result < 0).any():
        raise FloatingPointError('MagiNet 출력이 잘못되었습니다.')
    return result

def observability_mask(context, counts, ramp_valid, affected, consecutive):
    panel = context['panel']
    present = (counts > 0).any(0)
    topology_nodes = context['topology_nodes']
    observable = present & np.array([i in topology_nodes for i in range(len(panel.links))])
    for a, b in panel.edges.T:
        if a in topology_nodes and not present[a]:
            observable[b] = False
        if b in topology_nodes and not present[b]:
            observable[a] = False
    if context['mode'] == 'C':
        for link, _ in panel.ramp_map.values():
            observable[link] = False
    transition = (counts[:-1] > 0) & (counts[1:] > 0) & observable[None] & consecutive[:, None]
    if context['mode'] in ('A', 'B'):
        transition &= ramp_valid[:-1]
    return transition, (affected[:-1] | affected[1:])

def completed_link_fields(prediction_physical, d, context):
    values = d['observed'].copy()
    ti, si = d['ti'], d['si']
    values[ti, si, 0] = prediction_physical[:, 0]
    values[ti, si, 1] = prediction_physical[:, 1] * context['lanes'][si]
    links, counts = link_mean_numpy(values, d['active'], context['mapping'], len(context['panel'].links))
    affected = link_mean_numpy(d['target'][..., None].astype(float), d['active'],
                              context['mapping'], len(context['panel'].links))[0][..., 0] > 0
    return links, counts, affected

def physics_totals(prediction, d, context):
    panel = context['panel']
    links, counts, affected = completed_link_fields(prediction, d, context)
    truth = d['truth'][d['ti'], d['si']] * SCALE
    reference, _, _ = completed_link_fields(truth, d, context)
    n = links[..., 1] / np.maximum(links[..., 0], 1) * panel.lengths
    true_n = reference[..., 1] / np.maximum(reference[..., 0], 1) * panel.lengths
    good_n = (counts > 0) & affected
    error_n = (n - true_n)[good_n]
    transition, target_transition = observability_mask(context, counts, d['ramp_valid'], affected, d['consecutive'])
    error_dn_all = (np.diff(n, axis=0) - np.diff(true_n, axis=0))[transition]
    truth_dn_all = np.diff(true_n, axis=0)[transition]
    error_dn_target = (np.diff(n, axis=0) - np.diff(true_n, axis=0))[transition & target_transition]
    density = links[..., 1] / np.maximum(links[..., 0], 1)
    critical = panel.capacity / panel.free_speed
    wave = panel.capacity / np.maximum(panel.jam_density - critical, 1e-6)
    qfd = np.where(density <= critical, panel.free_speed * density,
                   wave * (panel.jam_density - density)).clip(0)
    good_fd = counts > 0
    fd_error = ((links[..., 1] - qfd) / panel.lanes)[good_fd]
    q_lane = (links[..., 1] / panel.lanes)[good_fd]
    fraction = link_mean_numpy(d['target'][..., None].astype(float), d['active'],
                               context['mapping'], len(panel.links))[0][..., 0]
    submitted = good_fd & (fraction > 0.5)
    return dict(n_sae=float(np.abs(error_n).sum()), n_sse=float(np.square(error_n).sum()), n_count=len(error_n),
                dn_sae=float(np.abs(error_dn_target).sum()), dn_sse=float(np.square(error_dn_target).sum()),
                dn_count=len(error_dn_target), lwr_num=float(np.abs(error_dn_all).sum()),
                lwr_den=float(np.abs(truth_dn_all).sum()), lwr_count=len(error_dn_all),
                fd_num=float(np.abs(fd_error).sum()), fd_den=float(np.abs(q_lane).sum()),
                fd_count=int(good_fd.sum()), submitted_count=int(submitted.sum()),
                low_flow=int((links[..., 1][submitted] < 50).sum()))

def state_totals(truth, prediction):
    error = np.asarray(prediction, np.float64) - truth
    return dict(count=len(error), speed_sse=float(np.square(error[:, 0]).sum()),
                flow_sse=float(np.square(error[:, 1]).sum()), speed_sae=float(np.abs(error[:, 0]).sum()),
                flow_sae=float(np.abs(error[:, 1]).sum()))

def summarize(records):
    daily = pd.DataFrame(records)
    if daily.empty:
        raise ValueError('평가 대상이 없습니다.')
    keys = ['candidate', 'alpha', 'panel', 'regime']
    sums = daily.groupby(keys, as_index=False).sum(numeric_only=True)
    expected = {(p, r) for p in daily.panel.unique() for r in range(3)}
    for _, part in sums.groupby('candidate'):
        if set(zip(part.panel, part.regime)) != expected or (part['count'] == 0).any():
            raise ValueError('평가의 panel×regime이 누락되었습니다.')
    sums['speed_rmse_kmh'] = np.sqrt(sums.speed_sse / sums['count'])
    sums['flow_rmse_per_lane'] = np.sqrt(sums.flow_sse / sums['count'])
    sums['speed_mae_kmh'] = sums.speed_sae / sums['count']
    sums['flow_mae_per_lane'] = sums.flow_sae / sums['count']
    sums['State'] = 0.54 * np.maximum(0, 1 - sums.speed_rmse_kmh / 25) + 0.46 * np.maximum(0, 1 - sums.flow_rmse_per_lane / 600)
    if 'lwr_num' in sums:
        sums['N_MAE_proxy'] = sums.n_sae / sums.n_count.replace(0, np.nan)
        sums['DN_MAE_proxy'] = sums.dn_sae / sums.dn_count.replace(0, np.nan)
        sums['LWR_proxy'] = np.maximum(0, 1 - sums.lwr_num / (sums.lwr_den + 1e-6))
        sums['FD_proxy'] = np.maximum(0, 1 - sums.fd_num / (sums.fd_den + 1e-6))
        sums.loc[sums.lwr_count == 0, 'LWR_proxy'] = 0.
        sums.loc[sums.fd_count == 0, 'FD_proxy'] = 0.
        sums['low_flow_fraction'] = sums.low_flow / sums.submitted_count.replace(0, np.nan)
        sums.loc[sums.low_flow_fraction > 0.20, 'FD_proxy'] = 0.
        sums['Physics_proxy'] = (sums.FD_proxy + 2 * sums.LWR_proxy) / 3
    metrics = ['State', 'speed_rmse_kmh', 'flow_rmse_per_lane', 'speed_mae_kmh', 'flow_mae_per_lane']
    metrics += [n for n in ('N_MAE_proxy', 'DN_MAE_proxy', 'FD_proxy', 'LWR_proxy', 'Physics_proxy') if n in sums]
    rows = []
    for name, part in sums.groupby('candidate', sort=False):
        per_panel = part.groupby('panel')[metrics].mean()
        row = dict(candidate=name, alpha=float(part.alpha.iloc[0]),
                   **{m: macro_mean(per_panel[m].to_dict()) for m in metrics})
        if 'FD_proxy' in row:
            row['selection_proxy'] = sum(row[k] * w for k, w in SELECT_WEIGHTS.items()) / sum(SELECT_WEIGHTS.values())
        rows.append(row)
    return pd.DataFrame(rows), sums, daily

def training_transitions(d, context):
    if '_transition' not in d:
        _, counts, affected = completed_link_fields(d['truth'][d['ti'], d['si']] * SCALE, d, context)
        transition, target_transition = observability_mask(context, counts, d['ramp_valid'], affected, d['consecutive'])
        d['_transition'] = transition & target_transition
    return d['_transition']


def joint_loss(output, d, starts, context, ct, physics_factor, return_components=False):
    device, length = output.device, output.shape[1]
    take = lambda key: torch.as_tensor(np.stack([d[key][s:s+length] for s in starts]), device=device)
    target, truth = take('target').bool(), take('truth').float()
    if not target.any():
        raise ValueError('학습 창에 대상 셀이 없습니다.')
    state_error = (output.float() - truth)[target]
    state = 0.54 * state_error[:, 0].square().mean() + 0.46 * state_error[:, 1].square().mean()
    zero = state * 0.
    losses = dict(State=state, N=zero, dN=zero, FD=zero)
    if physics_factor > 0:
        active, observed = take('active').float(), take('observed').float()
        units = torch.stack((torch.full_like(ct['lanes'], 25.), 600 * ct['lanes']), -1)
        predicted = torch.where(target[..., None], output.float() * units, observed)
        reference = torch.where(target[..., None], truth * units, observed)
        counts = torch.einsum('bts,sl->btl', active, ct['matrix'])
        aggregate = lambda values: torch.einsum('btsc,sl->btlc', values * active[..., None], ct['matrix']) / counts.clamp_min(1)[..., None]
        links, true_links = aggregate(predicted), aggregate(reference)
        affected = torch.einsum('bts,sl->btl', target.float(), ct['matrix']) > 0
        good_n = (counts > 0) & affected
        n = links[..., 1] / links[..., 0].clamp_min(1) * ct['lengths']
        true_n = true_links[..., 1] / true_links[..., 0].clamp_min(1) * ct['lengths']
        losses['N'] = (n - true_n)[good_n].abs().sum() / (true_n[good_n].abs().sum() + 1.)
        transition = training_transitions(d, context)
        good_dn = torch.as_tensor(np.stack([transition[s:s+length-1] for s in starts]), device=device)
        dn, true_dn = torch.diff(n, dim=1), torch.diff(true_n, dim=1)
        dn_denominator = true_dn[good_dn].abs().sum() + 1.
        losses['dN'] = (dn - true_dn)[good_dn].abs().sum() / dn_denominator
        if LOSS_WEIGHTS['FD'] > 0:
            density = links[..., 1] / links[..., 0].clamp_min(1)
            critical = ct['capacity'] / ct['free']
            wave = ct['capacity'] / (ct['jam'] - critical).clamp_min(1e-6)
            qfd = torch.where(density <= critical, ct['free'] * density, wave * (ct['jam'] - density)).clamp_min(0)
            error = ((links[..., 1] - qfd) / ct['link_lanes'])[good_n]
            losses['FD'] = error.abs().sum() / ((links[..., 1] / ct['link_lanes'])[good_n].detach().abs().sum() + 1.)
    total = state + physics_factor * sum(LOSS_WEIGHTS[k] * losses[k] for k in LOSS_WEIGHTS)
    if not torch.isfinite(total):
        raise FloatingPointError('학습 손실에 NaN/Inf가 있습니다.')
    return total, losses if return_components else {k: float(v.detach()) for k, v in losses.items()}





def error_slice_axes(d, context):
    ti, si = d['ti'], d['si']
    ages = d['aux'][ti, si, 4:6]
    gap = np.rint(ages.sum(-1) * 360)
    gap_bucket = np.select([np.any(ages >= 4., axis=-1), gap <= 10, gap <= 30, gap <= 60],
                           ['one_sided', '0-10', '10-30', '30-60'], default='60+')
    neighbors = context['station_graph'].astype(np.int32).copy()
    np.fill_diagonal(neighbors, 0)
    counts = (d['visible'][..., :2].all(-1).astype(np.int32) @ neighbors.T)[ti, si]
    neighbor_bucket = np.select([counts == 0, counts <= 2, counts <= 5], ['0', '1-2', '3-5'], default='6+')
    return {'hour_utc': d['slot'][ti] // 12, 'speed_gap_minutes': gap_bucket,
            'station': np.asarray(context['targets'].stations)[si], 'observed_neighbors': neighbor_bucket}


def error_slice_records(truth, prediction, axes, metadata):
    rows = []
    for dimension, buckets in axes.items():
        for bucket in np.unique(buckets):
            selected = buckets == bucket
            rows.append(dict(metadata, dimension=dimension, bucket=str(bucket),
                             **state_totals(truth[selected], prediction[selected])))
    return rows


def summarize_error_slices(records):
    keys = ['candidate', 'alpha', 'panel', 'regime', 'dimension', 'bucket']
    frame = pd.DataFrame(records).groupby(keys, as_index=False).sum(numeric_only=True)
    frame['speed_rmse_kmh'] = np.sqrt(frame.speed_sse / frame['count'])
    frame['flow_rmse_per_lane'] = np.sqrt(frame.flow_sse / frame['count'])
    frame['speed_mae_kmh'] = frame.speed_sae / frame['count']
    frame['flow_mae_per_lane'] = frame.flow_sae / frame['count']
    frame['slice_State'] = (0.54 * np.maximum(0, 1 - frame.speed_rmse_kmh / 25)
                            + 0.46 * np.maximum(0, 1 - frame.flow_rmse_per_lane / 600))
    return frame


def final_state_output(d, neural, alpha, processor=None):
    """Apply optional correction after the frozen interpolation/model blend."""
    mixed = (1 - alpha) * d['base'] + alpha * neural
    if processor is None:
        return mixed
    baseline = mixed.copy()
    corrected = np.asarray(processor(d, mixed))
    if (corrected.shape != baseline.shape or not np.isfinite(corrected).all() or (corrected < 0).any()
            or not np.array_equal(corrected[~d['target']], baseline[~d['target']])):
        raise ValueError('최종 보정은 유한한 비음수 예측이며 대상 셀만 변경해야 합니다.')
    return corrected


def evaluate(entries, contexts, tensors, model, period, alphas, folder, collect_diagnostics=False,
             postprocessors=None, finalprocessors=None):
    started, records, slice_records = time.perf_counter(), [], []
    selected = [e for e in entries if e['period'] == period]
    if not selected:
        raise ValueError(f'평가 날짜가 없습니다: {period}')
    for number, entry in enumerate(selected, 1):
        d, context = load_day(entry['path']), contexts[entry['panel']]
        output = maginet_predict(model, d, tensors[entry['panel']], batch_size=panel_batch_size(context))
        if postprocessors:
            output = postprocessors[entry['panel']](d, output)
        ti, si = d['ti'], d['si']
        truth, base, prediction = d['truth'][ti, si] * SCALE, d['base'][ti, si] * SCALE, output[ti, si] * SCALE
        axes = error_slice_axes(d, context) if collect_diagnostics else None
        for alpha in alphas:
            if finalprocessors:
                final = final_state_output(d, output, alpha, finalprocessors[entry['panel']])
                mixed = final[ti, si] * SCALE
            else:
                mixed = (1 - alpha) * base + alpha * prediction
            records.append(dict(candidate=f'alpha_{float(alpha)!r}', alpha=float(alpha),
                                panel=entry['panel'], date=entry['date'],
                                regime=entry['regime'], **state_totals(truth, mixed), **physics_totals(mixed, d, context)))
            if collect_diagnostics:
                slice_records.extend(error_slice_records(truth, mixed, axes,
                    {key: records[-1][key] for key in ('candidate', 'alpha', 'panel', 'date', 'regime')}))
        if number == 1 or number % 20 == 0 or number == len(selected):
            print(f'평가 {period}: {number}/{len(selected)}일', flush=True)
    macro, grouped, daily = summarize(records)
    folder.mkdir(parents=True, exist_ok=True)
    for name, frame in (('macro', macro), ('panel_regime', grouped), ('daily', daily)):
        frame.to_csv(folder / f'{period}_{name}.csv', index=False)
    if collect_diagnostics:
        summarize_error_slices(slice_records).to_csv(folder / f'{period}_error_slices.csv', index=False)
    if (grouped.lwr_count == 0).any():
        print('주의: LWR 대리 평가 대상이 없는 그룹이 있습니다. panel_regime의 lwr_count를 확인해 주세요.')
    return macro, grouped, daily, time.perf_counter() - started


def notebook_signature(contexts, entries, normalization):
    settings = dict(architecture='maginet-task13-residual-v2',
                    input_normalization={k: normalization[k] for k in ('mode', 'mean', 'std')},
                    output_scale=SCALE.tolist(), aux_names=list(AUX_NAMES), seed=SEED, window=WINDOW, stride=STRIDE,
                    width=WIDTH, blocks=BLOCKS, heads=HEADS, cheb_k=CHEB_K, graph_dim=GRAPH_DIM,
                    dropout=DROPOUT, mgl=MGL_THRESHOLD, batch=BATCH_SIZE, updates=UPDATES_PER_GROUP,
                    lr=LR, weight_decay=WEIGHT_DECAY, patience=PATIENCE,
                    warmup=WARMUP_EPOCHS, ramp=PHYSICS_RAMP_EPOCHS, loss_weights=LOSS_WEIGHTS,
                    select_weights=SELECT_WEIGHTS, periods=PERIODS, release=str(RELEASE_ROOT.resolve()),
                    training_dates={n: sorted(e['date'] for e in entries if e['panel'] == n and e['period'] == 'fit')
                                    for n in contexts},
                    panels={name: dict(stations=c['targets'].stations, links=c['targets'].links,
                                      station_codes=c['station_codes'].tolist(), panel_code=c['panel_code'],
                                      family=c['family_code'], direction=c['direction_code'], mode=c['mode'],
                                      dates=c['dates'], profile_hash=hashlib.sha256(c['profile'].tobytes()).hexdigest())
                            for name, c in contexts.items()})
    settings['improvements'] = dict(temporal_direct=TEMPORAL_DIRECT, learned_graph=LEARNED_GRAPH,
                                    mask_resample=None)
    settings['improvements']['spatial_mode'] = SPATIAL_MODE
    if CHECKPOINT_METRIC != 'selection_proxy':
        settings['checkpoint_metric'] = CHECKPOINT_METRIC
    return hashlib.sha256(json.dumps(settings, sort_keys=True).encode()).hexdigest(), settings


def run_signature(contexts, entries, normalization):
    _, settings = notebook_signature(contexts, entries, normalization)
    settings.pop('release')
    settings['format_version'] = 3
    settings['days_per_regime'] = DAYS_PER_REGIME
    # 기본 fixed 설정은 이전 로컬 체크포인트의 서명을 유지합니다.
    if BATCH_POLICY != 'fixed' or PANEL_BATCH:
        settings.update(batch_policy=BATCH_POLICY, panel_batch=dict(PANEL_BATCH),
                        panel_batches={n: panel_batch_size(c) for n, c in contexts.items()})
    inputs = []
    for entry in sorted(entries, key=lambda e: (e['panel'], e['period'], e['date'])):
        with zipfile.ZipFile(entry['path']) as archive:
            arrays = sorted((m.filename, m.file_size, m.CRC) for m in archive.infolist())
        inputs.append((entry['panel'], entry['period'], entry['date'], arrays))
    settings['input_hash'] = hashlib.sha256(json.dumps(inputs).encode()).hexdigest()
    settings['network_hashes'] = network_hashes(contexts)
    return hashlib.sha256(json.dumps(settings, sort_keys=True).encode()).hexdigest(), settings


def network_hashes(contexts):
    hashes = {}
    for name, context in contexts.items():
        digest = hashlib.sha256()
        for values in (context['station_graph'], context['mapping'], context['lanes'],
                       context['panel'].free_speed, context['panel'].lanes, context['panel'].lengths,
                       context['panel'].capacity, context['panel'].jam_density, context['panel'].edges):
            array = np.ascontiguousarray(values)
            digest.update(str((array.shape, array.dtype)).encode())
            digest.update(array.tobytes())
        digest.update(json.dumps(context['panel'].ramp_map, sort_keys=True).encode())
        hashes[name] = digest.hexdigest()
    return hashes


def save_checkpoint(payload, path):
    temporary = path.with_suffix('.pt.tmp')
    torch.save(payload, temporary)
    temporary.replace(path)


def optimizer_step(model, optimizer, scaler):
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1., error_if_nonfinite=not scaler.is_enabled())
    scaler.step(optimizer)
    scaler.update()




def train_maginet(entries, contexts, tensors, folder, epochs=None, resume=None):
    epochs = EPOCHS if epochs is None else epochs
    resume = RESUME if resume is None else resume
    reset_seed()
    folder.mkdir(parents=True, exist_ok=True)
    best_path, last_path = folder / 'best_maginet.pt', folder / 'last_maginet.pt'
    if not resume and any(p.exists() for p in (best_path, last_path, folder / 'best_state_maginet.pt')):
        raise FileExistsError(f'기존 학습을 보호합니다. RESUME=True 또는 새 REPORT_DIR를 사용해 주세요: {folder}')
    normalization = fit_input_normalization(entries, CACHE_METADATA_DIR / 'normalization', legacy_folder=folder)
    signature, settings = run_signature(contexts, entries, normalization)
    station_count = max(int(c['station_codes'].max()) for c in contexts.values()) + 1
    panel_count = max(c['panel_code'] for c in contexts.values()) + 1
    family_count = max(c['family_code'] for c in contexts.values()) + 1
    model = MagiNetTask13(station_count, panel_count, family_count, normalization).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP and AMP_DTYPE == torch.float16)
    best_path, last_path = folder / 'best_maginet.pt', folder / 'last_maginet.pt'
    start_epoch, best, bad_epochs, best_epoch, history = 1, -float('inf'), 0, 0, []
    phase_best = -float('inf')
    if resume:
        if not last_path.exists() or not best_path.exists():
            raise FileNotFoundError('재개하려면 last_maginet.pt와 best_maginet.pt가 모두 필요합니다.')
        state = torch.load(last_path, map_location=DEVICE, weights_only=True)
        if state['signature'] != signature:
            raise ValueError('설정·패널·학습 날짜가 달라 재개할 수 없습니다. REPORT_DIR를 바꾸어 새로 학습해 주세요.')
        model.load_state_dict(state['model'])
        optimizer.load_state_dict(state['optimizer'])
        scheduler.load_state_dict(state['scheduler'])
        scaler.load_state_dict(state['scaler'])
        start_epoch, best, bad_epochs, best_epoch, history = state['epoch'] + 1, state['best'], state['bad_epochs'], state['best_epoch'], state['history']
        phase_best = state.get('phase_best', -float('inf'))
        torch.set_rng_state(state['rng_cpu'].cpu())
        if DEVICE.type == 'cuda' and state['rng_cuda']:
            torch.cuda.set_rng_state_all([v.cpu() for v in state['rng_cuda']])
        if state.get('physics_schedule') != 'after-ramp-v1':
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)
            bad_epochs = 0
            if start_epoch <= WARMUP_EPOCHS + max(PHYSICS_RAMP_EPOCHS, 1):
                for group in optimizer.param_groups:
                    group['lr'] = LR
        if epochs < start_epoch:
            raise ValueError(f'EPOCHS를 마지막 완료 epoch {start_epoch-1}보다 크게 지정해 주세요.')
    elif best_path.exists() or last_path.exists():
        raise FileExistsError(f'기존 학습을 보호합니다. RESUME=True 또는 새 REPORT_DIR를 사용해 주세요: {folder}')
    fit_entries = [e for e in entries if e['period'] == 'fit']
    pools = {(name, r): [e for e in fit_entries if e['panel'] == name and e['regime'] == r]
             for name in contexts for r in range(3)}
    if any(not p for p in pools.values()):
        raise ValueError('공동 학습에 panel×regime별 날짜가 모두 필요합니다.')
    metadata = checkpoint_metadata(contexts, normalization, settings)
    metadata['physics_schedule'] = 'after-ramp-v1'
    # 재개 설정과 학습 그룹을 확인한 뒤 결과·캐시를 각각 저장합니다.
    (folder / 'config.json').write_text(json.dumps(settings, indent=2, ensure_ascii=False))
    (folder / 'input_normalization.json').write_text(json.dumps(normalization, indent=2, ensure_ascii=False))
    manifest_key = hashlib.sha256(json.dumps(entries, sort_keys=True).encode()).hexdigest()[:16]
    manifest_dir = CACHE_METADATA_DIR / 'manifests'
    manifest_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = manifest_dir / f'cache_manifest-{manifest_key}.csv'
    temporary = manifest_path.with_suffix('.csv.tmp')
    pd.DataFrame(entries).to_csv(temporary, index=False)
    temporary.replace(manifest_path)
    direction_count = {n: sum(p.rsplit('_', 1)[0] == n.rsplit('_', 1)[0] for p in contexts) for n in contexts}
    panel_batches = {n: panel_batch_size(c) for n, c in contexts.items()}
    print({'panel_batches': panel_batches}, flush=True)
    began = time.perf_counter()
    prior_elapsed = history[-1]['elapsed_seconds'] if history else 0.
    for epoch in range(start_epoch, epochs + 1):
        model.train()
        rng = np.random.default_rng(SEED + epoch)
        groups = list(pools)
        rng.shuffle(groups)
        totals = dict(State=0., N=0., dN=0., FD=0.)
        updates, train_windows, train_started = 0, 0, time.perf_counter()
        physics_factor = min(1., max(0., (epoch - WARMUP_EPOCHS) / max(PHYSICS_RAMP_EPOCHS, 1)))
        for name, regime in groups:
            pool = pools[(name, regime)]
            probabilities = np.array([e['targets'] for e in pool], np.float64)
            probabilities /= probabilities.sum()
            for step in range(UPDATES_PER_GROUP):
                if step % 4 == 0:
                    entry = pool[int(rng.choice(len(pool), p=probabilities))]
                    d = load_day(entry['path'])
                    starts = [s for s in window_starts(len(d['values'])) if d['target'][s:s+WINDOW].any()]
                    if not starts:
                        raise ValueError('대상 셀이 있는 학습 창이 없습니다.')
                chunk = rng.choice(starts, size=panel_batches[name], replace=True).tolist()
                batch = window_batch(d, chunk)
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
                    output = model(**batch, ct=tensors[name])
                loss, terms = joint_loss(output.float(), d, chunk, contexts[name], tensors[name], physics_factor)
                # 도로군마다 방향 수가 달라도 누적 가중치가 같도록 맞춥니다.
                scaler.scale(loss / direction_count[name]).backward()
                optimizer_step(model, optimizer, scaler)
                updates += 1
                train_windows += len(chunk)
                for k in totals:
                    totals[k] += terms[k]
            print(f'epoch {epoch}/{epochs}: {name}/{REGIMES[regime]} 학습 완료', flush=True)
        train_seconds = time.perf_counter() - train_started
        macro, _, _, eval_seconds = evaluate(entries, contexts, tensors, model, 'early', (1.,), folder / 'early')
        proxy = float(macro.iloc[0].selection_proxy)
        score = float(macro.iloc[0][CHECKPOINT_METRIC])
        if not np.isfinite(score):
            raise FloatingPointError('체크포인트 선택 점수가 유한하지 않습니다.')
        # 물리 손실의 가중치가 고정된 뒤부터 정체를 판단합니다.
        if physics_factor >= 1.:
            if score > phase_best + 1e-7:
                phase_best, bad_epochs = score, 0
            else:
                bad_epochs += 1
            scheduler.step(score)
        else:
            bad_epochs = 0
        if score > best + 1e-7:
            best, best_epoch = score, epoch
            payload = dict(model=model.state_dict(), signature=signature, epoch=epoch, score=score,
                           checkpoint_metric=CHECKPOINT_METRIC, **metadata)
            save_checkpoint(payload, best_path)
        row = dict(epoch=epoch, **{f'loss_{k}': v / updates for k, v in totals.items()},
                   optimizer_updates=updates, train_windows=train_windows,
                   early_State=float(macro.iloc[0].State), early_selection_proxy=proxy,
                   early_checkpoint_score=score,
                   physics_factor=physics_factor, lr=optimizer.param_groups[0]['lr'],
                   train_seconds=train_seconds, eval_seconds=eval_seconds,
                   elapsed_seconds=prior_elapsed + time.perf_counter() - began)
        history.append(row)
        pd.DataFrame(history).to_csv(folder / 'history.csv', index=False)
        save_checkpoint(dict(model=model.state_dict(), optimizer=optimizer.state_dict(), scheduler=scheduler.state_dict(),
                        scaler=scaler.state_dict(), signature=signature, epoch=epoch, best=best,
                        bad_epochs=bad_epochs, best_epoch=best_epoch, history=history,
                        phase_best=phase_best,
                        rng_cpu=torch.get_rng_state(), rng_cuda=torch.cuda.get_rng_state_all() if DEVICE.type == 'cuda' else [],
                        **metadata), last_path)
        print({'epoch': epoch, 'State': row['early_State'], 'selection_proxy': proxy,
               'checkpoint_metric': CHECKPOINT_METRIC, 'checkpoint_score': score,
               'best_epoch': best_epoch, 'physics_factor': physics_factor}, flush=True)
        if bad_epochs >= PATIENCE and physics_factor >= 1.:
            break
    best_state = torch.load(best_path, map_location=DEVICE, weights_only=True)
    if best_state['signature'] != signature:
        raise ValueError('저장된 best 체크포인트의 설정이 다릅니다.')
    model.load_state_dict(best_state['model'])
    model.eval()
    print({'best_epoch': best_state['epoch'], 'checkpoint_metric': CHECKPOINT_METRIC,
           'best_early_score': best_state['score']})
    return model, pd.DataFrame(history)

def predict_split(contexts, model, tensors, alpha, split, folder, postprocessors=None, finalprocessors=None):
    if split not in ('validation', 'private') or not 0 <= alpha <= 1:
        raise ValueError('추론 split 또는 alpha가 잘못되었습니다.')
    folder.mkdir(parents=True, exist_ok=True)
    paths = []
    for name, c in contexts.items():
        template_path = c['release'] / 'task1' / name / split / 'sample_submission_state.csv'
        template = pd.read_csv(template_path, dtype={k: str for k in KEYS})
        if template.empty or template.duplicated(KEYS).any() or set(template.panel) != {name}:
            raise ValueError(f'잘못된 제출 템플릿: {template_path}')
        files = dated_files(c['release'] / 'corridors' / name / split / 'mainline_states_masked')
        if not files:
            raise FileNotFoundError(f'공개 masked 입력이 없습니다: {name}/{split}')
        first = StationDay.load(next(iter(files.values())), c['panel'])
        target_index = TargetIndex(c['release'], name, first, split)
        if (first.station_ids, first.station_links) != (c['targets'].stations, c['targets'].links):
            raise ValueError(f'추론 관측소 구성이 다릅니다: {name}')
        out = template.copy()
        out[['speed_kmh', 'flow_vph']] = np.nan
        for number, (date, group) in enumerate(template.groupby(template.timestamp.str[:10], sort=False), 1):
            if date not in files:
                raise ValueError(f'추론 날짜가 누락되었습니다: {name}/{date}')
            d = make_day(c, date, split=split, target_index=target_index, input_files=files)
            ti, si = d['ti'], d['si']
            if alpha > 0:
                output = maginet_predict(model, d, tensors[name], batch_size=panel_batch_size(c))
                neural = postprocessors[name](d, output) if postprocessors else output
            else:
                neural = d['base']
            final = final_state_output(d, neural, alpha, finalprocessors[name] if finalprocessors else None)
            values = final[ti, si] * SCALE
            local = pd.DataFrame(dict(timestamp=d['timestamps'][ti], station_id=np.asarray(c['targets'].stations)[si],
                                      speed_kmh=values[:, 0], flow_vph=values[:, 1] * c['lanes'][si]))
            aligned = group[['timestamp', 'station_id']].merge(local, on=['timestamp', 'station_id'],
                                                               how='left', validate='one_to_one', sort=False)
            if aligned[['speed_kmh', 'flow_vph']].isna().any().any():
                raise ValueError(f'추론 대상 키가 일치하지 않습니다: {name}/{date}')
            out.loc[group.index, ['speed_kmh', 'flow_vph']] = aligned[['speed_kmh', 'flow_vph']].to_numpy()
            if number == 1 or number % 10 == 0:
                print(f'CSV 예측 {name}/{split}: {number}일', flush=True)
        values = out[['speed_kmh', 'flow_vph']].to_numpy()
        if not out[KEYS].equals(template[KEYS]) or not np.isfinite(values).all() or (values < 0).any():
            raise ValueError(f'출력 값 또는 템플릿 키가 잘못되었습니다: {name}')
        path = folder / f'task1-{name}-{split}.csv'
        temporary = path.with_suffix('.csv.tmp')
        out.to_csv(temporary, index=False)
        temporary.replace(path)
        paths.append(str(path))
    return paths


def checkpoint_metadata(contexts, normalization, settings):
    """Store inference inputs as plain containers and CPU tensors (weights_only safe)."""
    packed = {}
    for name, context in contexts.items():
        panel = context['panel']
        network = {}
        for key in PanelInfo.__dataclass_fields__:
            value = getattr(panel, key)
            network[key] = torch.from_numpy(value.copy()) if isinstance(value, np.ndarray) else value
        packed[name] = dict(network=network, stations=context['targets'].stations,
                            station_links=context['targets'].links, dates=context['dates'], mode=context['mode'],
                            panel_code=int(context['panel_code']), family_code=int(context['family_code']),
                            direction_code=int(context['direction_code']), topology_nodes=sorted(context['topology_nodes']),
                            cache_key=context.get('cache_key'))
        for key in ('mapping', 'lanes', 'station_codes', 'station_graph', 'profile'):
            packed[name][key] = torch.from_numpy(np.asarray(context[key]).copy())
    return dict(format_version=3, settings=settings, normalization=normalization, contexts=packed,
                experiment=EXPERIMENT, runtime=dict(torch=str(torch.__version__), device=str(DEVICE),
                                                   precision=str(AMP_DTYPE) if USE_AMP else 'float32',
                                                   cpu_threads=torch.get_num_threads()))


def unpack_contexts(packed, release):
    contexts = {}
    for name, saved in packed.items():
        network = {k: v.cpu().numpy().copy() if isinstance(v, torch.Tensor) else v
                   for k, v in saved['network'].items()}
        context = {k: saved[k] for k in ('dates', 'mode', 'panel_code', 'family_code', 'direction_code')}
        context.update(release=release, panel=PanelInfo(**network), topology_nodes=set(saved['topology_nodes']),
                       targets=SimpleNamespace(stations=saved['stations'], links=saved['station_links']),
                       cache_key=saved.get('cache_key'))
        for key in ('mapping', 'lanes', 'station_codes', 'station_graph', 'profile'):
            context[key] = saved[key].cpu().numpy().copy()
        contexts[name] = context
    return contexts


SETTING_NAMES = dict(input_normalization='INPUT_NORMALIZATION', seed='SEED', window='WINDOW', stride='STRIDE',
                     width='WIDTH', blocks='BLOCKS', heads='HEADS', cheb_k='CHEB_K', graph_dim='GRAPH_DIM',
                     dropout='DROPOUT', mgl='MGL_THRESHOLD', batch='BATCH_SIZE', updates='UPDATES_PER_GROUP',
                     lr='LR', weight_decay='WEIGHT_DECAY', patience='PATIENCE', warmup='WARMUP_EPOCHS',
                     ramp='PHYSICS_RAMP_EPOCHS', loss_weights='LOSS_WEIGHTS', select_weights='SELECT_WEIGHTS',
                     periods='PERIODS', days_per_regime='DAYS_PER_REGIME',
                     batch_policy='BATCH_POLICY', panel_batch='PANEL_BATCH', checkpoint_metric='CHECKPOINT_METRIC')
DEFAULT_CONFIG = {key: globals()[key] for key in SETTING_NAMES.values()}


def restore_settings(settings):
    if settings['architecture'] != 'maginet-task13-residual-v2':
        raise ValueError('지원하지 않는 MagiNet 구조입니다.')
    globals().update(BATCH_POLICY='fixed', PANEL_BATCH={}, CHECKPOINT_METRIC='selection_proxy')
    for saved_key, variable in SETTING_NAMES.items():
        if saved_key in settings:
            value = settings[saved_key]
            globals()[variable] = value['mode'] if saved_key == 'input_normalization' else value
    improvements = settings.get('improvements', {})
    if (not improvements.get('temporal_direct') or not improvements.get('learned_graph')
            or improvements.get('mask_resample') or improvements.get('spatial_mode') != 'channel'):
        raise ValueError('현재 channel_graph 구조의 MagiNet 체크포인트만 지원합니다.')
    if CHECKPOINT_METRIC not in ('selection_proxy', 'State'):
        raise ValueError('저장된 체크포인트 선택 기준이 잘못되었습니다.')


def load_local_checkpoint(path, release, saved=None):
    saved = torch.load(path, map_location='cpu', weights_only=True) if saved is None else saved
    if saved.get('format_version') != 3:
        raise ValueError('Colab 체크포인트는 먼저 import-colab 명령으로 가져와 주세요.')
    signature = hashlib.sha256(json.dumps(saved['settings'], sort_keys=True).encode()).hexdigest()
    if signature != saved['signature']:
        raise ValueError('체크포인트 설정 서명이 일치하지 않습니다.')
    restore_settings(saved['settings'])
    contexts = unpack_contexts(saved['contexts'], release)
    model = MagiNetTask13(max(int(c['station_codes'].max()) for c in contexts.values()) + 1,
                         max(c['panel_code'] for c in contexts.values()) + 1,
                         max(c['family_code'] for c in contexts.values()) + 1, saved['normalization']).to(DEVICE)
    model.load_state_dict(saved['model'])
    return model.eval(), contexts, saved


def file_digest(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def resolve_release(path=None):
    if path is None:
        pointer = PROJECT_ROOT / 'data' / 'release_root.txt'
        path = pointer.read_text(encoding='utf-8-sig').strip() if pointer.is_file() else RELEASE_ROOT
    path = Path(path).expanduser()
    if not path.is_absolute():
        path = PROJECT_ROOT / path
    path = path.resolve()
    if not (path / 'corridors').is_dir() or not (path / 'task1').is_dir():
        raise FileNotFoundError(f'corridors/와 task1/이 있는 데이터 경로를 지정해 주세요: {path}')
    return path


def checkpoint_folder(path):
    return path.parent.parent if path.parent.name == 'full' else path.parent


def configure(args):
    globals().update(DEFAULT_CONFIG)
    globals()['DATE_OVERRIDES'] = {}
    experiment = EXPERIMENT
    report_dir = (args.output_dir or PROJECT_ROOT / 'reports' / 'maginet' / experiment).resolve()
    checkpoint = (args.checkpoint or report_dir / 'full' / 'best_maginet.pt').resolve()
    source = None
    if args.command in ('evaluate', 'predict') or args.command == 'train' and args.resume:
        source_path = report_dir / 'full' / 'last_maginet.pt' if args.command == 'train' else checkpoint
        source = torch.load(source_path, map_location='cpu', weights_only=True)
        if source.get('format_version') != 3:
            raise ValueError('Colab 체크포인트는 먼저 import-colab 명령으로 가져와 주세요.')
        if args.command == 'train':
            restore_settings(source['settings'])
        experiment = source['experiment']
        if args.output_dir is None and args.command in ('evaluate', 'predict'):
            report_dir = checkpoint_folder(checkpoint)
        if args.command == 'train':
            globals()['DATE_OVERRIDES'] = {n: c['dates'] for n, c in source['settings']['panels'].items()}
    elif args.command == 'import-colab':
        source_folder = args.source_run / 'full' if (args.source_run / 'full' / 'config.json').is_file() else args.source_run
        old_settings = json.loads((source_folder / 'config.json').read_text(encoding='utf-8'))
        restore_settings(old_settings)
        globals()['DATE_OVERRIDES'] = {n: c['dates'] for n, c in old_settings['panels'].items()}
        if args.output_dir is None:
            report_dir = PROJECT_ROOT / 'reports' / 'maginet' / experiment
    else:
        globals().update(LOSS_WEIGHTS=DEFAULT_CONFIG['LOSS_WEIGHTS'].copy(), CHECKPOINT_METRIC='State')
    if args.command == 'train' and any(not np.isclose(LOSS_WEIGHTS.get(key, np.nan), value, rtol=0, atol=1e-8)
                                       for key, value in DEFAULT_CONFIG['LOSS_WEIGHTS'].items()):
        raise ValueError('학습·재개는 current003(N=0.05, dN=0.03, FD=0)만 지원합니다. 기존 다른 모델은 예측에 사용할 수 있습니다.')
    if args.checkpoint_metric is not None:
        globals()['CHECKPOINT_METRIC'] = 'State' if args.checkpoint_metric == 'state' else 'selection_proxy'
    for argument, variable in (('input_normalization', 'INPUT_NORMALIZATION'), ('seed', 'SEED'),
                               ('batch_size', 'BATCH_SIZE'), ('updates_per_group', 'UPDATES_PER_GROUP'),
                               ('epochs', 'EPOCHS'), ('patience', 'PATIENCE'), ('lr', 'LR'),
                               ('days_per_regime', 'DAYS_PER_REGIME'), ('width', 'WIDTH'),
                               ('blocks', 'BLOCKS'), ('heads', 'HEADS'), ('batch_policy', 'BATCH_POLICY')):
        value = getattr(args, argument)
        if value is not None:
            globals()[variable] = value
    if args.panel_batch is not None:
        overrides = {}
        for item in args.panel_batch:
            try:
                name, size = item.split('=', 1)
                size = int(size)
            except ValueError as error:
                raise ValueError('--panel-batch는 PANEL=양의정수 형식이어야 합니다.') from error
            if not name or size < 1 or name in overrides:
                raise ValueError('--panel-batch의 패널 중복·이름·배치 크기를 확인해 주세요.')
            overrides[name] = size
        globals()['PANEL_BATCH'] = overrides
    if BATCH_POLICY == 'adaptive' and args.batch_size is not None:
        raise ValueError('adaptive에서는 --batch-size 대신 --panel-batch로 배치를 지정해 주세요.')
    if (min(BATCH_SIZE, UPDATES_PER_GROUP, EPOCHS, WIDTH, BLOCKS, HEADS) < 1 or WIDTH % HEADS
            or WIDTH % 2 or not np.isfinite(LR) or LR <= 0 or PATIENCE < 1 or args.cpu_threads < 1
            or DAYS_PER_REGIME is not None and DAYS_PER_REGIME < 1):
        raise ValueError('모델 크기·학습량·학습률·날짜 수 설정을 확인해 주세요.')
    device = args.device
    if device == 'auto':
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
    if device == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('CUDA를 사용할 수 없습니다. --device cpu로 실행할 수 있습니다.')
    if args.precision == 'bf16' and (device != 'cuda' or not torch.cuda.is_bf16_supported()):
        raise ValueError('이 장치에서는 BF16을 사용할 수 없습니다.')
    amp = device == 'cuda' and args.precision != 'fp32'
    dtype = torch.bfloat16 if device == 'cuda' and torch.cuda.is_bf16_supported() else torch.float16
    cache_dir = (args.cache_dir or PROJECT_ROOT / 'data' / 'maginet_cache').resolve()
    selected = args.panel
    if source and args.command == 'train':
        selected = selected or sorted(source['contexts'])
    elif args.command == 'import-colab':
        selected = selected or sorted(old_settings['panels'])
    globals().update(EXPERIMENT=experiment, RELEASE_ROOT=resolve_release(args.release_root), REPORT_DIR=report_dir,
                     REPORT_BASE=report_dir.parent, CACHE_STORE_DIR=cache_dir, CACHE_DIR=cache_dir,
                     CACHE_ROOT=cache_dir, CACHE_CONTAINER=cache_dir, CACHE_METADATA_DIR=cache_dir / 'metadata',
                     CACHE_MODE=args.cache_mode, SAVED_CACHE_ZIP=args.cache_zip,
                     CACHE_PROFILE_DIR=args.legacy_profile_dir or (source_folder if args.command == 'import-colab'
                                                                  else cache_dir / 'legacy_profiles'),
                     PANELS=selected, DEVICE=torch.device(device), USE_AMP=amp, AMP_DTYPE=dtype,
                     RESUME=args.resume)
    torch.set_num_threads(args.cpu_threads)
    reset_seed()
    load_day.cache_clear()
    args.checkpoint = checkpoint
    args.saved_checkpoint = source
    if (args.select_retrieval or args.retrieval_selection is not None) and args.output_dir is None:
        globals()['REPORT_DIR'] = REPORT_DIR / 'retrieval'
    print(dict(command=args.command, experiment=experiment, device=device,
               precision=str(dtype) if amp else 'float32', release=str(RELEASE_ROOT),
               cache=str(cache_dir), reports=str(REPORT_DIR)), flush=True)


def prepare_inputs():
    names = available_panels(RELEASE_ROOT)
    unknown = set(PANEL_BATCH) - set(names)
    if unknown:
        raise ValueError(f'학습에 포함되지 않은 --panel-batch 패널입니다: {sorted(unknown)}')
    root, container = (restore_input_cache(names, SAVED_CACHE_ZIP, CACHE_DIR / 'imported') if SAVED_CACHE_ZIP
                       else (CACHE_DIR, CACHE_DIR))
    globals().update(CACHE_ROOT=root, CACHE_CONTAINER=container)
    contexts, entries, offset = {}, [], 0
    for code, name in enumerate(names):
        context = prepare_panel(RELEASE_ROOT, name, code, offset)
        offset += len(context['targets'].stations)
        contexts[name] = context
    attach_vocabulary(contexts)
    identity = dict(format='maginet-local-input-v1', networks=network_hashes(contexts),
                    panels={n: dict(stations=c['targets'].stations, links=c['targets'].links,
                                    fit_dates=c['dates']['fit'],
                                    profile_hash=hashlib.sha256(c['profile'].tobytes()).hexdigest())
                            for n, c in contexts.items()})
    key = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:20]
    destination = CACHE_DIR / 'inputs' / key
    for name, context in contexts.items():
        context['cache_key'] = key
        for period in ('fit', 'early', 'blend'):
            for date in context['dates'][period]:
                source = root / name / period / f'{date}.npz'
                target = destination / name / period / f'{date}.npz'
                if source.is_file() and not target.exists():
                    target.parent.mkdir(parents=True, exist_ok=True)
                    temporary = target.with_suffix('.npz.tmp')
                    shutil.copyfile(source, temporary)
                    temporary.replace(target)
        entries.extend(cache_context(context, destination))
    globals()['CACHE_ROOT'] = destination
    return contexts, entries


def evaluation_inputs(contexts, period):
    entries = []
    for name, context in contexts.items():
        folder = RELEASE_ROOT / 'corridors' / name / 'train'
        context['files'] = dated_files(folder / 'mainline_states')
        context['masked'] = dated_files(folder / 'mainline_states_masked')
        dates = context['dates'][period]
        if not dates:
            raise ValueError(f'체크포인트의 {name}/{period} 날짜가 없습니다.')
        first = StationDay.load(context['files'][dates[0]], context['panel'])
        target_index = TargetIndex(RELEASE_ROOT, name, first)
        if (target_index.stations, target_index.links) != (context['targets'].stations, context['targets'].links):
            raise ValueError(f'체크포인트의 관측소와 로컬 데이터가 다릅니다: {name}')
        context['targets'] = target_index
        root = CACHE_DIR / 'inputs' / context['cache_key'] if context.get('cache_key') else CACHE_ROOT
        entries.extend(cache_context(context, root, periods=(period,)))
    return entries


def select_alpha(entries, contexts, model, checkpoint):
    tensors = {n: tensor_context(c) for n, c in contexts.items()}
    macro, _, _, _ = evaluate(entries, contexts, tensors, model, 'blend', ALPHAS, REPORT_DIR,
                               collect_diagnostics=True)
    best = macro.sort_values([CHECKPOINT_METRIC, 'alpha'], ascending=[False, True]).iloc[0]
    selection = dict(alpha=float(best.alpha), selection_proxy=float(best.selection_proxy),
                     State=float(best.State), criterion=dict(State=1.) if CHECKPOINT_METRIC == 'State' else SELECT_WEIGHTS,
                     selected_on='2031-01-16~31',
                     panels=sorted(contexts), checkpoint_sha256=file_digest(checkpoint))
    path = REPORT_DIR / 'selected_alpha.json'
    path.write_text(json.dumps(selection, indent=2, ensure_ascii=False), encoding='utf-8')
    print(macro.to_string(index=False), flush=True)
    return selection['alpha']


def prediction_alpha(args, contexts):
    if args.alpha is not None:
        return args.alpha
    selection = args.selection or checkpoint_folder(args.checkpoint) / 'selected_alpha.json'
    saved = json.loads(selection.read_text(encoding='utf-8'))
    alpha = float(saved['alpha'])
    if (not 0 <= alpha <= 1 or saved['checkpoint_sha256'] != file_digest(args.checkpoint)
            or not set(contexts) <= set(saved['panels'])):
        raise ValueError('선택한 alpha의 값·체크포인트·패널이 다릅니다.')
    return alpha


def retrieval_memories(contexts):
    from historical_retrieval import HistoricalRetrieval
    entries = evaluation_inputs(contexts, 'fit')
    memories = {}
    for name in contexts:
        memories[name] = HistoricalRetrieval(e['path'] for e in entries if e['panel'] == name)
        print(f'과거 검색 준비: {name}, fit {len(memories[name].dates)}일', flush=True)
    return memories


def retrieval_metrics(frame, weights):
    weights = np.broadcast_to(weights, (len(frame), 2))
    mix = np.stack([1 - weights, weights], -1)
    sse = np.einsum('gci,gcij,gcj->gc', mix, np.stack(frame.gram), mix)
    rms = np.sqrt(np.maximum(sse, 0) / frame['count'].to_numpy()[:, None])
    state = .54 * np.maximum(0, 1 - rms[:, 0] / 25) + .46 * np.maximum(0, 1 - rms[:, 1] / 600)
    values = pd.DataFrame(dict(State=state, speed_rmse_kmh=rms[:, 0], flow_rmse_per_lane=rms[:, 1]), index=frame.index)
    return {k: macro_mean(v.to_dict()) for k, v in values.groupby(level='panel').mean().items()}


def select_retrieval(entries, contexts, tensors, model, memories, checkpoint, folder):
    from historical_retrieval import CONFIG, METHODS
    from scipy.optimize import minimize_scalar
    records = []
    for number, entry in enumerate(entries, 1):
        if entry['period'] != 'early':
            raise ValueError('검색 혼합 비율은 early 구간에서만 선택합니다.')
        day = load_day(entry['path'])
        output = maginet_predict(model, day, tensors[entry['panel']], batch_size=panel_batch_size(contexts[entry['panel']]))
        candidates = memories[entry['panel']].candidates(day, output)
        ti, si = day['ti'], day['si']
        truth = day['truth'][ti, si].astype(np.float64) * SCALE
        baseline_error = output[ti, si].astype(np.float64) * SCALE - truth
        for method, candidate in candidates.items():
            errors = np.stack([baseline_error, candidate[ti, si].astype(np.float64) * SCALE - truth], -1)
            records.append(dict(panel=entry['panel'], regime=entry['regime'], method=method,
                                count=len(truth), gram=np.einsum('nci,ncj->cij', errors, errors)))
        if number == 1 or number % 20 == 0 or number == len(entries):
            print(f'검색 비율 선택 early: {number}/{len(entries)}일', flush=True)
    comparisons = []
    for method in METHODS:
        frame = pd.DataFrame([r for r in records if r['method'] == method]).groupby(['panel', 'regime']).agg(
            count=('count', 'sum'), gram=('gram', 'sum'))
        if len(frame) != 3 * len(contexts):
            raise ValueError('검색 비율 선택에 panel×regime이 누락됐습니다.')
        weights = []
        for channel in (0, 1):
            def loss(w):
                return -retrieval_metrics(frame, np.eye(2)[channel] * w)['State']
            optimum = minimize_scalar(loss, bounds=(0, 1), method='bounded', options=dict(xatol=1e-7))
            if not optimum.success:
                raise RuntimeError('검색 혼합 비율 최적화에 실패했습니다.')
            weights.append(min((0., 1., float(optimum.x)), key=loss))
        comparisons.append(dict(method=method, speed_weight=weights[0], flow_weight=weights[1],
                                **retrieval_metrics(frame, weights), baseline_State=retrieval_metrics(frame, 0.)['State']))
    best = max(comparisons, key=lambda row: row['State'])
    selection = dict(format='maginet-retrieval-v1', config=CONFIG, method=best['method'],
                     weights=[best['speed_weight'], best['flow_weight']], selected_on='early',
                     checkpoint_sha256=file_digest(checkpoint),
                     panels={n: dict(fit_dates=memory.dates, memory_sha256=memory.sha256) for n, memory in memories.items()})
    folder.mkdir(parents=True, exist_ok=True)
    (folder / 'selected_retrieval.json').write_text(json.dumps(selection, indent=2), encoding='utf-8')
    pd.DataFrame(comparisons).to_csv(folder / 'retrieval_early_comparison.csv', index=False)
    print(pd.DataFrame(comparisons).to_string(index=False), flush=True)
    return selection


def retrieval_postprocessors(path, contexts, memories, checkpoint, folder):
    from historical_retrieval import CONFIG, METHODS
    saved = json.loads(path.read_text(encoding='utf-8'))
    weights = np.asarray(saved['weights'], dtype=float)
    if (saved.get('format') != 'maginet-retrieval-v1' or saved['config'] != CONFIG
            or saved['method'] not in METHODS or weights.shape != (2,) or not np.isfinite(weights).all()
            or (weights < 0).any() or (weights > 1).any()
            or saved['checkpoint_sha256'] != file_digest(checkpoint) or not set(contexts) <= set(saved['panels'])):
        raise ValueError('검색 선택 파일의 설정·비율·체크포인트·패널이 다릅니다.')
    for name, memory in memories.items():
        if saved['panels'][name]['memory_sha256'] != memory.sha256:
            raise ValueError(f'검색 비율을 선택한 fit 자료가 다릅니다: {name}')
    folder.mkdir(parents=True, exist_ok=True)
    (folder / 'retrieval_run.json').write_text(json.dumps(saved, indent=2), encoding='utf-8')
    return {name: partial(memory.apply, method=saved['method'], weights=weights) for name, memory in memories.items()}


def import_colab(contexts, entries, source_run):
    source = source_run / 'full' if (source_run / 'full' / 'config.json').is_file() else source_run
    original = json.loads((source / 'config.json').read_text(encoding='utf-8'))
    normalization = fit_input_normalization(entries, CACHE_METADATA_DIR / 'normalization')
    _, expected = notebook_signature(contexts, entries, normalization)
    expected['release'] = original.get('release')
    if json.dumps(expected, sort_keys=True) != json.dumps(original, sort_keys=True):
        raise ValueError('Colab 설정과 로컬 데이터·기간·과거 평균·모델 설정이 다릅니다.')
    old_signature = hashlib.sha256(json.dumps(original, sort_keys=True).encode()).hexdigest()
    signature, settings = run_signature(contexts, entries, normalization)
    target = REPORT_DIR / 'full'
    if (target / 'best_maginet.pt').exists() or (target / 'last_maginet.pt').exists():
        raise FileExistsError('가져오기 결과 폴더에 체크포인트가 이미 있습니다.')
    payloads = {}
    for name in ('best_maginet.pt', 'last_maginet.pt'):
        path = source / name
        if path.is_file():
            state = torch.load(path, map_location='cpu', weights_only=True)
            if state['signature'] != old_signature:
                raise ValueError(f'Colab config.json과 체크포인트 서명이 다릅니다: {path}')
            state.update(signature=signature, **checkpoint_metadata(contexts, normalization, settings))
            payloads[name] = state
    if 'best_maginet.pt' not in payloads:
        raise FileNotFoundError(source / 'best_maginet.pt')
    target.mkdir(parents=True, exist_ok=True)
    for name, state in payloads.items():
        save_checkpoint(state, target / name)
    for name, value in (('config.json', settings), ('input_normalization.json', normalization)):
        (target / name).write_text(json.dumps(value, indent=2, ensure_ascii=False), encoding='utf-8')
    if (source / 'history.csv').is_file():
        shutil.copyfile(source / 'history.csv', target / 'history.csv')
    print(f'Colab 모델을 별도 로컬 폴더에 가져왔습니다: {target}', flush=True)


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('command', choices=('prepare', 'train', 'evaluate', 'predict', 'import-colab'))
    parser.add_argument('--release-root', type=Path, help='기본값: data/release_root.txt')
    parser.add_argument('--cache-dir', type=Path)
    parser.add_argument('--cache-mode', choices=('build', 'reuse'), default='build')
    parser.add_argument('--cache-zip', type=Path)
    parser.add_argument('--legacy-profile-dir', type=Path)
    parser.add_argument('--output-dir', type=Path)
    parser.add_argument('--checkpoint', type=Path)
    parser.add_argument('--panel', action='append')
    parser.add_argument('--input-normalization', choices=('fixed', 'standard'))
    parser.add_argument('--epochs', type=int)
    parser.add_argument('--batch-size', type=int)
    parser.add_argument('--batch-policy', choices=('fixed', 'adaptive'),
                        help='adaptive: 관측소 128개 이하 배치 8, 초과 4; 기본 fixed')
    parser.add_argument('--panel-batch', action='append', metavar='PANEL=SIZE',
                        help='패널별 배치 재지정; 반복 가능, batch-policy보다 우선')
    parser.add_argument('--updates-per-group', type=int)
    parser.add_argument('--days-per-regime', type=int)
    parser.add_argument('--seed', type=int)
    parser.add_argument('--width', type=int)
    parser.add_argument('--blocks', type=int)
    parser.add_argument('--heads', type=int)
    parser.add_argument('--lr', type=float)
    parser.add_argument('--patience', type=int)
    parser.add_argument('--checkpoint-metric', choices=('state', 'proxy'),
                        help='최적 모델·scheduler·조기 종료 기준; 새 공간 실험 기본 state, 기존 실험 proxy')
    parser.add_argument('--resume', action='store_true')
    parser.add_argument('--device', choices=('auto', 'cpu', 'cuda'), default='auto')
    parser.add_argument('--precision', choices=('auto', 'fp32', 'bf16'), default='auto')
    parser.add_argument('--cpu-threads', type=int, default=4)
    parser.add_argument('--period', choices=('early', 'blend', 'february'), default='blend')
    parser.add_argument('--select-alpha', action='store_true')
    parser.add_argument('--alpha', type=float)
    parser.add_argument('--selection', type=Path)
    parser.add_argument('--split', choices=('validation', 'private', 'both'), default='validation')
    parser.add_argument('--source-run', type=Path)
    parser.add_argument('--select-retrieval', action='store_true', help='early에서 과거 검색 방법과 채널별 비율 선택')
    parser.add_argument('--retrieval-selection', type=Path, help='고정한 selected_retrieval.json으로 평가·추론')
    args = parser.parse_args(argv)
    if args.command in ('evaluate', 'predict'):
        training_only = ('input_normalization', 'seed', 'updates_per_group', 'epochs', 'patience', 'lr',
                         'days_per_regime', 'width', 'blocks', 'heads', 'checkpoint_metric')
        supplied = ['--' + key.replace('_', '-') for key in training_only if getattr(args, key) is not None]
        if supplied:
            parser.error(f'평가·추론에서는 저장된 학습 설정을 사용합니다. 지정할 수 없는 옵션: {", ".join(supplied)}')
    if args.command in ('evaluate', 'predict', 'import-colab') and (
            args.batch_policy is not None or args.panel_batch is not None):
        parser.error('패널별 배치는 train·prepare에서 지정하며 평가·추론은 저장된 설정을 사용합니다.')
    if args.command == 'import-colab' and args.source_run is None:
        parser.error('import-colab에는 --source-run이 필요합니다.')
    if args.alpha is not None and not 0 <= args.alpha <= 1:
        parser.error('--alpha는 0~1이어야 합니다.')
    if args.select_alpha and (args.command != 'evaluate' or args.period != 'blend'):
        parser.error('--select-alpha는 evaluate --period blend에서만 사용할 수 있습니다.')
    if args.resume and args.command != 'train':
        parser.error('--resume은 train에서만 사용할 수 있습니다.')
    retrieval = args.select_retrieval or args.retrieval_selection is not None
    if args.select_retrieval and (args.command != 'evaluate' or args.period != 'early'):
        parser.error('--select-retrieval은 evaluate --period early에서만 사용할 수 있습니다.')
    if retrieval and (args.command not in ('evaluate', 'predict') or args.select_alpha or args.selection
                      or args.alpha is not None and args.alpha != 1. or args.select_retrieval and args.retrieval_selection):
        parser.error('검색 옵션은 evaluate·predict에서만 쓰며 기존 alpha 선택과 함께 지정하지 않습니다.')
    if retrieval:
        args.alpha = 1.
    configure(args)
    if args.command in ('prepare', 'train', 'import-colab'):
        if args.command == 'train' and not RESUME:
            if any((REPORT_DIR / 'full' / n).exists() for n in
                   ('best_maginet.pt', 'last_maginet.pt', 'best_state_maginet.pt')):
                raise FileExistsError('기존 체크포인트가 있습니다. --resume 또는 새 --output-dir를 사용해 주세요.')
        contexts, entries = prepare_inputs()
        if args.command == 'prepare':
            normalization = fit_input_normalization(entries, CACHE_METADATA_DIR / 'normalization')
            print(dict(cached_days=len(entries), panels=sorted(contexts), normalization=normalization), flush=True)
        elif args.command == 'import-colab':
            import_colab(contexts, entries, args.source_run)
        else:
            tensors = {n: tensor_context(c) for n, c in contexts.items()}
            model, history = train_maginet(entries, contexts, tensors, REPORT_DIR / 'full')
            print(history.tail().to_string(index=False), flush=True)
            checkpoint = REPORT_DIR / 'full' / 'best_maginet.pt'
            select_alpha(entries, contexts, model, checkpoint)
    else:
        model, contexts, _ = load_local_checkpoint(args.checkpoint, RELEASE_ROOT, saved=args.saved_checkpoint)
        if args.batch_size is not None:
            globals().update(BATCH_SIZE=args.batch_size, BATCH_POLICY='fixed', PANEL_BATCH={})
        if args.panel:
            unknown = set(args.panel) - set(contexts)
            if unknown:
                raise ValueError(f'체크포인트에 없는 패널입니다: {sorted(unknown)}')
            contexts = {n: contexts[n] for n in sorted(set(args.panel))}
        postprocessors = None
        if retrieval:
            memories = retrieval_memories(contexts)
            if args.select_retrieval:
                entries = evaluation_inputs(contexts, 'early')
                select_retrieval(entries, contexts, {n: tensor_context(c) for n, c in contexts.items()}, model,
                                 memories, args.checkpoint, REPORT_DIR)
                return
            postprocessors = retrieval_postprocessors(args.retrieval_selection, contexts, memories,
                                                      args.checkpoint, REPORT_DIR)
        if args.command == 'evaluate':
            entries = evaluation_inputs(contexts, args.period)
            if args.select_alpha:
                select_alpha(entries, contexts, model, args.checkpoint)
            else:
                alpha = prediction_alpha(args, contexts) if args.period == 'february' else (
                    args.alpha if args.alpha is not None else 1.)
                alphas = tuple(sorted({0., alpha}))
                macro, _, _, _ = evaluate(entries, contexts, {n: tensor_context(c) for n, c in contexts.items()},
                                           model, args.period, alphas, REPORT_DIR, collect_diagnostics=True,
                                           postprocessors=postprocessors)
                print(macro.to_string(index=False), flush=True)
        else:
            alpha = prediction_alpha(args, contexts)
            splits = ('validation', 'private') if args.split == 'both' else (args.split,)
            tensors = {n: tensor_context(c) for n, c in contexts.items()}
            for split in splits:
                paths = predict_split(contexts, model, tensors, alpha, split, REPORT_DIR / 'submissions',
                                      postprocessors=postprocessors)
                print('\n'.join(paths), flush=True)


if __name__ == '__main__':
    main()
''')


In [ ]:
# @title
# 내장 run.py — cdf8882 기반 Colab Task 2·3 개선본
load_source('run', r'''"""Train Queue models and build the Task 4 baseline.

The organizer repository is read as an external checkout; no organizer data or
code is copied into this project. See MODEL_PLAN.md for the validation limits.
"""

from __future__ import annotations

import argparse
from pathlib import Path

import numpy as np
import pandas as pd


STATE_KEYS = ["panel", "timestamp", "station_id", "link_id", "mask_regime"]
QUEUE_KEYS = ["window_id", "timestamp", "link_id"]
ODME_KEYS = ["panel", "departure_time", "path_id"]
STATE_COLUMNS = STATE_KEYS + ["speed_kmh", "flow_vph"]
QUEUE_COLUMNS = QUEUE_KEYS + ["queue_pred"]
ODME_COLUMNS = ODME_KEYS + ["origin_zone", "destination_zone", "path_flow"]


def panels(release: Path, selected: list[str] | None) -> list[str]:
    available = sorted(p.name for p in (release / "corridors").iterdir() if p.is_dir())
    if selected:
        unknown = set(selected) - set(available)
        if unknown:
            raise ValueError(f"Unknown panels: {sorted(unknown)}")
        return sorted(set(selected))
    return available
















def fit_queue(args: argparse.Namespace) -> None:
    from learning import train
    train(args, "queue")








def odme_problem(release: Path, name: str, split: str):
    from scipy.sparse import csr_matrix
    network = release / "corridors" / name / "network"
    root = release / "task4" / name / split
    template = pd.read_csv(root / "sample_submission_path_flow.csv", dtype=str)
    if template.empty or template[ODME_COLUMNS[:-1]].isna().any().any():
        raise ValueError(f"Empty or incomplete ODME template: {name} {split}")
    if not template.panel.eq(name).all() or template.departure_time.nunique() != 1:
        raise ValueError(f"Mismatched ODME template period/panel: {name} {split}")
    if template[ODME_KEYS].duplicated().any() or template.path_id.duplicated().any():
        raise ValueError(f"Duplicate ODME template paths: {name} {split}")
    counts = pd.read_csv(root / "synthetic_link_counts.csv", dtype={"link_id": str})
    prior = pd.read_csv(root / "synthetic_weak_prior.csv", dtype={"path_id": str, "departure_time": str})
    incidence = pd.read_csv(network / "path_link_incidence.csv", dtype={"path_id": str, "link_id": str})
    paths = template.path_id.tolist()
    measured = counts.dropna(subset=["count"])
    if measured.link_id.duplicated().any() or prior.path_id.duplicated().any():
        raise ValueError("Duplicate measured links or priors")
    measured = measured[measured.link_id.isin(incidence.link_id)]
    if measured.empty:
        raise ValueError(f"No measured ODME links: {name} {split}")
    row = {link: i for i, link in enumerate(measured.link_id)}
    column = {path: i for i, path in enumerate(paths)}
    included = incidence[incidence.link_id.isin(row) & incidence.path_id.isin(column)]
    matrix = csr_matrix((np.ones(len(included)),
                         (included.link_id.map(row), included.path_id.map(column))),
                        shape=(len(row), len(column)))
    matrix.data[:] = 1.0  # Repeated incidence rows still mean one traversal.
    prior_values = pd.to_numeric(prior.path_flow, errors="coerce").to_numpy(dtype=np.float64)
    if not np.isfinite(prior_values).all() or (prior_values < 0).any():
        raise ValueError(f"Invalid ODME prior: {name} {split}")
    if "departure_time" in prior and not prior.departure_time.eq(template.departure_time.iloc[0]).all():
        raise ValueError(f"Mismatched ODME prior period: {name} {split}")
    base = pd.Series(prior_values, index=prior.path_id).reindex(paths, fill_value=0).to_numpy(dtype=np.float64)
    observed = pd.to_numeric(measured["count"], errors="coerce").to_numpy(dtype=np.float64)
    if not np.isfinite(observed).all() or not np.isfinite(base).all() or (observed < 0).any() or (base < 0).any():
        raise ValueError(f"Invalid counts/prior: {name} {split}")
    return template, matrix, observed, base


def odme_prior_variance(base, power=0, floor_ratio=0.1):
    base = np.asarray(base, dtype=np.float64)
    if base.ndim != 1 or not base.size or not np.isfinite(base).all() or (base < 0).any():
        raise ValueError("ODME prior must be a nonempty finite nonnegative vector")
    if power not in (0, 1, 2) or not np.isfinite(floor_ratio) or floor_ratio <= 0:
        raise ValueError("ODME prior power must be 0, 1, or 2, and floor ratio must be positive")
    if power == 0:
        return np.ones_like(base)
    positive = base[base > 0]
    scale = float(np.median(positive)) if positive.size else 1.0
    variance = (base / scale + floor_ratio) ** power
    variance /= np.median(variance)
    if not np.isfinite(variance).all() or (variance <= 0).any():
        raise ValueError("ODME prior variance is outside the numerical range")
    return variance


def solve_odme(matrix, observed, base, lam, prior_power=0, prior_floor_ratio=0.1):
    from scipy.optimize import lsq_linear, minimize
    from scipy.sparse import csr_matrix
    if lam < 0 or not np.isfinite(lam):
        raise ValueError("ODME lambda must be finite and nonnegative")
    variance = odme_prior_variance(base, prior_power, prior_floor_ratio)
    base = np.asarray(base, dtype=np.float64)
    observed = np.asarray(observed, dtype=np.float64)
    design = csr_matrix(matrix, dtype=np.float64)
    if (observed.ndim != 1 or not observed.size or design.shape != (len(observed), len(base))
            or not np.isfinite(observed).all() or (observed < 0).any()
            or not np.isfinite(design.data).all() or (design.data < 0).any()):
        raise ValueError("Invalid ODME matrix/count dimensions or values")
    if lam == 0:
        result = lsq_linear(design, observed, bounds=(0, np.inf), tol=1e-8, max_iter=2000)
        if not result.success:
            raise RuntimeError(f"ODME solver did not converge: {result.message}")
        return result.x
    precision = lam / variance
    if not np.isfinite(precision).all() or (precision <= 0).any():
        raise ValueError("ODME prior precision is outside the numerical range")
    # The dual has one variable per measured link, independent of path count.
    def objective(dual):
        flow = np.maximum(base - np.asarray(design.T @ dual).ravel() / precision, 0)
        cost = 0.5 * (dual @ dual) + observed @ dual + 0.5 * np.dot(precision * (flow - base), flow + base)
        return cost, dual + observed - design @ flow
    result = minimize(objective, np.zeros(len(observed)), jac=True, method="BFGS",
                      options={"gtol": 1e-5, "maxiter": 2000})
    flow = np.maximum(base - np.asarray(design.T @ result.x).ravel() / precision, 0)
    residual, difference = design @ flow - observed, flow - base
    primal = 0.5 * (residual @ residual + np.dot(precision * difference, difference))
    # Clipped flows minimize the Lagrangian; this gap avoids subtracting large costs.
    gap = 0.5 * np.sum((result.x - residual) ** 2)
    if not np.isfinite(flow).all() or not np.isfinite(primal + gap) or gap > 1e-7 * (1 + primal):
        raise RuntimeError(f"ODME solver did not converge: {result.message}")
    return flow


def predict_odme(release: Path, split: str, output: Path, lam: float = 0.05,
                 prior_power=0, prior_floor_ratio=0.1, selected=None, prior_only=False) -> None:
    parts = []
    for name in panels(release, selected):
        template, matrix, observed, base = odme_problem(release, name, split)
        solution = base.copy() if prior_only else solve_odme(matrix, observed, base, lam, prior_power, prior_floor_ratio)
        template["path_flow"] = solution
        parts.append(template[ODME_COLUMNS])
    if not parts:
        raise ValueError(f"No ODME panels for {split}")
    result = pd.concat(parts, ignore_index=True)
    output.parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(output, index=False)
















def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    commands = parser.add_subparsers(dest="command", required=True)
    for name in ("train-queue", "evaluate", "predict-odme"):
        command = commands.add_parser(name)
        command.add_argument("--release-root", type=Path, required=True)
        if name == "train-queue":
            command.add_argument("--panel", action="append")
            command.add_argument("--checkpoint", type=Path, required=True)
            command.add_argument("--epochs", type=int, default=75, help="Total epoch budget, including resumed epochs")
            command.add_argument("--width", type=int, default=64)
            command.add_argument("--blocks", type=int, default=2)
            command.add_argument("--heads", type=int, default=4)
            command.add_argument("--dropout", type=float, default=0.1)
            command.add_argument("--backend", choices=["mamba2", "mamba1", "scan"], default="mamba2")
            command.add_argument("--graph", action="store_true")
            command.add_argument("--wavelet-levels", type=int, choices=[0, 1, 2, 3], default=0, help="Haar trend/detail levels; 0 disables wavelet")
            command.add_argument("--wavelet-input", choices=["signals", "embedded"], help="New training defaults to signals; resume inherits the checkpoint")
            command.add_argument("--lr", type=float, default=1e-3)
            command.add_argument("--seed", type=int, default=42)
            command.add_argument("--max-days", type=int)
            command.add_argument("--batch-size", type=int, default=0, help="0 selects a panel-size-dependent batch")
            command.add_argument("--panel-batch", action="append", default=[], metavar="PANEL=N")
            command.add_argument("--update-windows", type=int,
                                 help="Windows per optimizer update; 0 selects Queue=48; omitted keeps legacy updates")
            command.add_argument("--workers", type=int, choices=[0, 1, 2], default=1)
            command.add_argument("--cpu-threads", type=int, default=2)
            command.add_argument("--precision", choices=["auto", "fp32", "bf16"], default="auto")
            command.add_argument("--bf16", action="store_true", help="Use CUDA bfloat16 autocast for model forward passes")
            command.add_argument("--patience", type=int, default=5, help="0 disables early stopping")
            command.add_argument("--scheduler-patience", type=int, default=2)
            command.add_argument("--grad-clip", type=float, default=1.0)
            command.add_argument("--log-every", type=int, default=25)
            command.add_argument("--resume", type=Path)
            command.add_argument("--windows-per-day", type=int, default=6)
            command.add_argument("--normal-windows", type=int, default=1)
            command.add_argument("--normal-weight", type=float, default=0.2)
            command.add_argument("--validation-windows", type=int, default=5, help="Fixed maximum per condition/panel")
            command.add_argument('--queue-thresholds', type=float, nargs='+', default=[.1, .2, .3, .4, .5, .6, .7, .8, .9])
            command.add_argument("--window-gap-minutes", type=int, default=360)
            command.add_argument("--history-coverage", type=float, default=0.5)
            command.add_argument("--future-coverage", type=float, default=0.5)
            command.add_argument("--soft-iou-weight", type=float, default=0.2)
            command.add_argument("--speed-weight", type=float, default=0.1)
            command.add_argument("--forward-only", dest="bidirectional", action="store_false")
            command.set_defaults(bidirectional=True)
            command.set_defaults(queue_origin=False, queue_resample=False, queue_condition_gap=False,
                                 queue_changes=False, queue_head="linear", queue_validation_spread=False, queue_public_contract=False)
            command.add_argument("--queue-origin", action="store_true", help="Append the released masked observation at T")
            command.add_argument("--queue-resample", action="store_true", help="Resample fit windows each epoch; keep evaluation fixed")
            command.add_argument("--queue-condition-gap", action="store_true", help="Apply fit window spacing within each condition")
            command.add_argument("--queue-changes", action="store_true", help="Add threshold margin and masked 5/15-minute changes")
            command.add_argument("--queue-head", choices=["linear", "pooled-mlp", "conditional"], default="linear")
            command.add_argument('--queue-future-prior', action='store_true')
            command.add_argument('--queue-label-mode', choices=['hard', 'soft', 'sample'], default='hard')
            command.add_argument('--queue-label-margin', type=float, default=5., help='km/h around each released cutoff')
            command.add_argument('--queue-balanced', action='store_true', help='Equal fit windows per panel/onset/ongoing')
            command.add_argument('--init-queue', type=Path, help='Warm start a new Queue experiment; never resume old optimizer/loss')
            command.add_argument("--queue-public-contract", action="store_true", help="Use the public coverage/persistence rules for Queue validation only")
            command.add_argument("--queue-validation-spread", action="store_true", help="Spread fixed validation windows across dates")
        elif name == "evaluate":
            command.add_argument("--checkpoint", type=Path, required=True)
            command.add_argument("--period", choices=["select", "holdout"], default="select")
            command.add_argument("--output", type=Path, required=True)
            command.add_argument("--panel", action="append")
            command.add_argument("--max-days", type=int)
            command.add_argument("--workers", type=int, choices=[0, 1, 2], default=1)
            command.add_argument("--bootstrap", type=int, default=0)
            command.add_argument("--validation-windows", type=int, help="Override the saved per-condition Queue validation limit")
            command.add_argument("--queue-validation-spread", action="store_true", default=None,
                                 help="Evaluate Queue on fixed windows spread across dates")
        elif name == "predict-odme":
            command.add_argument("--split", choices=["validation", "private"], required=True)
            command.add_argument("--panel", action="append")
            command.add_argument("--odme-lambda", type=float, default=0.05)
            command.add_argument("--prior-power", type=int, choices=[0, 1, 2], default=0)
            command.add_argument("--prior-floor-ratio", type=float, default=0.1)
            command.add_argument("--prior-only", action="store_true", help="Write the released prior without solving")
            command.add_argument("--output", type=Path, required=True)
    args = parser.parse_args()
    if args.command == "train-queue":
        fit_queue(args)
    elif args.command == "evaluate":
        from learning import evaluate
        evaluate(args)
    elif args.command == "predict-odme":
        predict_odme(args.release_root, args.split, args.output, args.odme_lambda,
                     args.prior_power, args.prior_floor_ratio, args.panel, args.prior_only)


if __name__ == "__main__":
    main()
''')


In [ ]:
# @title # 고정된 MagiNet의 링크 차량 수 보정과 최소 변경 투영
load_source('physics_corrector', r'''"""Link-count residual correction with a bounded weighted State projection."""
from pathlib import Path
from types import SimpleNamespace
import hashlib
import json
import zipfile

import numpy as np
import pandas as pd
import torch
from torch import nn
from maginet import __name__ as _maginet_name
import sys

m = sys.modules[_maginet_name]
VERSION = 'frozen-n-corrector-v2'
GAMMAS = (0., .25, .5, 1.)
ENDPOINT_NAMES = ('left', 'right', 'left_slope', 'right_slope', 'left_age', 'right_age',
                  'fraction', 'left_valid', 'right_valid', 'left_slope_valid', 'right_slope_valid')
FEATURES = (['base_speed', 'base_flow', 'speed_minus_interp', 'flow_minus_interp']
            + [f'{channel}_{name}' for channel in ('speed', 'flow') for name in ENDPOINT_NAMES]
            + ['observed_speed', 'observed_flow', 'speed_visible', 'flow_visible', 'target', 'quality',
               'log_N', 'previous_dN', 'next_dN', 'previous_valid', 'next_valid',
               'lanes', 'length', 'free_speed', 'capacity_per_lane', 'occupancy', 'occupancy_visible']
            + [f'N_{name}' for name in ENDPOINT_NAMES]
            + [f'neighbor_{name}' for name in ('speed', 'flow', 'occupancy',
                                              'speed_valid', 'flow_valid', 'occupancy_valid')])


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, allow_nan=False).encode()).hexdigest()


def atomic_json(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix('.json.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False))
    temporary.replace(path)


def segments(consecutive):
    cuts = [0, *(np.flatnonzero(~np.asarray(consecutive, bool)) + 1).tolist(), len(consecutive) + 1]
    return list(zip(cuts[:-1], cuts[1:]))


def endpoint_features(values, visible):
    """Two nearest public observations on each side; slopes are per five minutes."""
    size = len(values)
    t = np.arange(size)[:, None]
    left = np.maximum.accumulate(np.where(visible, t, -1), axis=0)
    right = np.minimum.accumulate(np.where(visible, t, size)[::-1], axis=0)[::-1]
    lv, rv = left >= 0, right < size
    take = lambda array, index: np.take_along_axis(array, np.clip(index, 0, size - 1), axis=0)
    left2 = np.where(left > 0, take(left, left - 1), -1)
    right2 = np.where(right < size - 1, take(right, right + 1), size)
    lsv, rsv = left2 >= 0, right2 < size
    a, b = np.where(lv, take(values, left), 0.), np.where(rv, take(values, right), 0.)
    ls = np.where(lsv, (a - take(values, left2)) / np.maximum(left - left2, 1), 0.)
    rs = np.where(rsv, (take(values, right2) - b) / np.maximum(right2 - right, 1), 0.)
    la, ra = np.where(lv, t - left, 0), np.where(rv, right - t, 0)
    fraction = np.where(lv & rv & (right > left), la / np.maximum(right - left, 1), 0.)
    return np.stack((a, b, ls, rs, np.log1p(la) / np.log(73), np.log1p(ra) / np.log(73),
                     fraction, lv, rv, lsv, rsv), -1).astype(np.float32)


def public_link_fields(d, context, base):
    visible = d['visible'][..., :2]
    values = np.where(visible, d['values'][..., :2], 0.)
    active = d['target'] | (visible.all(-1) & (d['aux'][..., 0] >= .75))
    complete = np.where(d['target'][..., None], base, values).copy()
    complete[..., 0] *= 25
    complete[..., 1] *= 600 * context['lanes']
    links, counts = m.link_mean_numpy(complete, active, context['mapping'], len(context['panel'].links))
    return links, counts, active


def projection_inputs(d, context, device):
    _, _, active = public_link_fields(d, context, d['base'])
    return dict(mapping=torch.as_tensor(context['mapping'], dtype=torch.long, device=device),
                lanes=torch.as_tensor(context['lanes'], dtype=torch.float32, device=device),
                lengths=torch.as_tensor(context['panel'].lengths, dtype=torch.float32, device=device),
                active=torch.as_tensor(active, device=device),
                observed=torch.as_tensor(np.where(d['visible'][..., :2], d['values'][..., :2], 0.),
                                         dtype=torch.float32, device=device),
                group=f"{context['panel'].name}/{int(d['regime'])}")


def correction_features(d, context, base):
    # The input whitelist deliberately excludes truth, observed and active, which
    # are evaluation/loss fields. Eligibility here uses released quality only.
    visible = d['visible'][..., :2]
    values = np.where(visible, d['values'][..., :2], 0.)
    target = d['target']
    if base.shape != values.shape or not np.isfinite(base).all() or (base < 0).any():
        raise ValueError('보정 기준 예측의 형태·값이 잘못되었습니다.')
    if visible[target].any():
        raise ValueError('보정 대상의 정답이 공개 입력에 노출되어 있습니다.')
    consecutive = m.consecutive_five_minutes(d['timestamps'])
    endpoints = []
    for channel in range(2):
        feature = np.zeros((*target.shape, len(ENDPOINT_NAMES)), np.float32)
        for a, b in segments(consecutive):
            feature[a:b] = endpoint_features(values[a:b, :, channel], visible[a:b, :, channel])
        endpoints.append(feature)
    quality = d['aux'][..., 0]
    panel, mapping = context['panel'], context['mapping']
    links, counts, active = public_link_fields(d, context, base)
    n = links[..., 1] / np.maximum(links[..., 0], 1) * panel.lengths
    valid = (counts[:-1] > 0) & (counts[1:] > 0) & consecutive[:, None]
    dn = np.where(valid, np.diff(n, axis=0), 0.)
    signed = np.sign(dn) * np.log1p(np.abs(dn))
    previous, following = np.zeros_like(n), np.zeros_like(n)
    previous[1:], following[:-1] = signed, signed
    pv, nv = np.zeros_like(n), np.zeros_like(n)
    pv[1:], nv[:-1] = valid, valid
    physics = np.stack((np.log1p(n), previous, following, pv, nv), -1)[:, mapping]
    spread = lambda x: np.broadcast_to(x, target.shape)
    public = np.stack((values[..., 0], values[..., 1], visible[..., 0], visible[..., 1],
                       target, quality), -1)
    static = np.stack((spread(context['lanes'] / 5), spread(np.log1p(panel.lengths[mapping])),
                       spread(panel.free_speed[mapping] / 25),
                       spread(panel.capacity[mapping] / panel.lanes[mapping] / 600)), -1)
    occupancy = np.stack((np.where(d['visible'][..., 2], d['values'][..., 2], 0.),
                          d['visible'][..., 2]), -1)
    affected = m.link_mean_numpy(target[..., None].astype(float), active, mapping, len(panel.links))[0][..., 0] > 0
    n_endpoints = np.zeros((*counts.shape, len(ENDPOINT_NAMES)), np.float32)
    for a, b in segments(consecutive):
        n_endpoints[a:b] = endpoint_features(np.log1p(n[a:b]), (counts[a:b] > 0) & ~affected[a:b])
    # Neighbour features use visible measurements from other physical links only.
    neighbors = (np.asarray(context['station_graph'], bool) & (mapping[:, None] != mapping[None, :])).astype(np.float32)
    known = d['visible'] & (quality[..., None] >= .75)
    measured = np.where(known, d['values'], 0.)
    neighbor_count = np.einsum('ij,tjc->tic', neighbors, known.astype(np.float32))
    neighbor_values = np.einsum('ij,tjc->tic', neighbors, measured) / np.maximum(neighbor_count, 1)
    features = np.concatenate((base, base - d['base'], *endpoints, public, physics, static, occupancy,
                               n_endpoints[:, mapping], neighbor_values, neighbor_count > 0), -1)
    if features.shape[-1] != len(FEATURES) or not np.isfinite(features).all():
        raise ValueError('보정 특징에 잘못된 열 또는 NaN/Inf가 있습니다.')
    return features.astype(np.float32), consecutive


class FrozenStateCorrector(nn.Module):
    def __init__(self, statistics):
        super().__init__()
        if statistics.get('fitted_period') != 'fit':
            raise ValueError('보정 통계는 fit 기간에서만 계산해야 합니다.')
        mean, std = statistics['mean'], statistics['std']
        self.groups = tuple(sorted(statistics['projection']))
        rows = [statistics['projection'][key] for key in self.groups]
        bounds = np.asarray([row['bounds'] for row in rows], np.float32)
        variance = np.asarray([row['channel_mse'] for row in rows], np.float32)
        n_bound = np.asarray([row['log_n_bound'] for row in rows], np.float32)
        n_variance = np.asarray([row['log_n_mse'] for row in rows], np.float32)
        if (len(mean) != len(FEATURES) or len(std) != len(FEATURES) or not rows
                or bounds.shape != (len(rows), 2) or variance.shape != bounds.shape
                or not np.isfinite([*mean, *std, *bounds.ravel(), *variance.ravel(), *n_bound, *n_variance]).all()
                or (np.asarray(std) <= 0).any() or (bounds < 0).any() or (variance < 0).any()
                or (n_bound < 0).any() or (n_variance < 0).any()):
            raise ValueError('보정 표준화·출력 상한이 잘못되었습니다.')
        self.register_buffer('mean', torch.as_tensor(mean, dtype=torch.float32))
        self.register_buffer('std', torch.as_tensor(std, dtype=torch.float32).clamp_min(1e-6))
        self.register_buffer('bounds', torch.as_tensor(bounds, dtype=torch.float32))
        self.register_buffer('variance', torch.as_tensor(variance, dtype=torch.float32).clamp_min(1e-6))
        self.register_buffer('n_bound', torch.as_tensor(n_bound, dtype=torch.float32))
        self.register_buffer('n_variance', torch.as_tensor(n_variance, dtype=torch.float32))
        self.net = nn.Sequential(nn.Conv1d(len(FEATURES), 32, 3, padding=1), nn.GELU(),
                                 nn.Conv1d(32, 32, 3, dilation=2, padding=2), nn.GELU(),
                                 nn.Conv1d(32, 1, 1))
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, features, base, target, consecutive, projection, gamma=1.):
        if not 0 <= gamma <= 1:
            raise ValueError('보정 강도는 0~1이어야 합니다.')
        if gamma == 0:
            return base
        if projection['group'] not in self.groups:
            raise ValueError(f"fit 통계가 없는 보정 그룹입니다: {projection['group']}")
        group = self.groups.index(projection['group'])
        mapping, active = projection['mapping'], projection['active']
        nodes = len(projection['lengths'])
        aggregate = lambda x: x.new_zeros((len(x), nodes, *x.shape[2:])).index_add(1, mapping, x)
        counts = aggregate(active.float()).clamp_min(1)
        normalized = ((features - self.mean) / self.std).clamp(-10, 10)
        normalized = aggregate(normalized * active[..., None]) / counts[..., None]
        # Never convolve across missing timestamps or different days.
        raw = torch.cat([self.net(normalized[a:b].permute(1, 2, 0)).permute(2, 0, 1)
                         for a, b in segments(consecutive)], 0)
        complete = torch.where(target[..., None], base, projection['observed'])
        units = torch.stack((torch.full_like(projection['lanes'], 25.), 600 * projection['lanes']), -1)
        links = aggregate(complete * units * active[..., None]) / counts[..., None]
        speed, flow = links[..., 0], links[..., 1]
        denominator = speed.clamp_min(1)
        lengths = projection['lengths']
        density = flow / denominator
        n = density * lengths
        # expm1 makes the zero-initialised path exactly identical to the baseline.
        delta_n = ((n + 1) * torch.expm1(gamma * self.n_bound[group] * torch.tanh(raw[..., 0]))).clamp_min(-n)
        delta_k = torch.where(lengths > 0, delta_n / lengths.clamp_min(1e-6), 0.)
        desired = density + delta_k
        # Below the scorer's 1 km/h floor, only flow can change density locally.
        av = torch.where(speed > 1, -25 * desired, 0.) / counts
        af = 600 * projection['lanes'][None] / counts[:, mapping]
        coefficients = torch.stack((av[:, mapping], af), -1) * target[..., None]
        covariance = self.variance[group] / base.new_tensor([.54, .46]) * (self.bounds[group] > 0)
        weighted = covariance * coefficients
        norm = aggregate((weighted * coefficients).sum(-1))
        # Fit-period N error is a conservative uncertainty scale, not confidence
        # predicted by the new corrector; gamma selection provides another shrinkage.
        noise = (denominator * (n + 1) / lengths.clamp_min(1e-6)).square() * self.n_variance[group]
        multiplier = delta_k * denominator / (norm + noise).clamp_min(1e-8)
        step = weighted * multiplier[:, mapping, None]
        # Contract the entire link step together to preserve its weighted direction.
        ratio = torch.where(step != 0, self.bounds[group] / step.abs().clamp_min(1e-12), 1.)
        ratio = torch.minimum(ratio, torch.where(step < 0, base / (-step).clamp_min(1e-12), 1.))
        station_rate = torch.where(target, ratio.clamp(0, 1).amin(-1), 1.)
        rate = base.new_ones((len(base), nodes)).scatter_reduce(
            1, mapping[None].expand(len(base), -1), station_rate, reduce='amin', include_self=True)
        speed_step = aggregate(step[..., 0] * 25) / counts
        floor_rate = torch.where((speed > 1) & (speed_step < 0),
                                 (speed - 1) / (-speed_step).clamp_min(1e-12), 1.).clamp(0, 1)
        rate = torch.minimum(rate, floor_rate)
        corrected = (base + step * rate[:, mapping, None]).clamp_min(0)
        return torch.where(target[..., None], corrected, base)


def base_prediction(model, d, ct, alpha):
    with torch.inference_mode():
        neural = m.maginet_predict(model, d, ct) if alpha else d['base']
    return ((1 - alpha) * d['base'] + alpha * neural).astype(np.float32)


class PredictionCache:
    def __init__(self, model, contexts, tensors, alpha, folder):
        self.model, self.contexts, self.tensors = model, contexts, tensors
        self.alpha, self.folder = alpha, Path(folder)

    def get(self, entry):
        d = m.load_day(entry['path'])
        with zipfile.ZipFile(entry['path']) as archive:
            identity = [(name, archive.getinfo(name).CRC, archive.getinfo(name).file_size)
                        for name in ('values.npy', 'visible.npy', 'aux.npy', 'base.npy',
                                     'target.npy', 'timestamps.npy')]
        path = self.folder / entry['panel'] / entry['period'] / f"{entry['date']}-{digest(identity)[:16]}.npy"
        if path.exists():
            base = np.load(path, allow_pickle=False)
        else:
            base = base_prediction(self.model, d, self.tensors[entry['panel']], self.alpha)
            path.parent.mkdir(parents=True, exist_ok=True)
            temporary = path.with_suffix('.npy.tmp')
            with temporary.open('wb') as stream:
                np.save(stream, base, allow_pickle=False)
            temporary.replace(path)
        if base.shape != d['base'].shape or not np.isfinite(base).all() or (base < 0).any():
            raise ValueError(f'기준 예측 캐시가 잘못되었습니다: {path}')
        return d, base


def validate_period(entries, period):
    ranges = dict(fit=('2030-06-01', '2030-12-31'), early=('2031-01-01', '2031-01-15'),
                  blend=('2031-01-16', '2031-01-31'))
    lo, hi = ranges[period]
    if not entries or any(e['period'] != period or not lo <= e['date'] <= hi for e in entries):
        raise ValueError(f'보정 {period}에는 {lo}~{hi} 날짜만 사용할 수 있습니다.')


def fit_statistics(entries, cache, seed):
    validate_period(entries, 'fit')
    rng = np.random.default_rng(seed)
    total, square, count, errors, n_errors = np.zeros(len(FEATURES)), np.zeros(len(FEATURES)), 0, {}, {}
    for number, entry in enumerate(entries, 1):
        d, base = cache.get(entry)
        context = cache.contexts[entry['panel']]
        x, _ = correction_features(d, context, base)
        flat = x.reshape(-1, len(FEATURES))
        selected = flat[rng.choice(len(flat), min(len(flat), 512), replace=False)].astype(np.float64)
        total += selected.sum(0)
        square += np.square(selected).sum(0)
        count += len(selected)
        residual = base[d['target']] - d['truth'][d['target']]
        if len(residual):
            key = f"{entry['panel']}/{int(entry['regime'])}"
            errors.setdefault(key, []).append(
                residual[rng.choice(len(residual), min(len(residual), 128), replace=False)])
            predicted, counts, active = public_link_fields(d, context, base)
            reference, _, _ = public_link_fields(d, context, d['truth'])
            affected = m.link_mean_numpy(d['target'][..., None].astype(float), active,
                                         context['mapping'], len(context['panel'].links))[0][..., 0] > 0
            valid = (counts > 0) & affected & (context['panel'].lengths > 0)
            to_n = lambda x: x[..., 1] / np.maximum(x[..., 0], 1) * context['panel'].lengths
            log_error = (np.log1p(to_n(reference)) - np.log1p(to_n(predicted)))[valid]
            if not len(log_error) or not np.isfinite(log_error).all():
                raise ValueError('보정 학습의 유효 차량 수 정답이 없습니다.')
            n_errors.setdefault(key, []).append(log_error[
                rng.choice(len(log_error), min(len(log_error), 128), replace=False)])
        if number == 1 or number % 20 == 0 or number == len(entries):
            print(f'보정 기준 예측·fit 통계: {number}/{len(entries)}일', flush=True)
    if not errors:
        raise ValueError('보정 학습 대상이 없습니다.')
    projection = {}
    for key, rows in errors.items():
        residual, log_error = np.concatenate(rows).astype(np.float64), np.concatenate(n_errors[key])
        projection[key] = dict(bounds=np.quantile(np.abs(residual), .9, axis=0).tolist(),
                               channel_mse=np.square(residual).mean(0).tolist(),
                               log_n_bound=float(np.quantile(np.abs(log_error), .9)),
                               log_n_mse=float(np.square(log_error).mean()))
    mean = total / count
    std = np.sqrt(np.maximum(square / count - mean**2, 0))
    return dict(mean=mean.tolist(), std=np.where(std > 1e-6, std, 1.).tolist(),
                projection=projection, quantile=.9, fitted_period='fit')


def correct_day(model, d, context, base, gamma):
    if model is None or gamma == 0:
        return base.copy()
    x, consecutive = correction_features(d, context, base)
    device = next(model.parameters()).device
    with torch.inference_mode():
        result = model(torch.as_tensor(x, device=device), torch.as_tensor(base, device=device),
                       torch.as_tensor(d['target'], device=device), consecutive,
                       projection_inputs(d, context, device), gamma).cpu().numpy()
    if result.shape != base.shape or not np.isfinite(result).all() or (result < 0).any():
        raise FloatingPointError('보정 출력이 잘못되었습니다.')
    return result


def score_candidates(entries, cache, candidates, folder, label):
    records = []
    for number, entry in enumerate(entries, 1):
        d, base = cache.get(entry)
        c = cache.contexts[entry['panel']]
        truth = d['truth'][d['ti'], d['si']] * m.SCALE
        for name, model, gamma in candidates:
            corrected = correct_day(model, d, c, base, gamma)
            predicted = corrected[d['ti'], d['si']] * m.SCALE
            records.append(dict(candidate=name, alpha=gamma, panel=entry['panel'], date=entry['date'],
                                regime=entry['regime'], **m.state_totals(truth, predicted),
                                **m.physics_totals(predicted, d, c)))
        if number == 1 or number % 20 == 0 or number == len(entries):
            print(f'보정 비교 {label}: {number}/{len(entries)}일', flush=True)
    macro, grouped, daily = m.summarize(records)
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    for name, frame in (('macro', macro), ('panel_regime', grouped), ('daily', daily)):
        frame.rename(columns={'alpha': 'gamma'}).to_csv(folder / f'{label}_{name}.csv', index=False)
    return macro.rename(columns={'alpha': 'gamma'})


def load_corrector(path, contract, device):
    saved = torch.load(path, map_location='cpu', weights_only=True)
    if saved.get('contract') != contract or saved.get('features') != FEATURES:
        raise ValueError('보정 모델의 기준 체크포인트·특징·설정이 다릅니다.')
    stats = saved['statistics']
    model = FrozenStateCorrector(stats).to(device)
    model.load_state_dict(saved['model'])
    return model.eval()


def train_variant(variant, entries, early, cache, statistics, settings, contract, folder):
    folder.mkdir(parents=True, exist_ok=True)
    seed, epochs = settings['seed'], settings['epochs']
    torch.manual_seed(seed)
    model = FrozenStateCorrector(statistics).to(m.DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    last = folder / 'last.pt'
    start, history, best = 1, [], {'State': -float('inf'), 'selection_proxy': -float('inf')}
    if last.exists():
        saved = torch.load(last, map_location=m.DEVICE, weights_only=True)
        if saved['contract'] != contract or saved['variant'] != variant:
            raise ValueError('다른 보정 실험을 재개할 수 없습니다.')
        model.load_state_dict(saved['model'])
        optimizer.load_state_dict(saved['optimizer'])
        start, history, best = saved['epoch'] + 1, saved['history'], saved['best']
    pools = {}
    for entry in entries:
        pools.setdefault((entry['panel'], entry['regime']), []).append(entry)
    expected = {(name, regime) for name in cache.contexts for regime in range(3)}
    if set(pools) != expected:
        raise ValueError('보정 학습의 panel×regime이 누락되었습니다.')
    for epoch in range(start, epochs + 1):
        model.train()
        rng = np.random.default_rng(seed + epoch)
        order = list(pools)
        rng.shuffle(order)
        losses_sum, updates = 0., 0
        for panel, regime in order:
            pool = pools[(panel, regime)]
            for _ in range(settings['steps_per_group']):
                entry = pool[int(rng.integers(len(pool)))]
                d, base = cache.get(entry)
                x, consecutive = correction_features(d, cache.contexts[panel], base)
                output = model(torch.as_tensor(x, device=m.DEVICE), torch.as_tensor(base, device=m.DEVICE),
                               torch.as_tensor(d['target'], device=m.DEVICE), consecutive,
                               projection_inputs(d, cache.contexts[panel], m.DEVICE))
                _, components = m.joint_loss(output[None], d, [0], cache.contexts[panel],
                                                cache.tensors[panel], float(variant == 'physics'), return_components=True)
                loss = components['State']
                if variant == 'physics':
                    loss = loss + .05 * components['N'] + .03 * components['dN']
                directions = sum(p.rsplit('_', 1)[0] == panel.rsplit('_', 1)[0] for p in cache.contexts)
                optimizer.zero_grad(set_to_none=True)
                if not torch.isfinite(loss):
                    raise FloatingPointError('보정 손실에 NaN/Inf가 있습니다.')
                (loss / directions).backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1., error_if_nonfinite=True)
                optimizer.step()
                losses_sum += float(loss.detach())
                updates += 1
            print(f'보정 {variant} epoch {epoch}/{epochs}: {panel}/{m.REGIMES[regime]}', flush=True)
        model.eval()
        row = score_candidates(early, cache, [(variant, model, 1.)], folder, 'early').iloc[0]
        if not np.isfinite([float(row[k]) for k in ('State', 'FD_proxy', 'LWR_proxy', 'selection_proxy')]).all():
            raise FloatingPointError('보정 체크포인트 선택 지표가 유한하지 않습니다.')
        payload = dict(model=model.state_dict(), contract=contract, features=FEATURES,
                       statistics=statistics, variant=variant, epoch=epoch)
        for metric in best:
            if float(row[metric]) > best[metric]:
                best[metric] = float(row[metric])
                m.save_checkpoint(payload, folder / f'best_{metric}.pt')
        history.append(dict(epoch=epoch, loss=losses_sum / updates, updates=updates,
                            **{k: float(row[k]) for k in ('State', 'FD_proxy', 'LWR_proxy', 'selection_proxy')}))
        pd.DataFrame(history).to_csv(folder / 'history.csv', index=False)
        m.save_checkpoint(dict(payload, optimizer=optimizer.state_dict(), history=history, best=best), last)
        print({'corrector': variant, **history[-1]}, flush=True)


def choose_candidate(macro):
    baseline = macro.loc[macro.candidate.eq('baseline')]
    if len(baseline) != 1:
        raise ValueError('보정 선택에는 기준 모델이 정확히 하나 필요합니다.')
    reference = baseline.iloc[0]
    metrics = ['State', 'FD_proxy', 'LWR_proxy', 'N_MAE_proxy', 'DN_MAE_proxy']
    if not np.isfinite(reference[['State', 'FD_proxy', 'LWR_proxy']].to_numpy(dtype=float)).all():
        raise ValueError('기준 모델 지표가 유한하지 않습니다.')
    finite = np.isfinite(macro[metrics].to_numpy()).all(1)
    accepted = (finite & (macro.State >= reference.State - 1e-10)
                & (macro.FD_proxy >= reference.FD_proxy - 1e-10)
                & (macro.N_MAE_proxy <= reference.N_MAE_proxy + 1e-10)
                & (macro.DN_MAE_proxy < reference.DN_MAE_proxy - 1e-6)
                & (macro.LWR_proxy > reference.LWR_proxy + 1e-6))
    comparison = macro.copy()
    comparison['accepted'] = accepted
    for metric in metrics:
        comparison[f'delta_{metric}'] = comparison[metric] - reference[metric]
    if not accepted.any():
        return reference, comparison
    selected = macro.loc[accepted].sort_values(['LWR_proxy', 'State', 'gamma'], ascending=[False, False, True]).iloc[0]
    return selected, comparison


def open_base(checkpoint, release, cache_dir, panels=None):
    checkpoint, cache_dir = Path(checkpoint).resolve(), Path(cache_dir).resolve()
    m.RELEASE_ROOT, m.CACHE_DIR, m.CACHE_ROOT = m.resolve_release(release), cache_dir, cache_dir
    m.CACHE_METADATA_DIR = cache_dir / 'metadata'
    m.TRAIN_MODEL = False
    model, contexts, _ = m.load_local_checkpoint(checkpoint, m.RELEASE_ROOT)
    if panels is not None:
        if not set(panels) <= set(contexts):
            raise ValueError('기준 모델에 없는 패널입니다.')
        contexts = {name: contexts[name] for name in sorted(panels)}
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    model.eval()
    alpha_path = m.checkpoint_folder(checkpoint) / 'selected_alpha.json'
    alpha = m.prediction_alpha(SimpleNamespace(alpha=None, selection=alpha_path, checkpoint=checkpoint), contexts)
    identity = dict(version=VERSION, checkpoint_sha256=m.file_digest(checkpoint), alpha=alpha,
                    panels=sorted(contexts), features=FEATURES,
                    source=globals().get('__source_sha256__', VERSION),
                    maginet_source=getattr(m, '__source_sha256__', 'embedded'))
    tensors = {name: m.tensor_context(c) for name, c in contexts.items()}
    cache_identity = dict(base=identity, precision=str(m.AMP_DTYPE) if m.USE_AMP else 'fp32',
                          device=m.DEVICE.type, torch=str(torch.__version__))
    cache = PredictionCache(model, contexts, tensors, alpha, cache_dir / 'corrector' / digest(cache_identity)[:20])
    return model, contexts, tensors, cache, identity


def train_and_select(checkpoint, release, cache_dir, output_dir, panels=None, epochs=10, steps_per_group=4, seed=42):
    if epochs < 1 or steps_per_group < 1:
        raise ValueError('보정 epoch와 그룹별 업데이트 수는 1 이상이어야 합니다.')
    model, contexts, tensors, cache, identity = open_base(checkpoint, release, cache_dir, panels)
    settings = dict(epochs=int(epochs), steps_per_group=int(steps_per_group), seed=int(seed),
                    width=32, lr=3e-4, weight_decay=1e-4, n_weight=.05, dn_weight=.03, quantile=.9,
                    projection='link-soft-weighted-v1', reliability_period='fit')
    identity['dates'] = {name: {p: c['dates'][p] for p in ('fit', 'early', 'blend')} for name, c in contexts.items()}
    contract = digest(dict(base=identity, settings=settings))
    folder = Path(output_dir) / contract[:20]
    folder.mkdir(parents=True, exist_ok=True)
    selection_path = folder / 'selection.json'
    if selection_path.exists():
        load_selection(selection_path, checkpoint, release, cache_dir, panels)
        print(f'완료된 보정 실험 재사용: {selection_path}')
        return selection_path
    atomic_json(dict(contract=contract, base=identity, settings=settings), folder / 'config.json')
    fit, early, blend = (m.evaluation_inputs(contexts, period) for period in ('fit', 'early', 'blend'))
    for entries, period in ((fit, 'fit'), (early, 'early'), (blend, 'blend')):
        validate_period(entries, period)
    stats_path = folder / 'statistics.json'
    statistics = json.loads(stats_path.read_text()) if stats_path.exists() else fit_statistics(fit, cache, seed)
    atomic_json(statistics, stats_path)
    for variant in ('state', 'physics'):
        train_variant(variant, fit, early, cache, statistics, settings, contract, folder / variant)
    candidates, paths = [('baseline', None, 0.)], {}
    for variant in ('state', 'physics'):
        for metric in ('State', 'selection_proxy'):
            path = folder / variant / f'best_{metric}.pt'
            corrector = load_corrector(path, contract, m.DEVICE)
            for gamma in GAMMAS[1:]:
                name = f'{variant}_{metric}_g{gamma:g}'
                candidates.append((name, corrector, gamma))
                paths[name] = path
    macro = score_candidates(blend, cache, candidates, folder, 'blend')
    chosen, comparison = choose_candidate(macro)
    comparison.to_csv(folder / 'comparison.csv', index=False)
    path = paths.get(chosen.candidate)
    atomic_json(dict(version=VERSION, base=identity, contract=contract, candidate=str(chosen.candidate),
                     gamma=float(chosen.gamma), selected_on='2031-01-16~31',
                     corrector=str(path.relative_to(folder)) if path else None,
                     corrector_sha256=m.file_digest(path) if path else None,
                     metrics={k: float(chosen[k]) if np.isfinite(chosen[k]) else None
                              for k in ('State', 'FD_proxy', 'LWR_proxy', 'Physics_proxy',
                                        'N_MAE_proxy', 'DN_MAE_proxy')}), selection_path)
    print(comparison.to_string(index=False), flush=True)
    return selection_path


def load_selection(selection_path, checkpoint, release, cache_dir, panels=None):
    selection_path = Path(selection_path).resolve()
    saved = json.loads(selection_path.read_text())
    base_model, contexts, tensors, cache, identity = open_base(checkpoint, release, cache_dir, panels)
    if (saved.get('version') != VERSION or saved.get('selected_on') != '2031-01-16~31'
            or any(saved['base'].get(key) != value for key, value in identity.items())
            or saved.get('gamma') not in GAMMAS):
        raise ValueError('보정 선택값의 기준 모델·alpha·패널·소스가 다릅니다. 해당 설정으로 다시 선택해 주세요.')
    corrector = None
    if saved['gamma']:
        path = (selection_path.parent / saved['corrector']).resolve()
        if not path.is_relative_to(selection_path.parent) or m.file_digest(path) != saved['corrector_sha256']:
            raise ValueError('선택된 보정 체크포인트가 변경되었습니다.')
        corrector = load_corrector(path, saved['contract'], m.DEVICE)
    elif saved.get('corrector') is not None or saved.get('candidate') != 'baseline':
        raise ValueError('보정 미적용 선택값이 잘못되었습니다.')
    callbacks = {name: (lambda d, values, c=c: correct_day(corrector, d, c, values, saved['gamma']))
                 for name, c in contexts.items()}
    return saved, base_model, contexts, tensors, cache, callbacks


def evaluate_selected(selection_path, checkpoint, release, cache_dir, panels=None, period='february'):
    if period not in ('early', 'blend', 'february'):
        raise ValueError('지원하지 않는 보정 평가 기간입니다.')
    saved, base_model, contexts, tensors, cache, callbacks = load_selection(
        selection_path, checkpoint, release, cache_dir, panels)
    entries = m.evaluation_inputs(contexts, period)
    corrector = (load_corrector(Path(selection_path).parent / saved['corrector'], saved['contract'], m.DEVICE)
                 if saved['gamma'] else None)
    return score_candidates(entries, cache, [('baseline', None, 0.), ('selected', corrector, saved['gamma'])],
                            Path(selection_path).parent, period)


def predict_selected(selection_path, checkpoint, release, cache_dir, panels=None):
    saved, model, contexts, tensors, cache, callbacks = load_selection(
        selection_path, checkpoint, release, cache_dir, panels)
    paths = []
    for split in ('validation', 'private'):
        paths.extend(m.predict_split(contexts, model, tensors, cache.alpha, split,
                                     Path(selection_path).parent / 'submissions', finalprocessors=callbacks))
    return [Path(path) for path in paths]
''')


## 4. 실행 준비와 Task 2 backend 점검

작은 입력으로 Task 2의 순전파·역전파를 먼저 확인합니다. 선택한 backend가 이 GPU에서 실패하면 긴 MagiNet 학습 전에 중단합니다.


In [ ]:
maginet = MODULES["maginet"]
physics_corrector = MODULES["physics_corrector"]
learning = MODULES["learning"]
runner = MODULES["run"]
fast_model = MODULES["fast_model"]

def call_queue(arguments):
    previous = sys.argv[:]
    print("Queue:", shlex.join(map(str, arguments)), flush=True)
    try:
        sys.argv = ["task123-notebook", *map(str, arguments)]
        runner.main()
    finally:
        sys.argv = previous
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

def call_maginet(arguments):
    print("MagiNet:", shlex.join(map(str, arguments)), flush=True)
    try:
        maginet.main(list(map(str, arguments)))
    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


def training_action(best, last, budget, resume):
    """Choose fresh training, resume, or reuse for one task independently."""
    if budget < 1:
        raise ValueError("목표 epoch는 1 이상이어야 합니다.")
    if not best.exists() and not last.exists():
        return "new"
    if not resume:
        raise FileExistsError(f"기존 모델을 보호합니다. RESUME=True 또는 새 결과 폴더를 사용해 주세요: {best.parent}")
    if not best.is_file() or not last.is_file():
        raise FileNotFoundError(f"재개에는 best와 last 체크포인트가 모두 필요합니다: {best}, {last}")
    saved = torch.load(last, map_location="cpu", weights_only=True)
    epoch, bad_epochs = saved["epoch"], saved["bad_epochs"]
    if saved.get("format_version") == 3:
        settings = saved["settings"]
        minimum = (settings["warmup"] + max(settings["ramp"], 1) if saved.get('physics_schedule') == 'after-ramp-v1'
                   else settings["warmup"] + settings["ramp"] + 2)
        stopped = bad_epochs >= settings["patience"] and epoch >= minimum
    elif saved.get("format_version") == 2 and saved.get("task") == "queue":
        patience = saved["training_args"]["patience"]
        stopped = patience > 0 and bad_epochs >= patience
    else:
        raise ValueError(f"지원하지 않는 재개 체크포인트입니다: {last}")
    return "complete" if epoch >= budget or stopped else "resume"

panel_args = [value for panel in (selected_panels or []) for value in ("--panel", panel)]
queue_eval_args = ["--queue-validation-spread", "--validation-windows",
                   str(VALIDATION_WINDOWS)]
device = "cuda" if torch.cuda.is_available() else "cpu"
features = len(MODULES["data"].queue_feature_names("--queue-changes" in QUEUE_FLAGS))
probe = fast_model.TrafficQueueModel(features=features, node_count=2, width=64, blocks=1,
            heads=4, backend=EFFECTIVE_BACKEND,
            graph="--graph" in QUEUE_FLAGS,
            queue_head=QUEUE_FLAGS[QUEUE_FLAGS.index("--queue-head") + 1] if "--queue-head" in QUEUE_FLAGS else "linear",
            queue_future_prior="--queue-future-prior" in QUEUE_FLAGS).to(device)
try:
    with torch.autocast(device, dtype=torch.bfloat16, enabled=device == "cuda" and QUEUE_PRECISION == "bf16"):
        logits, speed = probe(torch.zeros(1, 13 if "--queue-origin" in QUEUE_FLAGS else 12, 2, features, device=device),
                              torch.tensor([1, 2], device=device), ((0, 2),),
                              torch.tensor([[0], [1]], device=device),
                              condition=torch.zeros(1, dtype=torch.long, device=device),
                              future=torch.zeros(1, 6, 2, len(MODULES["data"].QUEUE_FUTURE_NAMES), device=device))
        loss = logits.float().square().mean() + speed.float().square().mean()
    loss.backward()
    assert logits.shape == speed.shape == (1, 6, 2)
    assert torch.isfinite(loss) and all(torch.isfinite(p.grad).all() for p in probe.parameters() if p.grad is not None)
finally:
    del probe
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
(RESULTS / "notebook_run.json").write_text(json.dumps({
    "source_commit": SOURCE_COMMIT, "source_sha256": SOURCE_HASHES,
    "mode": RUN_MODE, "panels": selected_panels, "experiment": MAGINET_EXPERIMENT,
    "queue_backend": EFFECTIVE_BACKEND, "queue_flags": QUEUE_FLAGS,
    "queue_label_mode": QUEUE_LABEL_MODE, "queue_label_margin": QUEUE_LABEL_MARGIN,
    "queue_init_checkpoint": str(QUEUE_INIT_CHECKPOINT) if QUEUE_INIT_CHECKPOINT is not None else None,
    "torch": str(torch.__version__), "precision": QUEUE_PRECISION,
    "state_epochs": MAGINET_EPOCHS,
    "queue_epochs": QUEUE_EPOCHS,
}, ensure_ascii=False, indent=2), encoding="utf-8")
print("Task 2 backend 순전파·역전파 확인 완료")


In [14]:
# 4번 backend 점검 코드 셀 바로 아래, 5번 학습 셀 위에 새 코드 셀로 붙여 넣습니다.
# 같은 배포본으로 만든 본인 task13 MagiNet 캐시를 명시적으로 재사용합니다.
# 원본 출처 해시가 없는 구형 캐시는 날짜·관측소·마스크·fit 과거 평균 일치를 검사합니다.
from pathlib import Path
from types import FunctionType
DRIVE_CACHE_ZIP = Path('/content/drive/MyDrive/dataset/trafficflow/trafficflow_maginet_cache/preprocessed-86qnwmwe.zip')
DRIVE_CACHE_PROFILE_DIR = Path('/content/drive/MyDrive/dataset/trafficflow/trafficflow_maginet_reports/full')
DRIVE_CACHE_METADATA_DIR = Path(
    "/content/drive/MyDrive/dataset/trafficflow/trafficflow_maginet_reports/full"
)
# 기존 task123 캐시와 분리된 로컬 폴더에 ZIP을 한 번만 풉니다.
CACHE_DIR = WORK_ROOT / 'cache' / 'maginet-drive-86qnwmwe'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
if not DRIVE_CACHE_ZIP.is_file():
    raise FileNotFoundError(DRIVE_CACHE_ZIP)

def _drive_load_fit_profile(panel, targets, fit_dates, files):
    metadata = dict(format='maginet-input-v1', panel=panel.name, links=panel.links,
                    stations=targets.stations, station_links=targets.links,
                    fit_dates=fit_dates, aux_names=list(AUX_NAMES), scale=SCALE.tolist())
    key = hashlib.sha256(json.dumps(metadata, sort_keys=True).encode()).hexdigest()[:16]
    folder = CACHE_METADATA_DIR / 'profiles' / panel.name / key
    folder.mkdir(parents=True, exist_ok=True)
    profile_path, metadata_path = folder / 'fit_profile.npy', folder / 'cache_metadata.json'
    legacy = CACHE_ROOT / panel.name
    legacy_metadata = legacy / 'cache_metadata.json'
    if legacy_metadata.exists() and json.loads(legacy_metadata.read_text()) != metadata:
        raise ValueError(f'이전 캐시의 학습 날짜·관측소·입력 형식이 다릅니다: {panel.name}')
    if not metadata_path.exists() and legacy_metadata.exists():
        legacy_profile = legacy / 'fit_profile.npy'
        if not legacy_profile.is_file():
            raise FileNotFoundError(f'이전 캐시 과거 평균이 누락되었습니다: {legacy_profile}')
        profile = np.load(legacy_profile, allow_pickle=False)
        if profile.shape != (2016, len(panel.links), 3) or not np.isfinite(profile).all():
            raise ValueError(f'이전 캐시 과거 평균의 형태·값이 잘못되었습니다: {panel.name}')
        temporary = profile_path.with_suffix('.npy.tmp')
        shutil.copyfile(legacy_profile, temporary)
        temporary.replace(profile_path)
        metadata_temporary = metadata_path.with_suffix('.json.tmp')
        metadata_temporary.write_text(json.dumps(metadata, ensure_ascii=False, sort_keys=True))
        metadata_temporary.replace(metadata_path)
    if metadata_path.exists():
        if json.loads(metadata_path.read_text()) != metadata:
            raise ValueError(f'저장 과거 평균의 학습 날짜·관측소·입력 형식이 다릅니다: {metadata_path}')
        if not profile_path.is_file():
            raise FileNotFoundError(f'캐시 과거 평균이 누락되었습니다: {profile_path}')
        profile = np.load(profile_path, allow_pickle=False)
    else:
        candidates = [CACHE_PROFILE_DIR / f'fit_profile-{panel.name}.npy']
        candidates += sorted(CACHE_CONTAINER.rglob(f'fit_profile-{panel.name}.npy'))
        profile = None
        for candidate in dict.fromkeys(candidates):
            config_path = candidate.parent / 'config.json'
            if not candidate.is_file() or not config_path.is_file():
                continue
            config = json.loads(config_path.read_text())
            saved = config.get('panels', {}).get(panel.name, {})
            if (saved.get('stations') != targets.stations or saved.get('links') != targets.links
                    or saved.get('dates', {}).get('fit') != fit_dates):
                raise ValueError(f'이전 과거 평균의 관측소·fit 날짜가 다릅니다: {config_path}')
            profile = np.load(candidate, allow_pickle=False)
            if hashlib.sha256(profile.tobytes()).hexdigest() != saved.get('profile_hash'):
                raise ValueError(f'과거 평균의 hash가 설정과 다릅니다: {candidate}')
            print(f'저장 과거 평균 재사용: {candidate}', flush=True)
            break
        if profile is None:
            raise FileNotFoundError(f'{panel.name}: 저장된 fit 과거 평균이 없습니다. DRIVE_CACHE_PROFILE_DIR에 기존 full/config.json과 fit_profile-{panel.name}.npy가 있는 폴더를 지정해 주세요.')
        if profile.shape != (2016, len(panel.links), 3) or not np.isfinite(profile).all():
            raise ValueError(f'저장 과거 평균의 형태·값이 잘못되었습니다: {panel.name}')
        temporary = profile_path.with_suffix('.npy.tmp')
        with temporary.open('wb') as stream:
            np.save(stream, profile, allow_pickle=False)
        temporary.replace(profile_path)
        metadata_temporary = metadata_path.with_suffix('.json.tmp')
        metadata_temporary.write_text(json.dumps(metadata, ensure_ascii=False, sort_keys=True))
        metadata_temporary.replace(metadata_path)
    if profile.shape != (2016, len(panel.links), 3) or not np.isfinite(profile).all():
        raise ValueError(f'캐시 과거 평균의 형태·값이 잘못되었습니다: {panel.name}')
    return profile.astype(np.float32, copy=False)

def _drive_cache_context(context, cache_root, date_overrides=None, periods=('fit', 'early', 'blend')):
    started, entries = time.perf_counter(), []
    for period in periods:
        dates = context['dates'][period] if date_overrides is None else date_overrides[period]
        folder = cache_root / context['panel'].name / period
        for number, date in enumerate(dates, 1):
            path = folder / f'{date}.npz'
            if not path.is_file():
                raise FileNotFoundError(f'저장 ZIP에 필요한 캐시가 없습니다: {path}. 재전처리는 수행하지 않습니다.')
            with np.load(path, allow_pickle=False) as data:
                regime, targets = validate_cache_targets(data, context, date)
                if TRAIN_MODEL and MASK_RESAMPLE and period == 'fit':
                    validate_resampling_occupancy(data)
            entries.append(dict(panel=context['panel'].name, date=date, period=period, regime=regime,
                                targets=targets, path=str(path)))
            if number == 1 or number % 20 == 0 or number == len(dates):
                print(f'기존 캐시 재사용 {context["panel"].name}/{period}: {number}/{len(dates)}일', flush=True)
    context['cache_seconds'] = context.get('cache_seconds', 0.) + time.perf_counter() - started
    return entries

# 내장 원본 셀을 수정하지 않고 구형 캐시 로더만 연결합니다.
maginet.load_fit_profile = FunctionType(_drive_load_fit_profile.__code__, maginet.__dict__,
                                       argdefs=_drive_load_fit_profile.__defaults__)
maginet.cache_context = FunctionType(_drive_cache_context.__code__, maginet.__dict__,
                                    argdefs=_drive_cache_context.__defaults__)
_base_call = getattr(call_maginet, '_without_drive_cache', call_maginet)

def _call_with_drive_cache(arguments, _base=_base_call):
    args = list(arguments)
    for flag, value in (('--cache-dir', CACHE_DIR), ('--cache-zip', DRIVE_CACHE_ZIP),
                        ('--cache-mode', 'reuse'), ('--legacy-profile-dir', DRIVE_CACHE_PROFILE_DIR)):
        if flag in args:
            args[args.index(flag) + 1] = value
        else:
            args += [flag, value]
    return _base(args)

_call_with_drive_cache._without_drive_cache = _base_call
call_maginet = _call_with_drive_cache
run_file = RESULTS / 'notebook_run.json'
if run_file.is_file():
    record = json.loads(run_file.read_text())
    record['legacy_cache'] = dict(zip=str(DRIVE_CACHE_ZIP), profiles=str(DRIVE_CACHE_PROFILE_DIR),
                                 validation='trusted-legacy-template-and-profile', raw_source_hash_verified=False)
    pending = run_file.with_suffix('.json.tmp')
    pending.write_text(json.dumps(record, ensure_ascii=False, indent=2))
    pending.replace(run_file)
print('기존 Drive ZIP 재사용 설정 완료. 누락·불일치 시 새로 전처리하지 않고 중단합니다.')


기존 Drive ZIP 재사용 설정 완료. 누락·불일치 시 새로 전처리하지 않고 중단합니다.


## 5. Task 1·3 학습

`current003`의 N·dN 손실과 기존 모델 구조를 사용합니다. 새 학습의 체크포인트 선택 기준은 State이며, 재개할 때는 저장된 기준과 학습 설정을 유지합니다. 모델·보고서는 `<실험>-current003/`에 저장합니다.


In [ ]:
MAGINET_CHECKPOINT = MAGINET_DIR / "full" / "best_maginet.pt"
if RUN_MODE != "predict":
    budget = MAGINET_EPOCHS
    action = training_action(MAGINET_CHECKPOINT, MAGINET_DIR / "full/last_maginet.pt", budget, RESUME)
    print("Task 1·3 학습 상태:", action, "(complete는 저장된 모델을 사용합니다.)")
    if action != "complete":
        args = ["train", "--release-root", RELEASE, "--cache-dir", CACHE_DIR,
                "--output-dir", MAGINET_DIR,
                "--epochs", budget, "--batch-size", MAGINET_BATCH_SIZE,
                '--cpu-threads', CPU_THREADS, '--precision', 'auto', *panel_args]
        if action == "new":
            args += ["--checkpoint-metric", "state", "--seed", SEED]
        if action == "resume":
            args += ["--resume"]
        call_maginet(args)
if not MAGINET_CHECKPOINT.is_file():
    raise FileNotFoundError(MAGINET_CHECKPOINT)
print("Task 1·3 체크포인트:", MAGINET_CHECKPOINT)


## 6. Task 2 학습

새 기본 실험은 `QUEUE_TAG="conditional-future-sampled-v4"`입니다. 공통 인코더 뒤의 onset/ongoing 출력층을 분리하고, 미래 6시점의 요일·시각·fit-only 과거 평균을 0 초기화된 출력 보정 경로로 제공합니다. 조건은 과거 `[T-60,T)`에서만 정하며, 실제 제출에는 공개 `window_index.condition`을 우선 사용합니다. 공개 `T`는 masked 파일에서만 읽습니다.

`QUEUE_LABEL_MODE="sample"`은 학습에서 `p=clip(0.5+(v_cut-v)/(2m),0,1)`로 만든 확률로 정체 정답을 뽑습니다. `m=QUEUE_LABEL_MARGIN`이며 기본 5km/h는 검증할 시작값입니다. 같은 마스크를 BCE·IoU에 사용하고, 원래 hard label은 평가·창 선정에 유지합니다. 표본별 seed·epoch로 재현하며 검증·제출에서는 샘플링하지 않습니다. `"soft"`는 확률을 직접, `"hard"`는 기존 라벨을 사용합니다.

`--queue-balanced`는 적격 fit 창의 위치만 캐시하고 패널×onset/ongoing마다 같은 수를 뽑습니다. 부족한 그룹은 복원추출하며 epoch마다 다시 선택합니다. 주 학습에는 두 정체 조건을 사용하고 정상 창은 기존 검증 진단에 남깁니다. 공통 48개 창 누적으로 마이크로 배치 차이를 줄입니다. 반복 추출한 같은 날·같은 창의 특징은 한 번만 생성해 공유합니다. `training_group_counts`와 `updates_by_panel`에 실제 수를 저장합니다.

기존 Queue로 시작하려면 `QUEUE_INIT_CHECKPOINT`에 기존 `queue.pt`를 지정합니다. 같은 과거 입력·도로 구조의 인코더와 출력층을 복사하고, pooled-mlp 출력층은 두 조건에 동일하게 복제합니다. 미래 통계 보정은 0부터 시작하며 새 optimizer와 새 결과 폴더를 사용합니다. `RESUME=True`는 같은 v4 실험의 optimizer·epoch·라벨 설정을 복원하는 용도입니다. 구조·라벨 설정을 바꿀 때는 새 `QUEUE_TAG`를 사용해 주세요.

미래 과거 평균은 6~12월에 계산한 체크포인트 profile만 사용합니다. 새 기능의 점수 효과는 같은 대상과 별도 날짜 블록에서 비교해 판단합니다. 기존 예측을 재현하려면 원래 Queue tag와 저장된 체크포인트를 사용합니다.

Wavelet 구현과 `--wavelet-levels`·`--wavelet-input` 옵션은 유지합니다. 예를 들어 `QUEUE_FLAGS`에 `["--wavelet-levels", "1", "--wavelet-input", "signals"]`를 추가하면 신호 입력에 적용합니다. Adaptive graph·missing-aware embedding·reliability graph와 예전 라벨 보정 옵션은 제거했습니다.


In [ ]:
if RUN_MODE != "predict":
    budget = QUEUE_EPOCHS
    last = QUEUE_CHECKPOINT.with_name("queue.last.pt")
    action = training_action(QUEUE_CHECKPOINT, last, budget, RESUME)
    print("Task 2 학습 상태:", action, "(complete는 저장된 모델을 사용합니다.)")
    if action != "complete":
        args = ["train-queue", "--release-root", RELEASE, "--checkpoint", QUEUE_CHECKPOINT,
                "--backend", EFFECTIVE_BACKEND, "--epochs", budget,
                "--precision", QUEUE_PRECISION, "--cpu-threads", CPU_THREADS,
                "--seed", SEED, "--workers", 0, *queue_eval_args, *QUEUE_FLAGS, *panel_args,
                "--queue-label-mode", QUEUE_LABEL_MODE, "--queue-label-margin", QUEUE_LABEL_MARGIN,
                "--queue-thresholds", *QUEUE_THRESHOLDS]
        if action == "new" and QUEUE_INIT_CHECKPOINT is not None:
            args += ["--init-queue", QUEUE_INIT_CHECKPOINT, "--lr", 1e-4]
        if action == 'new' and QUEUE_PUBLIC_VALIDATION:
            args += ['--queue-public-contract']
        if action == "resume":
            args += ["--resume", last]
        call_queue(args)
if not QUEUE_CHECKPOINT.is_file():
    raise FileNotFoundError(QUEUE_CHECKPOINT)
print("Task 2 체크포인트:", QUEUE_CHECKPOINT)


## 7. 평가·보간 비교·과거 검색

`blend_macro.csv`의 `alpha_0.0`은 시간 보간, `alpha_1.0`은 모델 단독입니다. 검색을 켜면 별도 `retrieval/` 아래에 검색을 적용한 모델과 보간의 비교를 저장합니다. 검색 방법·두 채널의 혼합 비율은 1월 전반에서 선택하며, Task 2에는 검색 결과를 넣지 않습니다. Queue는 다음 7.1절에서 같은 고정 표본으로 원래 임계값과 후처리 후보를 비교합니다. `RUN_MODE="predict"`에서는 이 셀의 평가와 검색 비율 선택을 자동으로 건너뜁니다.


In [ ]:
if RUN_MODE == "predict":
    print("예측 모드: 평가·재선택 없이 저장된 모델과 혼합 선택값을 사용합니다.")
else:
    eval_base = ["--release-root", RELEASE, "--cache-dir", CACHE_DIR,
                 "--checkpoint", MAGINET_CHECKPOINT, "--cpu-threads", CPU_THREADS, *panel_args]

    alpha_selection = MAGINET_DIR / "selected_alpha.json"
    try:
        selected_alpha = json.loads(alpha_selection.read_text(encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError):
        selected_alpha = {}
    saved_model = torch.load(MAGINET_CHECKPOINT, map_location="cpu", weights_only=True)
    needed_panels = set(selected_panels or saved_model["contexts"])
    del saved_model
    if (selected_alpha.get("checkpoint_sha256") != maginet.file_digest(MAGINET_CHECKPOINT)
            or not needed_panels <= set(selected_alpha.get("panels", []))):
        call_maginet(["evaluate", *eval_base, "--period", "blend", "--select-alpha"])

    call_maginet(["evaluate", *eval_base, "--period", "blend", "--alpha", 1])
    if USE_RETRIEVAL:
        call_maginet(["evaluate", *eval_base, "--period", "early", "--select-retrieval"])
        call_maginet(["evaluate", *eval_base, "--period", "blend", "--retrieval-selection", RETRIEVAL_SELECTION])
    if RUN_FEBRUARY:
        extra = ["--retrieval-selection", RETRIEVAL_SELECTION] if USE_RETRIEVAL else []
        call_maginet(["evaluate", *eval_base, "--period", "february", *extra])
    display(pd.read_csv(MAGINET_DIR / "blend_macro.csv"))
    if USE_RETRIEVAL:
        display(pd.read_csv(MAGINET_DIR / "retrieval" / "retrieval_early_comparison.csv"))
        display(pd.read_csv(MAGINET_DIR / "retrieval" / "blend_macro.csv"))


## 7.1. Queue 고정 검증·확률·후처리 비교

관측 링크에서 과거 관측률 70% 이상이고 미래 정체가 있는 구간을 고르며, persistence IoU가 0.9를 넘는 쉬운 구간을 제외합니다. 조건별 간격은 360분입니다. 숨겨진 정답을 쓰는 공식 점수와 달리 공개 관측 라벨로 만든 대리 검증입니다.

같은 고정 표본의 확률과 고정 검증 목록을 캐시하고 저장된 임계값, 전부 0인 window의 최대 확률 한 곳만 1로 만드는 후보, 전역 임계값, 과거 관측만으로 구분한 조건별 임계값을 비교합니다. 선택은 1월에서만 합니다. `RUN_FEBRUARY=True`일 때 2월에는 선택값을 고정합니다. 상세 window·5~30분별 TP/FP/FN과 정체 개수를 CSV로 저장합니다.

`CALIBRATE_QUEUE=False`이면 같은 체크포인트의 기존 선택값을 사용하며 없으면 원래 임계값을 사용합니다. 재학습 없이 비교하려면 `RUN_MODE="predict"`, 기존 모델의 실험명·Queue tag/backend을 지정해 주세요.

전부 0인지 판단하는 대상은 제출 템플릿의 링크·미래 6시점 전체입니다. 미래 유효성 마스크는 점수에만 사용하며 보정 위치를 고르는 데 쓰지 않습니다. 공개 관측에서 조건별 표본이 부족하면 실제 개수를 기록하고, 한 조건이 전혀 없으면 선택을 중단합니다.


In [ ]:
QUEUE_CALIBRATION_DIR = QUEUE_DIR / 'calibration-public-v1'
QUEUE_SELECTION = QUEUE_CALIBRATION_DIR / 'selection.json'
ACTIVE_QUEUE_SELECTION = None
if CALIBRATE_QUEUE:
    january_rows, january_manifest = learning.collect_queue_probabilities(
        QUEUE_CHECKPOINT, RELEASE, 'select', QUEUE_CALIBRATION_DIR, VALIDATION_WINDOWS, SEED, CPU_THREADS)
    selection = learning.calibrate_queue(january_rows, january_manifest, QUEUE_CHECKPOINT,
                                         QUEUE_CALIBRATION_DIR, QUEUE_THRESHOLDS)
    display(pd.read_csv(QUEUE_CALIBRATION_DIR / 'january_policy_comparison.csv'))
    print('고정 window_signature:', january_manifest['window_signature'])
    print('1월에서 선택한 Queue 후처리:', selection['policy'])
    del january_rows
if QUEUE_SELECTION.is_file():
    selection = learning.load_queue_selection(QUEUE_SELECTION, QUEUE_CHECKPOINT)
    ACTIVE_QUEUE_SELECTION = QUEUE_SELECTION
else:
    print('Queue 선택 파일이 없어 저장된 임계값을 사용합니다. CALIBRATE_QUEUE=True로 1월에서 선택할 수 있습니다.')
if RUN_FEBRUARY:
    if ACTIVE_QUEUE_SELECTION is None:
        raise ValueError('2월 고정 비교에는 먼저 1월 Queue 후처리를 선택해 주세요.')
    february_rows, february_manifest = learning.collect_queue_probabilities(
        QUEUE_CHECKPOINT, RELEASE, 'holdout', QUEUE_CALIBRATION_DIR, VALIDATION_WINDOWS, SEED, CPU_THREADS)
    queue_saved = torch.load(QUEUE_CHECKPOINT, map_location='cpu', weights_only=True)
    policies = dict(saved=dict(thresholds=dict(onset=queue_saved['threshold'], ongoing=queue_saved['threshold']), force_nonempty=False),
                    selected=selection['policy'])
    reports = {}
    for label, policy in policies.items():
        metrics, windows, leads = learning.queue_policy_metrics(february_rows, policy)
        learning.atomic_queue_csv(windows, QUEUE_CALIBRATION_DIR / f'february_{label}_windows.csv')
        learning.atomic_queue_csv(leads, QUEUE_CALIBRATION_DIR / f'february_{label}_leads.csv')
        reports[label] = metrics
    learning.write_json(QUEUE_CALIBRATION_DIR / 'february_fixed.json',
                        dict(proxy_only=True, selection=selection, manifest=february_manifest, metrics=reports))
    print('1월 선택값을 고정한 2월 Queue 대리 비교:', reports)
    del february_rows, queue_saved


## 7.2. 고정 모델의 차량 수 보정과 최소 변경 투영

기존 모델의 **창별 예측 통합 → 보간 혼합**을 완료한 뒤 링크·시각별 차량 수 `N`의 보정량 하나를 예측합니다. MagiNet은 고정하고, 관측된 셀을 유지하면서 Task 1 대상의 속도·차로당 교통량만 조정합니다. Task 3은 이 최종 출력에서 계산됩니다.

기존 체크포인트로 보정만 학습하려면 `RUN_MODE="predict"`, `CORRECTOR_MODE="train"`과 원래 `MAGINET_EXPERIMENT`을 지정합니다. `USE_RETRIEVAL=False`로 두고 환경·내장 소스·실행 준비·Task 1 체크포인트 경로를 준비한 뒤 이 셀을 실행합니다. Queue 학습은 필요하지 않습니다. 기존 `selected_alpha.json`이 없으면 이 셀에서 1월 후반으로 선택하며, 기존 파일이 다른 체크포인트를 가리키면 중단합니다.

- A: 기존 모델. B: N 보정·투영 + State 손실. C: 같은 구조 + State/N/dN 손실(`0.05/0.03`, FD 손실 0). B/C의 초기화·표본 순서·학습량은 같습니다.
- 관측소 특징을 유효 링크별로 집계한 뒤 두 시간 합성곱(32차원, kernel 3, dilation 1·2)으로 링크마다 하나의 `log1p(N)` 잔차를 예측합니다. 공개 끝점·기울기·거리, 예측·보간 차이, N·dN, 점유율, 실제 그래프의 다른 링크 관측을 사용합니다. N 끝점은 보정 대상이 없는 공개 링크에서만 읽으며, 시각이 끊기면 끝점 검색과 합성곱을 분리합니다.
- 투영은 속도·총교통량을 station→link 평균으로 집계한 뒤 `N = mean(q_total)/max(mean(v),1) × length_km`에 맞춥니다. 보정된 밀도와의 잔차를 줄이는 가중 최소 변경을 계산하며, 정규화된 채널별 fit MSE가 작은 채널을 더 보호합니다. State 비중 `0.54/0.46`도 반영합니다. 원예측의 `log1p(N)` fit MSE는 보수적인 약화 척도로 쓰며 새 보정기의 예측 신뢰도라고 해석하지 않습니다.
- 링크 단위 조정 방향을 유지한 채 함께 축소하여 채널별 fit 절대오차 90백분위·비음수 범위를 지킵니다. 평균 속도 1km/h 이하에서는 유량만 움직이고, 그 위에서는 분모 경계를 넘지 않게 축소합니다. 관측값과 비대상 출력은 바꾸지 않습니다. 출력층의 0 초기화와 `gamma=0`은 기존 출력을 그대로 유지합니다.
- 특징 표준화, 패널×regime별 채널 MSE·보정 상한과 N 잔차 척도는 6~12월에서만 계산합니다. 1월 전반의 best-State/best-selection_proxy를 각각 보관하고, 후반에서 `gamma=0/0.25/0.5/1`을 비교합니다. State·FD가 낮지 않고 N MAE가 커지지 않으며 dN MAE·LWR 대리 지표가 개선된 후보를 선택합니다. 통과 후보가 없으면 A를 유지합니다. `comparison.csv`에 모든 선택 지표의 차이를 저장합니다.
- 기존 모델이 이미 학습한 기간의 오차로 보정층을 학습하므로 fit 손실 감소만으로 개선을 주장하지 않습니다. `RUN_FEBRUARY=True`는 선택을 고정한 추가 진단이며 2월에서 다시 선택하지 않습니다.

모델·통계·선택은 `MAGINET_DIR/physics-corrector/<설정 해시>/`에 저장합니다. `frozen-n-corrector-v2`는 기존 2출력 v1 보정과 별도 해시를 사용하므로 이전 결과를 덮어쓰거나 재개하지 않습니다. 기존 MagiNet과 혼합 선택은 재사용합니다. 같은 v2 설정은 마지막 완료 epoch부터 재개하고, 완료된 선택은 재사용합니다. 기준 모델·alpha·패널·소스 해시가 다르면 이전 보정 선택을 거부합니다. 예측 캐시는 런타임 `CACHE_DIR/corrector/`에 둡니다. 새 런타임에서 적용만 하려면 `CORRECTOR_MODE="predict"`와 출력된 `CORRECTOR_SELECTION` 경로를 지정합니다.

N·dN의 정답 오차를 목표로 학습하며, 비공개 경계 유량을 추정해 강제 보존식을 만들지 않습니다. 공개 데이터의 물리 지표는 공식 Task 3 점수가 아니며, 보정 효과는 같은 대상의 실제 비교로 판단합니다. [공식 평가 구현](https://github.com/jacky850/trafficflowbench-public/blob/main/src/task3/score_task3.py), 공식 출처 확인일: 2026-10-28.


In [ ]:
ACTIVE_CORRECTOR_SELECTION = None
if CORRECTOR_MODE != "off":
    if USE_RETRIEVAL:
        raise ValueError("물리 보정 비교에서는 USE_RETRIEVAL=False로 고정해 주세요.")
    if CORRECTOR_MODE == "train":
        alpha_path = MAGINET_DIR / "selected_alpha.json"
        if not alpha_path.is_file():
            call_maginet(["evaluate", "--release-root", RELEASE, "--checkpoint", MAGINET_CHECKPOINT,
                          "--cache-dir", CACHE_DIR, "--period", "blend", "--select-alpha",
                          "--cpu-threads", CPU_THREADS, *panel_args])
        ACTIVE_CORRECTOR_SELECTION = physics_corrector.train_and_select(
            MAGINET_CHECKPOINT, RELEASE, CACHE_DIR, MAGINET_DIR / "physics-corrector",
            panels=selected_panels, epochs=CORRECTOR_EPOCHS,
            steps_per_group=CORRECTOR_STEPS_PER_GROUP, seed=SEED)
    else:
        ACTIVE_CORRECTOR_SELECTION = Path(CORRECTOR_SELECTION)
        physics_corrector.load_selection(ACTIVE_CORRECTOR_SELECTION, MAGINET_CHECKPOINT,
                                         RELEASE, CACHE_DIR, selected_panels)
    print("CORRECTOR_SELECTION =", str(ACTIVE_CORRECTOR_SELECTION))
    display(pd.read_csv(ACTIVE_CORRECTOR_SELECTION.parent / "comparison.csv"))
    if RUN_FEBRUARY:
        display(physics_corrector.evaluate_selected(ACTIVE_CORRECTOR_SELECTION, MAGINET_CHECKPOINT,
                                                    RELEASE, CACHE_DIR, selected_panels, period="february"))
else:
    print("물리 보정 비활성화: 기존 모델 예측을 사용합니다.")


## 8. validation·private 예측

Task 1 CSV는 총 교통량 `flow_vph`로 저장하며 Task 3에도 사용합니다. Queue는 1월에서 선택한 후처리로 0/1을 출력하며 `-baseline.csv`에는 원래 체크포인트 임계값, `-probabilities.csv`에는 확률을 함께 저장합니다. `RUN_MODE="predict"`에서는 평가를 건너뛰고 저장된 보간·검색 선택값을 사용합니다. 검색을 끄면 1월 평가 원본·train masked 자료는 필요하지 않으며, Queue의 도로 구성 확인을 위한 train 원본 최소 하루는 유지해 주세요. 검색을 사용할 경우에는 같은 fit 자료와 저장된 검색 선택 파일이 있어야 합니다.


In [ ]:
STATE_FILES = []
QUEUE_FILES = []
if MAKE_PREDICTIONS:
    if CORRECTOR_MODE != "off":
        if ACTIVE_CORRECTOR_SELECTION is None:
            raise RuntimeError("먼저 7.2 물리 보정 선택 셀을 실행해 주세요.")
        STATE_FILES = physics_corrector.predict_selected(ACTIVE_CORRECTOR_SELECTION, MAGINET_CHECKPOINT,
                                                         RELEASE, CACHE_DIR, selected_panels)
    else:
        args = ["predict", "--release-root", RELEASE, "--checkpoint", MAGINET_CHECKPOINT,
                "--cache-dir", CACHE_DIR, "--split", "both", "--cpu-threads", CPU_THREADS, *panel_args]
        if USE_RETRIEVAL:
            args += ["--retrieval-selection", RETRIEVAL_SELECTION]
        call_maginet(args)
        state_folder = MAGINET_DIR / ("retrieval/submissions" if USE_RETRIEVAL else "submissions")
        saved = torch.load(MAGINET_CHECKPOINT, map_location="cpu", weights_only=True)
        predicted_panels = selected_panels or sorted(saved["contexts"])
        STATE_FILES = [state_folder / f"task1-{panel}-{split}.csv"
                       for split in ("validation", "private") for panel in predicted_panels]
        if not all(p.is_file() for p in STATE_FILES):
            raise FileNotFoundError("Task 1 예측 파일이 모두 생성되지 않았습니다.")
        del saved
    for split in ("validation", "private"):
        path = QUEUE_DIR / f"queue-{split}.csv"
        learning.predict_queue_calibrated(QUEUE_CHECKPOINT, RELEASE, split, path, ACTIVE_QUEUE_SELECTION)
        QUEUE_FILES.append(path)
    print("Task 1 CSV:", *map(str, STATE_FILES), sep="\n")
    print("Task 2 CSV:", *map(str, QUEUE_FILES), sep="\n")



## 9. 최종 제출 병합

전체 패널 모델에서만 제출을 만듭니다. `TASK4_FILES`가 있으면 기존 Task 4 출력을 사용하고, 없으면 `BUILD_TASK4_BASELINE=True`일 때 원래 NNLS를 사용합니다. Task 4를 바꾸는 실험은 수행하지 않습니다. 모든 task의 자연 키를 `submission_key.csv`에 연결하며 누락·중복·템플릿 밖 키·음수·비유한 값·Queue의 0/1 위반을 거부합니다. 완성 전에는 `.pending.csv`만 쓰고, 전부 맞을 때만 `submission.csv`로 확정합니다.


In [ ]:
TASK_KEYS = {"state": runner.STATE_KEYS, "queue": runner.QUEUE_KEYS, "odme": runner.ODME_KEYS}
TASK_VALUES = {"state": ["speed_kmh", "flow_vph"], "queue": ["queue_pred"], "odme": ["path_flow"]}
OUTPUT_COLUMNS = ["submission_id", "task", "speed_kmh", "flow_vph", "queue_pred", "path_flow"]

def merge_submission(files, key_path, output, chunksize=250_000):
    tables, used = {}, {}
    for task in TASK_KEYS:
        if not files[task]:
            raise ValueError(f"{task} CSV가 필요합니다.")
        columns = TASK_KEYS[task] + TASK_VALUES[task]
        frame = pd.concat([pd.read_csv(p, dtype=str, usecols=columns) for p in files[task]], ignore_index=True)
        if frame.empty or frame[TASK_KEYS[task]].isna().any().any() or frame.duplicated(TASK_KEYS[task]).any():
            raise ValueError(f"{task}: 비었거나 누락·중복 키가 있습니다.")
        for column in TASK_VALUES[task]:
            frame[column] = pd.to_numeric(frame[column], errors="coerce")
        values = frame[TASK_VALUES[task]].to_numpy(float)
        if not np.isfinite(values).all() or (values < 0).any() or (task == "queue" and not np.isin(values, [0, 1]).all()):
            raise ValueError(f"{task}: 잘못된 예측값입니다.")
        tables[task] = frame.set_index(TASK_KEYS[task])
        used[task] = np.zeros(len(frame), bool)
    output = Path(output)
    output.parent.mkdir(parents=True, exist_ok=True)
    pending = output.with_suffix(".pending.csv")
    written = 0
    chunks = pd.read_csv(key_path, dtype=str, chunksize=chunksize)
    try:
        for chunk in chunks:
            if chunk[["submission_id", "task"]].isna().any().any() or not chunk.task.isin(TASK_KEYS).all():
                raise ValueError("제출 키의 ID 또는 task가 잘못되었습니다.")
            out = chunk[["submission_id", "task"]].reset_index(drop=True).copy()
            for column in OUTPUT_COLUMNS[2:]:
                out[column] = 0.
            chunk = chunk.reset_index(drop=True)
            for task, table in tables.items():
                mask = out.task.eq(task)
                if not mask.any():
                    continue
                wanted = pd.MultiIndex.from_frame(chunk.loc[mask, TASK_KEYS[task]])
                indices = table.index.get_indexer(wanted)
                if (indices < 0).any():
                    raise ValueError(f"{task}: 제출 템플릿에 필요한 예측이 누락됐습니다.")
                if len(np.unique(indices)) != len(indices) or used[task][indices].any():
                    raise ValueError(f"{task}: 제출 키에 자연 키가 중복됐습니다.")
                out.loc[mask, TASK_VALUES[task]] = table.iloc[indices][TASK_VALUES[task]].to_numpy()
                used[task][indices] = True
            out[OUTPUT_COLUMNS].to_csv(pending, index=False, mode="w" if written == 0 else "a", header=written == 0)
            written += len(out)
            print(f"제출 병합: {written:,}행", flush=True)
        if written == 0 or any(not mask.all() for mask in used.values()):
            raise ValueError("제출 키가 비었거나 템플릿 밖의 예측 키가 있습니다.")
        pending.replace(output)
    except BaseException:
        pending.unlink(missing_ok=True)
        raise
    finally:
        chunks.close()
    return written

SUBMISSION_PATH = SAVE_ROOT / "submission.csv"
if MAKE_SUBMISSION and MAKE_PREDICTIONS:
    state_saved = torch.load(MAGINET_CHECKPOINT, map_location="cpu", weights_only=True)
    queue_saved = torch.load(QUEUE_CHECKPOINT, map_location="cpu", weights_only=True)
    state_coverage = set(selected_panels or state_saved["contexts"])
    required_queue = {p.name for p in (RELEASE / "task2").iterdir()
                      if p.name not in {"D12_I405_N", "D12_I405_S"}
                      and (p / "validation" / "sample_submission_queue.csv").is_file()}
    full_coverage = state_coverage == set(all_panels) and required_queue <= set(queue_saved["trained_panels"])
    del state_saved, queue_saved
    if not full_coverage:
        print("일부 패널 모델입니다. 패널별 CSV를 확인하고 전체 학습 후 제출을 생성해 주세요.")
    else:
        odme_files = list(map(Path, TASK4_FILES))
        if not odme_files:
            if not BUILD_TASK4_BASELINE:
                raise ValueError("TASK4_FILES를 지정하거나 BUILD_TASK4_BASELINE을 켜 주세요.")
            for split in ("validation", "private"):
                path = RESULTS / f"odme-baseline-{split}.csv"
                runner.predict_odme(RELEASE, split, path, 0.05, 0, 0.1)
                odme_files.append(path)
        count = merge_submission({"state": STATE_FILES, "queue": QUEUE_FILES, "odme": odme_files},
                                 RELEASE / "submission_key.csv", SUBMISSION_PATH)
        print(f"완성된 제출: {SUBMISSION_PATH} ({count:,}행)")


## 10. 저장할 결과

체크포인트·보고서·예측은 실행 중 Drive의 `SAVE_ROOT`에 바로 저장합니다. 학습 셀의 epoch 완료·체크포인트 저장 로그를 확인한 뒤 런타임을 종료해 주세요. 재개하면 마지막으로 완료한 epoch부터 이어집니다.

- `SAVE_ROOT/reports/full/maginet/<실험>/full/`: `best_maginet.pt`, `last_maginet.pt`, 설정·학습 기록
- 같은 실험의 `selected_alpha.json`, `retrieval/selected_retrieval.json`: 저장한 체크포인트에 맞는 혼합 선택값
- `SAVE_ROOT/reports/full/queue/<이름>-<backend>/`: `queue.pt`, `queue.last.pt`, epoch 로그, 평가, 예측
- `SAVE_ROOT/submission.csv`: 전체 task·패널의 키가 일치할 때만 생성되는 Kaggle 업로드 파일

`WORK_ROOT/cache/`와 `WORK_ROOT/data/processed_v2/`는 임시 파일입니다. 런타임이 바뀌면 원본에서 다시 생성합니다. 같은 Drive 결과를 다시 사용할 때는 `RESTORE_ROOT=None`을 유지하고, 학습을 이어갈 때만 `RESUME=True`로 설정해 주세요. 새로 학습하려면 별도 결과 폴더를 사용합니다.

소스 수정이나 GPU·정밀도 변경 후에는 이전 실행과 같은 수치가 나온다고 가정하지 마세요. State·`selection_proxy`·Physics 대리 지표·공식 Kaggle 종합 점수는 서로 구분합니다.

